# 09 Topic Modelling

This GitHub copy has been cleaned for readability: duplicate/empty cells were removed and saved cell outputs were cleared. Code and analysis workflow are retained.

**Execution note:** Original project paths may reference Google Drive/Colab. Update paths and provide required source files/API credentials in your own environment before rerunning.

In [ ]:

# ============================================================
# NOTEBOOK 09 — TOPIC MODELLING
# ICMB-130 Indian Cinema Multimodal Benchmark Dataset
# CELL 1: MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/gdrive2')

print("\nGoogle Drive mounted successfully.")


In [ ]:

# ============================================================
# CELL 2: VERIFY PROJECT DIRECTORY
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

print("=" * 60)
print("NOTEBOOK 09 — PROJECT DIRECTORY CHECK")
print("=" * 60)

print("Project root:", PROJECT_ROOT)
print("Project exists:", PROJECT_ROOT.exists())

if PROJECT_ROOT.exists():
    print("\nProject folders and files:")

    for item in sorted(PROJECT_ROOT.iterdir()):
        print(
            "[DIR] " if item.is_dir() else "[FILE]",
            item.name
        )
else:
    print("\nProject directory not found.")
    print("Please verify the Google Drive path.")


In [ ]:

# ============================================================
# NOTEBOOK 09 — CELL 3
# INSPECT DATA DIRECTORY
# ============================================================

from pathlib import Path

DATA_DIR = PROJECT_ROOT / "01_data"

print("=" * 70)
print("NOTEBOOK 09 — DATA DIRECTORY INSPECTION")
print("=" * 70)

for item in sorted(DATA_DIR.iterdir()):
    print(f"\n{'[DIR]' if item.is_dir() else '[FILE]'} {item.name}")

    if item.is_dir():
        children = sorted(item.iterdir())
        for child in children[:30]:
            print(
                f"    {'[DIR]' if child.is_dir() else '[FILE]'} "
                f"{child.name}"
            )

        if len(children) > 30:
            print(f"    ... and {len(children) - 30} more items")

print("\nData directory inspection complete.")


In [ ]:

# ============================================================
# NOTEBOOK 09 — CELL 4
# INSPECT SUBTITLE CORPORA AND EXISTING AUDITS
# ============================================================

from pathlib import Path

PROCESSED_DIR = DATA_DIR / "processed"

CORPUS_DIRS = [
    PROCESSED_DIR / "subtitle_corpus",
    PROCESSED_DIR / "subtitle_corpus_v2",
    DATA_DIR / "subtitles_txt",
]

for folder in CORPUS_DIRS:
    print("\n" + "=" * 75)
    print(f"FOLDER: {folder}")
    print("=" * 75)

    if not folder.exists():
        print("Folder does not exist.")
        continue

    all_files = sorted(p for p in folder.rglob("*") if p.is_file())

    print("Total files:", len(all_files))

    # Display all files with names relevant to the audit
    # and a sample of the remaining files.
    audit_files = [
        p for p in all_files
        if any(
            word in p.name.lower()
            for word in [
                "manifest", "audit", "readiness",
                "validation", "report", "summary"
            ]
        )
    ]

    print("\nManifest / audit / validation files:")
    if audit_files:
        for p in audit_files:
            print(" ", p.relative_to(folder))
    else:
        print("  None found by filename.")

    print("\nFirst 20 files:")
    for p in all_files[:20]:
        print(" ", p.relative_to(folder))

    if len(all_files) > 20:
        print(f" ... and {len(all_files) - 20} more files")

print("\nInspection complete.")


In [ ]:
# NOTEBOOK 09 — FINAL CORPUS INTEGRITY CHECK

from pathlib import Path
import pandas as pd
from IPython.display import display

CORPUS_V2_PATH = (
    PROJECT_ROOT
    / "06_outputs"
    / "topic_modelling"
    / "ICMB_130_TOPIC_MODELLING_CORPUS_V2.csv"
)

# Read the saved version, not just the in-memory DataFrame.
final_corpus = pd.read_csv(CORPUS_V2_PATH)

final_corpus["source_exists"] = final_corpus["source_file"].apply(
    lambda p: Path(str(p)).is_file() if pd.notna(p) else False
)

checks = {
    "Total records": len(final_corpus),
    "Unique movie IDs": final_corpus["movie_record_id"].nunique(),
    "Years covered": final_corpus["year"].nunique(),
    "First year": final_corpus["year"].min(),
    "Last year": final_corpus["year"].max(),
    "Subtitle source files existing": int(final_corpus["source_exists"].sum()),
    "Documents with non-empty dialogue": int(
        final_corpus["clean_dialogue"].fillna("").str.strip().ne("").sum()
    ),
    "Documents with at least 500 words": int(
        final_corpus["clean_word_count"].ge(500).sum()
    ),
    "Total cleaned words": int(final_corpus["clean_word_count"].sum()),
}

for label, value in checks.items():
    print(f"{label}: {value}")

print("\nRecords per year:")
display(final_corpus.groupby("year").size().rename("movie_count").to_frame())

assert len(final_corpus) == 130
assert final_corpus["movie_record_id"].nunique() == 130
assert final_corpus["source_exists"].all()
assert final_corpus["clean_dialogue"].fillna("").str.strip().ne("").all()

print("\nINTEGRITY CHECK PASSED")

In [ ]:
# NOTEBOOK 09 — NMF TOPIC MODELLING BASELINE

import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
from IPython.display import display

CORPUS_V2_PATH = (
    PROJECT_ROOT
    / "06_outputs"
    / "topic_modelling"
    / "ICMB_130_TOPIC_MODELLING_CORPUS_V2.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load the saved and verified corpus.
final_corpus = pd.read_csv(CORPUS_V2_PATH)

assert len(final_corpus) == 130
assert final_corpus["movie_record_id"].nunique() == 130

documents = (
    final_corpus["clean_dialogue"]
    .fillna("")
    .astype(str)
    .tolist()
)

# Remove common subtitle and transcription boilerplate.
custom_stopwords = {
    "copyright", "transcribed", "corrected", "source",
    "downloaded", "subtitles", "subtitle", "enjoy",
    "flick", "yify", "yts", "telegram", "channel",
    "official", "fixed", "synced", "translation",
    "translator", "http", "https"
}

stop_words = sorted(set(ENGLISH_STOP_WORDS).union(custom_stopwords))

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    stop_words=stop_words,
    token_pattern=r"(?u)\b\w\w+\b",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.85,
    max_features=12000,
    sublinear_tf=True,
    dtype=np.float64
)

X = vectorizer.fit_transform(documents)

N_TOPICS = 10

nmf_model = NMF(
    n_components=N_TOPICS,
    init="nndsvda",
    random_state=42,
    max_iter=500
)

W = nmf_model.fit_transform(X)
H = nmf_model.components_

feature_names = vectorizer.get_feature_names_out()

# Show the strongest terms in each topic.
topic_rows = []

for topic_idx, weights in enumerate(H):
    top_indices = weights.argsort()[::-1][:12]

    topic_rows.append({
        "topic_id": topic_idx + 1,
        "top_terms": ", ".join(feature_names[top_indices]),
        "reconstruction_weight": round(float(weights.sum()), 4)
    })

topic_terms_df = pd.DataFrame(topic_rows)

print("Document-term matrix shape:", X.shape)
print("Number of topics:", N_TOPICS)
print("NMF iterations:", nmf_model.n_iter_)
print("Reconstruction error:", round(float(nmf_model.reconstruction_err_), 4))

display(topic_terms_df)

# Save topic terms for reproducibility.
topic_terms_path = OUTPUT_DIR / "NMF_TOPIC_TERMS.csv"
topic_terms_df.to_csv(topic_terms_path, index=False, encoding="utf-8-sig")

print("\nSaved topic terms to:", topic_terms_path)

In [ ]:
# NOTEBOOK 09 — REPRESENTATIVE MOVIES FOR EACH NMF TOPIC

import numpy as np
import pandas as pd
from IPython.display import display

# Confirm that the model output aligns with the 130-film corpus.
assert W.shape[0] == len(final_corpus)
assert W.shape[1] == N_TOPICS

# Normalize each movie's topic weights so they sum to 1.
row_totals = W.sum(axis=1, keepdims=True)
W_normalized = np.divide(
    W,
    row_totals,
    out=np.zeros_like(W),
    where=row_totals != 0
)

# Save document-topic weights.
doc_topic_df = final_corpus[
    ["movie_record_id", "year", "rank", "movie_title", "film_language"]
].copy()

for topic_idx in range(N_TOPICS):
    doc_topic_df[f"topic_{topic_idx + 1}_weight"] = W_normalized[:, topic_idx]

doc_topic_df["dominant_topic"] = (
    W_normalized.argmax(axis=1) + 1
)

doc_topic_df["dominant_topic_weight"] = W_normalized.max(axis=1)

weights_path = OUTPUT_DIR / "NMF_DOCUMENT_TOPIC_WEIGHTS.csv"
doc_topic_df.to_csv(weights_path, index=False, encoding="utf-8-sig")

# Display the top 10 representative films for every topic.
representative_rows = []

for topic_idx in range(N_TOPICS):
    weight_col = f"topic_{topic_idx + 1}_weight"

    top_films = doc_topic_df.nlargest(10, weight_col)

    for position, (_, film) in enumerate(top_films.iterrows(), start=1):
        representative_rows.append({
            "topic_id": topic_idx + 1,
            "position": position,
            "movie_record_id": film["movie_record_id"],
            "year": film["year"],
            "movie_title": film["movie_title"],
            "film_language": film["film_language"],
            "topic_weight": round(film[weight_col], 4)
        })

representative_df = pd.DataFrame(representative_rows)

representatives_path = (
    OUTPUT_DIR / "NMF_REPRESENTATIVE_MOVIES.csv"
)
representative_df.to_csv(
    representatives_path, index=False, encoding="utf-8-sig"
)

# Show each topic's top 5 films for convenient interpretation.
for topic_idx in range(N_TOPICS):
    print(f"\n{'=' * 70}")
    print(f"TOPIC {topic_idx + 1}")
    print(topic_terms_df.loc[
        topic_terms_df["topic_id"] == topic_idx + 1, "top_terms"
    ].iloc[0])

    display(
        representative_df[
            representative_df["topic_id"] == topic_idx + 1
        ].head(5)
    )

print("\nSaved document-topic weights:", weights_path)
print("Saved representative movies:", representatives_path)

In [ ]:
# NOTEBOOK 09 — REFINED NMF MODEL
# Keeps the original baseline unchanged.

import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
from IPython.display import display

# Load the saved, verified corpus.
CORPUS_V2_PATH = (
    PROJECT_ROOT / "06_outputs" / "topic_modelling"
    / "ICMB_130_TOPIC_MODELLING_CORPUS_V2.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"

refined_corpus = pd.read_csv(CORPUS_V2_PATH)

assert len(refined_corpus) == 130
assert refined_corpus["movie_record_id"].nunique() == 130

# Correct only the clearly observed OCR-like substitutions.
ocr_corrections = {
    "wiii": "will",
    "iike": "like",
    "piease": "please",
    "aii": "all",
    "teii": "tell",
    "iife": "life",
    "iet": "let",
    "oniy": "only",
    "ieave": "leave",
    "iove": "love",
    "heiio": "hello",
}

def correct_obvious_ocr(text):
    text = str(text)
    for wrong, right in ocr_corrections.items():
        text = re.sub(
            rf"(?<!\w){re.escape(wrong)}(?!\w)",
            right,
            text,
            flags=re.IGNORECASE
        )
    return text

documents_refined = (
    refined_corpus["clean_dialogue"]
    .fillna("")
    .map(correct_obvious_ocr)
    .tolist()
)

custom_stopwords = {
    "copyright", "transcribed", "corrected", "source",
    "downloaded", "subtitles", "subtitle", "enjoy",
    "flick", "yify", "yts", "telegram", "channel",
    "official", "fixed", "synced", "translation",
    "translator", "http", "https"
}

stop_words = sorted(set(ENGLISH_STOP_WORDS).union(custom_stopwords))

vectorizer_refined = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    stop_words=stop_words,
    # Keep Unicode words; exclude numbers and underscore-only tokens.
    token_pattern=r"(?u)\b[^\W\d_]{2,}\b",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.85,
    max_features=12000,
    sublinear_tf=True,
    dtype=np.float64
)

X_refined = vectorizer_refined.fit_transform(documents_refined)

nmf_refined = NMF(
    n_components=10,
    init="nndsvda",
    random_state=42,
    max_iter=500
)

W_refined = nmf_refined.fit_transform(X_refined)
H_refined = nmf_refined.components_
terms_refined = vectorizer_refined.get_feature_names_out()

# Extract top 12 terms for each refined topic.
refined_topic_rows = []

for topic_idx, weights in enumerate(H_refined):
    top_idx = weights.argsort()[::-1][:12]
    refined_topic_rows.append({
        "topic_id": topic_idx + 1,
        "top_terms": ", ".join(terms_refined[top_idx])
    })

refined_topics_df = pd.DataFrame(refined_topic_rows)

print("Refined document-term matrix:", X_refined.shape)
print("NMF iterations:", nmf_refined.n_iter_)
print("Reconstruction error:", round(float(nmf_refined.reconstruction_err_), 4))
display(refined_topics_df)

# Save separately; do not overwrite the baseline files.
refined_topics_path = OUTPUT_DIR / "NMF_TOPIC_TERMS_REFINED.csv"
refined_topics_df.to_csv(
    refined_topics_path, index=False, encoding="utf-8-sig"
)

print("\nSaved refined topics to:", refined_topics_path)

In [ ]:

# NOTEBOOK 09 — CHECK CORPUS COLUMN NAMES

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
CORPUS_PATH = (
    PROJECT_ROOT
    / "06_outputs"
    / "topic_modelling"
    / "ICMB_130_TOPIC_MODELLING_CORPUS_V2.csv"
)

corpus_df = pd.read_csv(CORPUS_PATH)

print("Shape:", corpus_df.shape)
print("\nColumn names:")
for i, col in enumerate(corpus_df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nFirst 2 rows (text shortened where applicable):")
display(corpus_df.head(2))


In [ ]:

# NOTEBOOK 09 — AUDIT EXISTING ICMB-130 METADATA FILES
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

files_to_check = [
    PROJECT_ROOT / "01_data/processed/ICMB_130_MASTER_DATA.csv",
    PROJECT_ROOT / "01_data/processed/ICMB_130_FEATURES.csv",
    PROJECT_ROOT / "01_data/processed/ICMB_130_TMDB_MAPPING.csv",
    PROJECT_ROOT / "01_data/raw/ICMB_130_TMDB_METADATA.csv",
    PROJECT_ROOT / "01_data/raw/ICMB_130_TMDB_FINANCIALS.csv",
]

for path in files_to_check:
    print("\n" + "=" * 80)
    print(path.name)
    print("=" * 80)

    if not path.exists():
        print("FILE NOT FOUND:", path)
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    if "movie_record_id" in df.columns:
        print(
            "Unique movie IDs:",
            df["movie_record_id"].nunique()
        )

    # Show completeness of potentially useful metadata fields
    relevant = [
        col for col in df.columns
        if any(term in col.lower() for term in [
            "genre", "industry", "language", "director",
            "cast", "keyword", "country", "runtime",
            "budget", "revenue", "gross", "tmdb",
            "movie_record_id", "match_status"
        ])
    ]

    if relevant:
        coverage = pd.DataFrame({
            "column": relevant,
            "non_missing": [
                df[col].notna().sum() for col in relevant
            ],
            "coverage_pct": [
                round(df[col].notna().mean() * 100, 1)
                for col in relevant
            ]
        })
        display(coverage)

    display(df.head(3))


In [ ]:

# NOTEBOOK 09 — PREPARE METADATA-ENRICHED ANALYSIS TABLE

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
PROCESSED = PROJECT_ROOT / "01_data" / "processed"
RAW = PROJECT_ROOT / "01_data" / "raw"
OUTPUT = PROJECT_ROOT / "06_outputs" / "topic_modelling"

corpus_path = OUTPUT / "ICMB_130_TOPIC_MODELLING_CORPUS_V2.csv"
metadata_path = RAW / "ICMB_130_TMDB_METADATA.csv"
mapping_path = PROCESSED / "ICMB_130_TMDB_MAPPING.csv"
features_path = PROCESSED / "ICMB_130_FEATURES.csv"

corpus = pd.read_csv(corpus_path)
metadata = pd.read_csv(metadata_path)
mapping = pd.read_csv(mapping_path)
features = pd.read_csv(features_path)

# Select useful metadata fields without duplicating the join key
metadata_cols = [
    "movie_record_id", "genres", "overview", "runtime",
    "tmdb_original_language", "production_countries",
    "production_companies", "spoken_languages",
    "vote_average", "vote_count", "popularity"
]
metadata_cols = [
    c for c in metadata_cols if c in metadata.columns
]

mapping_cols = [
    "movie_record_id", "industry", "match_status",
    "final_score", "candidate_count"
]
mapping_cols = [
    c for c in mapping_cols if c in mapping.columns
]

feature_cols = [
    "movie_record_id", "india_gross_inr_cr",
    "log_india_gross", "revenue_budget_ratio",
    "india_gross_share_pct", "budget_available",
    "worldwide_revenue_available"
]
feature_cols = [
    c for c in feature_cols if c in features.columns
]

# Check key uniqueness before joining
for name, df in [
    ("corpus", corpus),
    ("metadata", metadata),
    ("mapping", mapping),
    ("features", features)
]:
    assert df["movie_record_id"].is_unique, (
        f"{name} has duplicate movie_record_id values"
    )

# Preserve every corpus movie and join metadata by stable ID
analysis_df = corpus.merge(
    metadata[metadata_cols],
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

analysis_df = analysis_df.merge(
    mapping[mapping_cols],
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

analysis_df = analysis_df.merge(
    features[feature_cols],
    on="movie_record_id",
    how="left",
    validate="one_to_one"
)

print("Enriched analysis table:", analysis_df.shape)
print("Unique movie IDs:", analysis_df["movie_record_id"].nunique())
print("Genre coverage:", analysis_df["genres"].notna().sum(), "/ 130")
print("Industry coverage:", analysis_df["industry"].notna().sum(), "/ 130")

print("\nGenre examples:")
display(
    analysis_df[
        ["movie_record_id", "movie_title", "year", "genres", "industry"]
    ].head(10)
)

# Save as a separate output; source files remain untouched
out_path = OUTPUT / "ICMB_130_TOPIC_ANALYSIS_METADATA.csv"
analysis_df.to_csv(out_path, index=False)
print("\nSaved:", out_path)


In [ ]:

# NOTEBOOK 09 — REFINED NMF REPRESENTATIVE MOVIES
# Uses the saved V2 subtitle corpus and preserves existing outputs.

import re
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"
CORPUS_PATH = OUTPUT_DIR / "ICMB_130_TOPIC_MODELLING_CORPUS_V2.csv"

corpus_df = pd.read_csv(CORPUS_PATH)

assert len(corpus_df) == 130
assert corpus_df["movie_record_id"].is_unique
assert "clean_dialogue" in corpus_df.columns

# Apply the same explicit OCR corrections as the refined model
ocr_corrections = {
    "wiii": "will",
    "iike": "like",
    "piease": "please",
    "aii": "all",
    "teii": "tell",
    "iife": "life",
    "iet": "let",
    "oniy": "only",
    "ieave": "leave",
    "iove": "love",
    "heiio": "hello"
}

def correct_obvious_ocr(text):
    text = str(text)
    for wrong, right in ocr_corrections.items():
        text = re.sub(
            rf"\b{re.escape(wrong)}\b",
            right,
            text,
            flags=re.IGNORECASE
        )
    return text

cleaned_documents = (
    corpus_df["clean_dialogue"]
    .fillna("")
    .apply(correct_obvious_ocr)
    .tolist()
)

# Recreate TF-IDF matrix
custom_stopwords = {
    "subtitle", "subtitles", "www", "com",
    "movie", "movies", "english"
}
stop_words = list(set(ENGLISH_STOP_WORDS).union(custom_stopwords))

tfidf_vectorizer = TfidfVectorizer(
    max_features=12000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.85,
    stop_words=stop_words,
    token_pattern=r"(?u)\b[^\W\d_]{2,}\b"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_documents)
print("Refined document-term matrix:", tfidf_matrix.shape)

# Fit refined NMF
nmf_model = NMF(
    n_components=10,
    init="nndsvda",
    random_state=42,
    max_iter=500
)

W = nmf_model.fit_transform(tfidf_matrix)

print("NMF iterations:", nmf_model.n_iter_)
print("Reconstruction error:", round(nmf_model.reconstruction_err_, 4))

# Save top terms without overwriting earlier outputs
terms = tfidf_vectorizer.get_feature_names_out()
topic_rows = []

for topic_idx, component in enumerate(nmf_model.components_, start=1):
    top_indices = component.argsort()[::-1][:12]
    topic_rows.append({
        "topic_id": topic_idx,
        "top_terms": ", ".join(terms[top_indices])
    })

pd.DataFrame(topic_rows).to_csv(
    OUTPUT_DIR / "NMF_TOPIC_TERMS_REFINED_REBUILT.csv",
    index=False
)

# Document-topic weights
topic_columns = [f"topic_{i}" for i in range(1, 11)]
weights_df = pd.DataFrame(W, columns=topic_columns)

weights_df.insert(
    0, "movie_record_id", corpus_df["movie_record_id"].values
)

weights_df.insert(
    1, "movie_title", corpus_df["movie_title"].values
)
weights_df.insert(2, "year", corpus_df["year"].values)

weights_df.to_csv(
    OUTPUT_DIR / "NMF_DOCUMENT_TOPIC_WEIGHTS_REFINED_REBUILT.csv",
    index=False
)

# Representative movies for each topic
results = corpus_df[
    ["movie_record_id", "movie_title", "year", "film_language"]
].copy()

results["dominant_topic"] = W.argmax(axis=1) + 1
results["topic_weight"] = W.max(axis=1)

for topic_id in range(1, 11):
    print(f"\n{'=' * 65}")
    print(f"TOPIC {topic_id} — TOP 5 REPRESENTATIVE MOVIES")
    print("=" * 65)

    topic_movies = (
        results[results["dominant_topic"] == topic_id]
        .sort_values("topic_weight", ascending=False)
        .head(5)
    )
    display(topic_movies.round({"topic_weight": 4}))

results.to_csv(
    OUTPUT_DIR / "NMF_REPRESENTATIVE_MOVIES_REFINED_REBUILT.csv",
    index=False
)

print("\nRefined model outputs saved separately.")


In [ ]:

# NOTEBOOK 09 — COMPARE ORIGINAL AND REBUILT REFINED TOPIC TERMS

from pathlib import Path
import pandas as pd

OUTPUT_DIR = (
    Path("/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics")
    / "06_outputs"
    / "topic_modelling"
)

files_to_compare = [
    "NMF_TOPIC_TERMS_REFINED.csv",
    "NMF_TOPIC_TERMS_REFINED_REBUILT.csv",
    "NMF_TOPIC_TERMS.csv"
]

for filename in files_to_compare:
    path = OUTPUT_DIR / filename
    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    if not path.exists():
        print("File not found.")
        continue

    df = pd.read_csv(path)
    print("Shape:", df.shape)
    display(df.head(10))


In [ ]:

from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

print("=" * 70)
print("CHECKING ORIGINAL REFINED MODEL OUTPUTS")
print("=" * 70)

files_to_check = [
    "NMF_TOPIC_TERMS_REFINED.csv",
    "NMF_DOCUMENT_TOPIC_WEIGHTS_REFINED.csv",
    "NMF_REPRESENTATIVE_MOVIES_REFINED.csv",
    "NMF_TOPIC_TERMS_REFINED_REBUILT.csv",
    "NMF_DOCUMENT_TOPIC_WEIGHTS_REFINED_REBUILT.csv",
    "NMF_REPRESENTATIVE_MOVIES_REFINED_REBUILT.csv",
]

for filename in files_to_check:
    path = OUTPUT_DIR / filename
    print(f"{filename}: {'EXISTS' if path.exists() else 'NOT FOUND'}")
    if path.exists():
        df = pd.read_csv(path)
        print(f"    Shape: {df.shape}")
        print(f"    Columns: {df.columns.tolist()}")

print("\nCheck complete. No files were modified.")


In [ ]:

from pathlib import Path
import json

NOTEBOOK_PATH = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "02_notebooks/09_Topic_Modelling.ipynb"
)

print("=" * 75)
print("SEARCHING NOTEBOOK 09 FOR ORIGINAL NMF SETTINGS")
print("=" * 75)

if not NOTEBOOK_PATH.exists():
    print("Notebook file not found:", NOTEBOOK_PATH)
else:
    with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    keywords = [
        "TfidfVectorizer",
        "NMF(",
        "ocr_corrections",
        "correct_obvious_ocr",
        "custom_stopwords",
        "stop_words",
        "NMF_TOPIC_TERMS_REFINED.csv",
        "reconstruction_err_",
    ]

    found = 0

    for cell_no, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        matched = [
            keyword for keyword in keywords
            if keyword.lower() in source.lower()
        ]

        if matched:
            found += 1
            print(f"\n{'=' * 75}")
            print(f"CELL INDEX: {cell_no}")
            print("MATCHED:", ", ".join(matched))
            print("-" * 75)
            print(source[:12000])

    if found == 0:
        print("No matching code cells found.")
    else:
        print(f"\nTotal matching code cells: {found}")

print("\nNotebook inspection complete. No files were modified.")


In [ ]:

# NOTEBOOK 09 — REPRODUCE ORIGINAL REFINED NMF
# Uses the exact preprocessing and model settings from Cell 7.
# Saves outputs under new filenames; preserves all existing files.

import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"
CORPUS_PATH = OUTPUT_DIR / "ICMB_130_TOPIC_MODELLING_CORPUS_V2.csv"

corpus = pd.read_csv(CORPUS_PATH)

assert len(corpus) == 130
assert corpus["movie_record_id"].nunique() == 130
assert "clean_dialogue" in corpus.columns

# Exact OCR corrections from original refined Cell 7.
ocr_corrections = {
    "wiii": "will",
    "iike": "like",
    "piease": "please",
    "aii": "all",
    "teii": "tell",
    "iife": "life",
    "iet": "let",
    "oniy": "only",
    "ieave": "leave",
    "iove": "love",
    "heiio": "hello",
}

def correct_obvious_ocr(text):
    text = str(text)
    for wrong, right in ocr_corrections.items():
        text = re.sub(
            rf"(?<!\w){re.escape(wrong)}(?!\w)",
            right,
            text,
            flags=re.IGNORECASE
        )
    return text

documents = (
    corpus["clean_dialogue"]
    .fillna("")
    .map(correct_obvious_ocr)
    .tolist()
)

# Exact custom stopwords from original refined Cell 7.
custom_stopwords = {
    "copyright", "transcribed", "corrected", "source",
    "downloaded", "subtitles", "subtitle", "enjoy",
    "flick", "yify", "yts", "telegram", "channel",
    "official", "fixed", "synced", "translation",
    "translator", "http", "https"
}

stop_words = sorted(
    set(ENGLISH_STOP_WORDS).union(custom_stopwords)
)

# Exact TF-IDF settings from original refined Cell 7.
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    stop_words=stop_words,
    token_pattern=r"(?u)\b[^\W\d_]{2,}\b",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.85,
    max_features=12000,
    sublinear_tf=True,
    dtype=np.float64
)

X = vectorizer.fit_transform(documents)

# Exact NMF settings from original refined Cell 7.
model = NMF(
    n_components=10,
    init="nndsvda",
    random_state=42,
    max_iter=500
)

W = model.fit_transform(X)
H = model.components_
terms = vectorizer.get_feature_names_out()

print("=" * 70)
print("ORIGINAL REFINED NMF — REPRODUCTION CHECK")
print("=" * 70)
print("Document-term matrix:", X.shape)
print("NMF iterations:", model.n_iter_)
print("Reconstruction error:", round(float(model.reconstruction_err_), 4))

# Reconstructed topic terms.
topic_rows = []
for topic_idx, component in enumerate(H):
    top_indices = component.argsort()[::-1][:12]
    topic_rows.append({
        "topic_id": topic_idx + 1,
        "top_terms": ", ".join(terms[top_indices]),
        "reconstruction_weight": round(float(component.sum()), 4)
    })

topics_df = pd.DataFrame(topic_rows)

# Save under new filenames to protect the original topic-term file.
topics_path = OUTPUT_DIR / "NMF_TOPIC_TERMS_REFINED_REPRODUCED.csv"
topics_df.to_csv(topics_path, index=False, encoding="utf-8-sig")

# Document-topic weights.
topic_columns = [f"topic_{i}" for i in range(1, 11)]
weights_df = pd.DataFrame(W, columns=topic_columns)

weights_df.insert(
    0, "movie_record_id", corpus["movie_record_id"].values
)
weights_df.insert(
    1, "movie_title", corpus["movie_title"].values
)
weights_df.insert(2, "year", corpus["year"].values)

if "film_language" in corpus.columns:
    weights_df.insert(
        3, "film_language", corpus["film_language"].values
    )

weights_path = (
    OUTPUT_DIR / "NMF_DOCUMENT_TOPIC_WEIGHTS_REFINED_REPRODUCED.csv"
)
weights_df.to_csv(weights_path, index=False, encoding="utf-8-sig")

# One representative-movie row per film, with its dominant topic.
representatives_df = weights_df.copy()
topic_weight_columns = [f"topic_{i}" for i in range(1, 11)]

representatives_df["dominant_topic"] = (
    representatives_df[topic_weight_columns].to_numpy().argmax(axis=1) + 1
)
representatives_df["topic_weight"] = (
    representatives_df[topic_weight_columns].to_numpy().max(axis=1)
)

# Inspect the top five dominant-topic films for each topic.
for topic_id in range(1, 11):
    print(f"\nTOPIC {topic_id} — TOP REPRESENTATIVE MOVIES")
    display(
        representatives_df[
            representatives_df["dominant_topic"] == topic_id
        ]
        .sort_values("topic_weight", ascending=False)
        .head(5)
        .round({"topic_weight": 4})
    )

representatives_path = (
    OUTPUT_DIR / "NMF_REPRESENTATIVE_MOVIES_REFINED_REPRODUCED.csv"
)
representatives_df.to_csv(
    representatives_path, index=False, encoding="utf-8-sig"
)

print("\nSaved new outputs:")
print(topics_path)
print(weights_path)
print(representatives_path)

# Compare reproduced topic terms with the originally saved terms.
original_terms_path = OUTPUT_DIR / "NMF_TOPIC_TERMS_REFINED.csv"

if original_terms_path.exists():
    original_terms = pd.read_csv(original_terms_path)
    print("\nORIGINAL VS REPRODUCED TOPIC TERMS")
    comparison = original_terms.merge(
        topics_df[["topic_id", "top_terms"]],
        on="topic_id",
        how="outer",
        suffixes=("_original", "_reproduced")
    )
    display(comparison)

print("\nCompleted. Existing outputs were not overwritten.")


In [ ]:

# NOTEBOOK 09 — TOPIC INTERPRETATION TABLE
# Reads the reproduced outputs; does not rerun or modify the model.

from pathlib import Path
import pandas as pd
from IPython.display import display

OUTPUT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

terms_path = OUTPUT_DIR / "NMF_TOPIC_TERMS_REFINED_REPRODUCED.csv"
weights_path = (
    OUTPUT_DIR / "NMF_DOCUMENT_TOPIC_WEIGHTS_REFINED_REPRODUCED.csv"
)

terms_df = pd.read_csv(terms_path)
weights_df = pd.read_csv(weights_path)

topic_columns = [f"topic_{i}" for i in range(1, 11)]

# Find the five films with the largest raw weight for each topic.
representative_rows = []

for topic_id in range(1, 11):
    col = f"topic_{topic_id}"

    top_films = (
        weights_df[
            ["movie_record_id", "movie_title", "year", col]
        ]
        .sort_values(col, ascending=False)
        .head(5)
    )

    film_list = "; ".join(
        f"{row.movie_title} ({int(row.year)})"
        for row in top_films.itertuples(index=False)
    )

    representative_rows.append({
        "topic_id": topic_id,
        "representative_films_top5": film_list
    })

representatives = pd.DataFrame(representative_rows)

interpretation_df = terms_df.merge(
    representatives,
    on="topic_id",
    how="left"
)

# Add a blank field for evidence-based manual interpretation.
interpretation_df["provisional_label"] = ""
interpretation_df["interpretation_notes"] = ""

display(interpretation_df)

output_path = (
    OUTPUT_DIR / "NMF_TOPIC_INTERPRETATION_TABLE.csv"
)
interpretation_df.to_csv(
    output_path, index=False, encoding="utf-8-sig"
)

print("\nSaved interpretation table:")
print(output_path)
print("Rows:", len(interpretation_df))
print("Existing model outputs were not modified.")


In [ ]:

from pathlib import Path
import pandas as pd
from IPython.display import display

OUTPUT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

terms_path = (
    OUTPUT_DIR / "NMF_TOPIC_TERMS_REFINED_REPRODUCED.csv"
)

terms_df = pd.read_csv(terms_path)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

print("FULL TOP TERMS FOR ALL 10 TOPICS")
display(terms_df[["topic_id", "top_terms"]].to_string(index=False))

print("\nNo files were modified.")


In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

interpretation_path = (
    OUTPUT_DIR / "NMF_TOPIC_INTERPRETATION_TABLE.csv"
)

interpretation_df = pd.read_csv(interpretation_path)

topic_labels = {
    1: "Police and Law Enforcement",
    2: "Hindi Script / Language Effect",
    3: "Historical and Royal Narratives",
    4: "Geopolitical and National-Security Action",
    5: "General Conversational Dialogue",
    6: "Action and Film-Specific Vocabulary",
    7: "OCR / Subtitle Text Artifacts",
    8: "Sports and Competition",
    9: "Heist and Crime Action",
    10: "Family and Relationship Narratives",
}

topic_notes = {
    1: "Police and legal-enforcement vocabulary. Check whether the topic captures a recurring theme across films rather than only genre-specific dialogue.",
    2: "Devanagari terms dominate. Likely influenced by Hindi-language or script representation; do not interpret as a substantive narrative theme without further validation.",
    3: "Strong historical and royal vocabulary, supported by representative historical and period films.",
    4: "Geopolitical and national-security vocabulary involving borders, terrorism, Pakistan, and Kashmir.",
    5: "Generic conversational expressions. Weak thematic specificity; retain as a provisional topic pending validation.",
    6: "Mixed action, setting, character, and film-specific vocabulary. Interpret cautiously and inspect representative documents.",
    7: "Several malformed tokens suggest OCR or subtitle text artifacts. Treat as a potential data-quality topic.",
    8: "Sports and competition vocabulary, supported by representative sports-centred films.",
    9: "Crime and heist vocabulary mixed with character names. Interpret as a provisional crime-action topic.",
    10: "Family and relationship vocabulary mixed with possible OCR artifacts. Interpret cautiously.",
}

interpretation_df["provisional_label"] = (
    interpretation_df["topic_id"].map(topic_labels)
)

interpretation_df["interpretation_notes"] = (
    interpretation_df["topic_id"].map(topic_notes)
)

interpretation_df.to_csv(interpretation_path, index=False)

print("Updated interpretation table:")
print("Shape:", interpretation_df.shape)
display(
    interpretation_df[
        ["topic_id", "provisional_label", "interpretation_notes"]
    ]
)

print("\nSaved to:", interpretation_path)

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

weights_path = (
    OUTPUT_DIR / "NMF_DOCUMENT_TOPIC_WEIGHTS_REFINED_REPRODUCED.csv"
)

weights_df = pd.read_csv(weights_path)

print("=" * 65)
print("NMF DOCUMENT-TOPIC WEIGHT FILE: COLUMN INSPECTION")
print("=" * 65)

print("\nShape:", weights_df.shape)
print("\nExact column names:")
for i, col in enumerate(weights_df.columns, start=1):
    print(f"{i:02d}. {col}")

print("\nFirst 3 rows:")
display(weights_df.head(3))

print("\nNumeric columns:")
print(weights_df.select_dtypes(include="number").columns.tolist())

print("\nNo files were modified.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

OUTPUT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

weights_path = (
    OUTPUT_DIR / "NMF_DOCUMENT_TOPIC_WEIGHTS_REFINED_REPRODUCED.csv"
)
interpretation_path = (
    OUTPUT_DIR / "NMF_TOPIC_INTERPRETATION_TABLE.csv"
)

weights_df = pd.read_csv(weights_path)
interpretation_df = pd.read_csv(interpretation_path)

topic_cols = [f"topic_{i}" for i in range(1, 11)]

# Validate expected columns
missing_cols = [col for col in topic_cols if col not in weights_df.columns]
if missing_cols:
    raise ValueError(f"Missing expected topic columns: {missing_cols}")

# Check data quality
print("=" * 65)
print("NMF TOPIC DISTRIBUTION VALIDATION")
print("=" * 65)
print("Rows:", len(weights_df))
print("Unique movie IDs:", weights_df["movie_record_id"].nunique())
print("Duplicate movie IDs:", weights_df["movie_record_id"].duplicated().sum())
print("Missing topic weights:", int(weights_df[topic_cols].isna().sum().sum()))
print("Negative topic weights:", int((weights_df[topic_cols] < 0).sum().sum()))

# Assign each film to its highest-weight topic
weights_df["dominant_topic_id"] = (
    weights_df[topic_cols].to_numpy().argmax(axis=1) + 1
)
weights_df["dominant_topic_weight"] = (
    weights_df[topic_cols].max(axis=1)
)

# Summarize dominant-topic assignments
summary = (
    weights_df.groupby("dominant_topic_id")
    .agg(
        dominant_film_count=("movie_record_id", "count"),
        mean_dominant_weight=("dominant_topic_weight", "mean"),
        median_dominant_weight=("dominant_topic_weight", "median"),
        max_dominant_weight=("dominant_topic_weight", "max"),
    )
    .reindex(range(1, 11), fill_value=0)
    .reset_index()
    .rename(columns={"dominant_topic_id": "topic_id"})
)

summary["film_share_pct"] = (
    summary["dominant_film_count"] / len(weights_df) * 100
).round(2)

summary = summary.merge(
    interpretation_df[["topic_id", "provisional_label"]],
    on="topic_id",
    how="left"
)

summary = summary[
    [
        "topic_id",
        "provisional_label",
        "dominant_film_count",
        "film_share_pct",
        "mean_dominant_weight",
        "median_dominant_weight",
        "max_dominant_weight",
    ]
]

print("\nDOMINANT TOPIC SUMMARY")
display(summary.round(4))

print(
    "\nTotal dominant-topic assignments:",
    int(summary["dominant_film_count"].sum())
)

print("\nNo files were modified.")

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd
import sys

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

CORPUS_PATH = (
    PROJECT_ROOT
    / "06_outputs/topic_modelling"
    / "ICMB_130_TOPIC_MODELLING_CORPUS_V2.csv"
)

print("=" * 70)
print("BERTopic READINESS CHECK")
print("=" * 70)

# 1. Check the source corpus
if not CORPUS_PATH.exists():
    raise FileNotFoundError(f"Corpus not found: {CORPUS_PATH}")

corpus_df = pd.read_csv(CORPUS_PATH)

print("\nCORPUS")
print("Path:", CORPUS_PATH)
print("Shape:", corpus_df.shape)
print("Columns:", corpus_df.columns.tolist())

required_cols = [
    "movie_record_id",
    "year",
    "movie_title",
    "film_language",
    "clean_dialogue",
]

missing_cols = [
    col for col in required_cols
    if col not in corpus_df.columns
]

print("Missing required columns:", missing_cols)

if not missing_cols:
    print("Unique movie IDs:", corpus_df["movie_record_id"].nunique())
    print("Duplicate movie IDs:", corpus_df["movie_record_id"].duplicated().sum())
    print("Missing dialogue documents:", corpus_df["clean_dialogue"].isna().sum())

    corpus_df["document_word_count"] = (
        corpus_df["clean_dialogue"]
        .fillna("")
        .astype(str)
        .str.split()
        .str.len()
    )

    print(
        "Documents with fewer than 100 words:",
        int((corpus_df["document_word_count"] < 100).sum())
    )

    print("\nFILM-LANGUAGE COUNTS")
    display(
        corpus_df["film_language"]
        .fillna("Missing")
        .value_counts()
        .rename_axis("film_language")
        .reset_index(name="film_count")
    )

    print("\nSAMPLE DOCUMENT METADATA")
    display(
        corpus_df[
            ["movie_record_id", "year", "movie_title",
             "film_language", "document_word_count"]
        ].head(8)
    )

    sample_text = str(corpus_df.iloc[0]["clean_dialogue"])
    print("\nSAMPLE DIALOGUE EXCERPT (first 700 characters)")
    print(sample_text[:700])

# 2. Check installed packages
print("\nPACKAGE AVAILABILITY")
packages = [
    "bertopic",
    "sentence_transformers",
    "transformers",
    "umap",
    "hdbscan",
    "torch",
    "sklearn",
]

for package in packages:
    installed = importlib.util.find_spec(package) is not None
    print(f"{package}: {'available' if installed else 'not installed'}")

# 3. Check runtime
print("\nRUNTIME")
print("Python:", sys.version.split()[0])

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("PyTorch runtime check:", str(exc))

print("\nNo files were modified and no packages were installed.")

In [ ]:
import re
import pandas as pd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
CORPUS_PATH = (
    PROJECT_ROOT / "06_outputs/topic_modelling"
    / "ICMB_130_TOPIC_MODELLING_CORPUS_V2.csv"
)

df = pd.read_csv(CORPUS_PATH)

# Normalize language labels in this analysis copy only
language_map = {
    "hi": "Hindi",
    "te": "Telugu",
    "ta": "Tamil",
    "kn": "Kannada",
}

df["analysis_language"] = (
    df["film_language"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
    .replace(language_map)
)

# Audit text patterns without modifying the dialogue
spaced_letters_pattern = re.compile(
    r"\b(?:[A-Za-z]\s+){3,}[A-Za-z]\b"
)
unicode_word_pattern = re.compile(r"\b[^\W\d_]+\b", re.UNICODE)

def audit_document(text):
    text = str(text) if pd.notna(text) else ""

    spaced_sequences = spaced_letters_pattern.findall(text)
    tokens = unicode_word_pattern.findall(text)

    non_ascii_tokens = [
        token for token in tokens
        if not token.isascii()
    ]

    suspicious_examples = [
        token for token in tokens
        if any(char in token.lower() for char in ["ö", "ä", "ü", "ï", "ñ", "ø"])
    ]

    return pd.Series({
        "character_count": len(text),
        "word_count": len(text.split()),
        "spaced_letter_sequences": len(spaced_sequences),
        "spaced_letter_examples": "; ".join(spaced_sequences[:5]),
        "non_ascii_token_count": len(non_ascii_tokens),
        "non_ascii_examples": "; ".join(non_ascii_tokens[:10]),
        "suspicious_character_examples": "; ".join(suspicious_examples[:10]),
    })

audit = df.apply(
    lambda row: audit_document(row["clean_dialogue"]),
    axis=1
)

audit_df = pd.concat(
    [
        df[[
            "movie_record_id", "year", "movie_title",
            "film_language", "analysis_language"
        ]],
        audit,
    ],
    axis=1
)

print("=" * 70)
print("SUBTITLE TEXT QUALITY AUDIT")
print("=" * 70)

print("\nCorpus totals")
print("Films:", len(audit_df))
print(
    "Films with spaced-letter sequences:",
    int((audit_df["spaced_letter_sequences"] > 0).sum())
)
print(
    "Films with non-ASCII tokens:",
    int((audit_df["non_ascii_token_count"] > 0).sum())
)
print(
    "Total spaced-letter sequences:",
    int(audit_df["spaced_letter_sequences"].sum())
)
print(
    "Total non-ASCII tokens:",
    int(audit_df["non_ascii_token_count"].sum())
)

print("\nMost affected documents — spaced-letter sequences")
display(
    audit_df.sort_values(
        "spaced_letter_sequences", ascending=False
    ).head(15)
)

print("\nDocuments with the most non-ASCII tokens")
display(
    audit_df.sort_values(
        "non_ascii_token_count", ascending=False
    ).head(15)
)

print("\nNormalized analysis language counts")
display(
    df["analysis_language"]
    .value_counts()
    .rename_axis("analysis_language")
    .reset_index(name="film_count")
)

print("\nAudit only: no files were modified.")

In [ ]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

CORPUS_PATH = (
    PROJECT_ROOT
    / "06_outputs/topic_modelling"
    / "ICMB_130_TOPIC_MODELLING_CORPUS_V2.csv"
)

df = pd.read_csv(CORPUS_PATH)

df["document_word_count"] = (
    df["clean_dialogue"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

print("=" * 70)
print("SHORT DOCUMENT REVIEW")
print("=" * 70)

short_docs = df.loc[
    df["document_word_count"] < 100,
    [
        "movie_record_id",
        "movie_title",
        "year",
        "film_language",
        "document_word_count",
        "source_file",
    ],
]

display(short_docs)

print("\n" + "=" * 70)
print("SUSPICIOUS CHARACTER REVIEW")
print("=" * 70)

for _, row in df.iterrows():
    text = str(row["clean_dialogue"])
    count = len(re.findall(r"[A-Za-z]*ö[A-Za-z]*", text, flags=re.I))

    if count:
        examples = re.findall(
            r"\b[A-Za-z]*ö[A-Za-z]*\b",
            text,
            flags=re.I,
        )[:12]

        print(
            f"\n{row['movie_record_id']} | {row['movie_title']} "
            f"| suspicious words: {count}"
        )
        print("Examples:", examples)

print("\n" + "=" * 70)
print("SPACED-LETTER SEQUENCE REVIEW")
print("=" * 70)

pattern = re.compile(r"\b(?:[A-Za-z]\s+){3,}[A-Za-z]\b")

for _, row in df.iterrows():
    matches = pattern.findall(str(row["clean_dialogue"]))

    if matches:
        print(
            f"\n{row['movie_record_id']} | {row['movie_title']}"
        )
        print("Examples:", matches[:8])

print("\nReview only. No files were modified.")

In [ ]:
from pathlib import Path
import re
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_PATH = (
    PROJECT_ROOT
    / "06_outputs/topic_modelling"
    / "ICMB_130_TOPIC_MODELLING_CORPUS_V2.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_PATH)

# Work on a new analysis copy
df["dialogue_for_bertopic"] = (
    df["clean_dialogue"].fillna("").astype(str)
)

# These are known subtitle boilerplate phrases, not film dialogue.
boilerplate_patterns = [
    r"(?i)\bcopyright\s+from\s+ecotone\b",
    r"(?i)\bcopyright\s+from\s+simed(?:ia)?\b",
    r"(?i)\bhope\s+you\s+(?:have\s+)?liked\b",
    r"(?i)\benjoyed\s+the\s+movie\b",
    r"(?i)\bdownload(?:ed)?\s+from\b",
    r"(?i)\bsubtitles?\s+(?:by|downloaded|provided)\b",
    r"(?i)\bwww\.[^\s]+",
    r"(?i)\bhttps?://[^\s]+",
]

# Only target spaced-letter patterns that match identified promotional
# phrases. Do not join arbitrary spaced letters across the document.
spaced_promo_patterns = [
    r"(?i)\bC\s*o\s*p\s*y\s*r\s*i\s*g\s*h\s*t\s+f\s*r\s*o\s*m\s+e\s*c\s*O\s*t\s*O\s*n\s*e\b",
    r"(?i)\bH\s*o\s*p\s*e\s+Y\s*o\s*u\b",
    r"(?i)\bE\s*n\s*j\s*o\s*y\s*e\s*d\s+T\s*h\s*e\s+M\s*o\s*v\s*i\s*e\b",
]

cleaned_texts = []
removed_counts = []
correction_counts = []

for _, row in df.iterrows():
    text = row["dialogue_for_bertopic"]
    before = text

    for pattern in boilerplate_patterns + spaced_promo_patterns:
        text = re.sub(pattern, " ", text)

    # The audit identified this corruption in these two specific films.
    if row["movie_record_id"] in {"2000_1", "2011_2"}:
        text = re.sub(r"(?i)\byöu\b", "you", text)
        text = re.sub(r"(?i)\byöunger\b", "younger", text)

    # Collapse whitespace without altering legitimate non-English scripts.
    text = re.sub(r"\s+", " ", text).strip()

    cleaned_texts.append(text)
    removed_counts.append(max(0, len(before) - len(text)))
    correction_counts.append(
        int(row["movie_record_id"] in {"2000_1", "2011_2"})
    )

df["dialogue_for_bertopic"] = cleaned_texts
df["characters_removed_by_cleanup"] = removed_counts
df["targeted_corruption_reviewed"] = correction_counts
df["bertopic_word_count"] = (
    df["dialogue_for_bertopic"].str.split().str.len()
)

# Flag rather than invent or fabricate dialogue for these films.
df["dialogue_status"] = "ready_for_review"
df.loc[
    df["bertopic_word_count"] < 100,
    "dialogue_status"
] = "replacement_required"

print("=" * 70)
print("DIALOGUE-ONLY PREPROCESSING REVIEW")
print("=" * 70)

print("Rows:", len(df))
print("Unique movie IDs:", df["movie_record_id"].nunique())
print(
    "Documents requiring replacement:",
    int((df["dialogue_status"] == "replacement_required").sum())
)

display(
    df[
        [
            "movie_record_id",
            "movie_title",
            "bertopic_word_count",
            "characters_removed_by_cleanup",
            "dialogue_status",
        ]
    ].sort_values("bertopic_word_count").head(15)
)

print("\nSample after cleanup:")
sample_id = "2000_1"
print(
    df.loc[
        df["movie_record_id"] == sample_id,
        "dialogue_for_bertopic"
    ].iloc[0][:700]
)

# Save a NEW intermediate file; source corpus remains unchanged.
review_path = OUTPUT_DIR / "ICMB_130_BERTOPIC_DIALOGUE_REVIEW.csv"
df.to_csv(review_path, index=False)

print("\nSaved review copy:", review_path)
print("Original corpus and NMF outputs were not modified.")

In [ ]:
from pathlib import Path
import re
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"

REVIEW_PATH = (
    OUTPUT_DIR / "ICMB_130_BERTOPIC_DIALOGUE_REVIEW.csv"
)

OUTPUT_PATH = (
    OUTPUT_DIR / "ICMB_130_BERTOPIC_DIALOGUE_RECOVERED_REVIEW.csv"
)

review_df = pd.read_csv(REVIEW_PATH)

TARGET_IDS = {"2009_2", "2020_5"}

def decode_subtitle_file(path):
    """Decode using the file's BOM before trying other encodings."""
    path = Path(str(path))
    raw = path.read_bytes()

    if raw.startswith((b"\xff\xfe", b"\xfe\xff")):
        return raw.decode("utf-16"), "UTF-16 BOM"
    if raw.startswith(b"\xef\xbb\xbf"):
        return raw.decode("utf-8-sig"), "UTF-8 BOM"

    for encoding in ("utf-8", "cp1252", "latin-1"):
        try:
            return raw.decode(encoding), encoding
        except UnicodeDecodeError:
            continue

    raise UnicodeError(f"Could not decode subtitle file: {path}")

def extract_dialogue(srt_text):
    """Remove SRT structure and obvious non-dialogue promotional lines."""
    srt_text = srt_text.replace("\ufeff", "").replace("\r\n", "\n")
    srt_text = srt_text.replace("\r", "\n")

    dialogue_lines = []

    timestamp_pattern = re.compile(
        r"^\s*\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->"
    )

    for line in srt_text.split("\n"):
        line = line.strip()

        if not line:
            continue

        # Subtitle sequence numbers
        if re.fullmatch(r"\d+", line):
            continue

        # Subtitle timing lines
        if timestamp_pattern.match(line):
            continue

        # Remove subtitle formatting tags, preserving their text
        line = re.sub(r"<[^>]+>", " ", line)
        line = re.sub(r"\{\\[^}]+\}", " ", line)
        line = line.strip()

        if not line:
            continue

        # Remove obvious standalone credits/promotional text only
        normalized = re.sub(r"\s+", " ", line).strip().lower()
        if re.search(
            r"copyright\s+from|hope\s+you\s+(?:have\s+)?liked|"
            r"enjoyed\s+the\s+movie|subtitles?\s+(?:by|provided)",
            normalized,
        ):
            continue

        dialogue_lines.append(line)

    return re.sub(r"\s+", " ", " ".join(dialogue_lines)).strip()

recovery_results = []

for idx, row in review_df.iterrows():
    if row["movie_record_id"] not in TARGET_IDS:
        continue

    raw_text, encoding_used = decode_subtitle_file(row["source_file"])
    recovered_text = extract_dialogue(raw_text)

    review_df.at[idx, "dialogue_for_bertopic"] = recovered_text
    review_df.at[idx, "bertopic_word_count"] = len(recovered_text.split())
    review_df.at[idx, "dialogue_status"] = (
        "recovered_pending_review"
        if len(recovered_text.split()) >= 100
        else "still_requires_review"
    )

    recovery_results.append({
        "movie_record_id": row["movie_record_id"],
        "movie_title": row["movie_title"],
        "encoding_used": encoding_used,
        "raw_file_word_count": len(raw_text.split()),
        "recovered_dialogue_word_count": len(recovered_text.split()),
        "dialogue_status": review_df.at[idx, "dialogue_status"],
        "dialogue_preview": recovered_text[:500],
    })

# Save a new file; do not overwrite either the source corpus or prior review.
review_df.to_csv(OUTPUT_PATH, index=False)

print("=" * 70)
print("RECOVERED SUBTITLE REVIEW")
print("=" * 70)

display(pd.DataFrame(recovery_results))

print("\nSaved new review file:", OUTPUT_PATH)
print("Original V2 corpus and NMF outputs remain unchanged.")

In [ ]:
from pathlib import Path
import pandas as pd
import re

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

INPUT_PATH = (
    PROJECT_ROOT
    / "06_outputs/topic_modelling"
    / "ICMB_130_BERTOPIC_DIALOGUE_RECOVERED_REVIEW.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"

df = pd.read_csv(INPUT_PATH)

df["dialogue_for_bertopic"] = (
    df["dialogue_for_bertopic"].fillna("").astype(str)
)

df["bertopic_word_count"] = (
    df["dialogue_for_bertopic"].str.split().str.len()
)

timestamp_pattern = re.compile(
    r"\b\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->"
)

df["contains_timestamp"] = df["dialogue_for_bertopic"].apply(
    lambda text: bool(timestamp_pattern.search(text))
)

print("=" * 70)
print("FULL CORPUS VALIDATION — PRE-BERTOPIC")
print("=" * 70)

print("Rows:", len(df))
print("Unique movie IDs:", df["movie_record_id"].nunique())
print("Duplicate movie IDs:", df["movie_record_id"].duplicated().sum())
print("Missing dialogue:", int(df["dialogue_for_bertopic"].str.strip().eq("").sum()))
print("Documents below 100 words:", int((df["bertopic_word_count"] < 100).sum()))
print("Documents below 500 words:", int((df["bertopic_word_count"] < 500).sum()))
print("Documents containing timestamps:", int(df["contains_timestamp"].sum()))

print("\nDocument length summary:")
display(df["bertopic_word_count"].describe().to_frame("word_count"))

print("\nShortest documents:")
display(
    df[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "bertopic_word_count",
            "dialogue_status",
        ]
    ]
    .sort_values("bertopic_word_count")
    .head(10)
)

print("\nNo files were modified.")

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

files_to_check = {
    "Original V2 corpus": (
        OUTPUT_DIR / "ICMB_130_TOPIC_MODELLING_CORPUS_V2.csv"
    ),
    "Earlier dialogue review": (
        OUTPUT_DIR / "ICMB_130_BERTOPIC_DIALOGUE_REVIEW.csv"
    ),
    "Recovered dialogue review": (
        OUTPUT_DIR / "ICMB_130_BERTOPIC_DIALOGUE_RECOVERED_REVIEW.csv"
    ),
}

for label, path in files_to_check.items():
    print("\n" + "=" * 70)
    print(label)
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    data = pd.read_csv(path)
    print("Shape:", data.shape)

    if "movie_record_id" not in data.columns:
        print("movie_record_id column not found.")
        continue

    for movie_id in ["2009_2", "2020_5"]:
        matches = data.loc[
            data["movie_record_id"].astype(str) == movie_id
        ]

        if matches.empty:
            print(f"{movie_id}: not found")
            continue

        for _, row in matches.iterrows():
            if "dialogue_for_bertopic" in data.columns:
                text = str(row["dialogue_for_bertopic"])
            elif "clean_dialogue" in data.columns:
                text = str(row["clean_dialogue"])
            else:
                text = ""

            print(
                f"{movie_id} | {row.get('movie_title', '')} "
                f"| stored words: {len(text.split())}"
            )
            print("Preview:", text[:180])

print("\nDiagnostic only: no files were modified.")

In [ ]:
from pathlib import Path
import re
import pandas as pd

PROJECT_ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"

INPUT_PATH = (
    OUTPUT_DIR / "ICMB_130_BERTOPIC_DIALOGUE_RECOVERED_REVIEW.csv"
)

OUTPUT_PATH = (
    OUTPUT_DIR / "ICMB_130_BERTOPIC_DIALOGUE_RECOVERED_V2.csv"
)

df = pd.read_csv(INPUT_PATH)

movie_id = "2020_5"
mask = df["movie_record_id"].astype(str).eq(movie_id)

if mask.sum() != 1:
    raise ValueError(
        f"Expected exactly one row for {movie_id}; found {mask.sum()}."
    )

source_path = Path(df.loc[mask, "source_file"].iloc[0])

if not source_path.exists():
    raise FileNotFoundError(f"Subtitle file not found: {source_path}")

raw = source_path.read_bytes()

# Respect the BOM and decode the original SRT.
if raw.startswith((b"\xff\xfe", b"\xfe\xff")):
    subtitle_text = raw.decode("utf-16")
elif raw.startswith(b"\xef\xbb\xbf"):
    subtitle_text = raw.decode("utf-8-sig")
else:
    subtitle_text = raw.decode("utf-8")

def extract_srt_dialogue(text):
    text = text.replace("\ufeff", "").replace("\r\n", "\n")
    text = text.replace("\r", "\n")
    lines_out = []

    timestamp = re.compile(
        r"^\s*\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->"
    )

    for line in text.split("\n"):
        line = line.strip()

        if not line or re.fullmatch(r"\d+", line):
            continue
        if timestamp.match(line):
            continue

        line = re.sub(r"<[^>]+>", " ", line)
        line = re.sub(r"\{\\[^}]+\}", " ", line)
        line = re.sub(r"\s+", " ", line).strip()

        if not line:
            continue

        normalized = line.lower()
        if re.search(
            r"copyright\s+from|hope\s+you\s+(?:have\s+)?liked|"
            r"enjoyed\s+the\s+movie|subtitles?\s+(?:by|provided)",
            normalized,
        ):
            continue

        lines_out.append(line)

    return re.sub(r"\s+", " ", " ".join(lines_out)).strip()

recovered_text = extract_srt_dialogue(subtitle_text)
recovered_words = len(recovered_text.split())

if recovered_words < 100:
    raise ValueError(
        f"Recovery produced only {recovered_words} words. "
        "Do not save this result; inspect the source file."
    )

df.loc[mask, "dialogue_for_bertopic"] = recovered_text
df.loc[mask, "bertopic_word_count"] = recovered_words
df.loc[mask, "dialogue_status"] = "recovered_pending_review"

df.to_csv(OUTPUT_PATH, index=False)

print("Film:", df.loc[mask, "movie_title"].iloc[0])
print("Recovered words:", recovered_words)
print("Preview:", recovered_text[:600])
print("\nSaved corrected copy:", OUTPUT_PATH)
print("Original corpus and previous review files remain unchanged.")

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

files = [
    "ICMB_130_BERTOPIC_DIALOGUE_RECOVERED_REVIEW.csv",
    "ICMB_130_BERTOPIC_DIALOGUE_RECOVERED_V2.csv",
]

for filename in files:
    path = OUTPUT_DIR / filename

    print("\n" + "=" * 70)
    print(filename)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    data = pd.read_csv(path)
    print("Shape:", data.shape)

    for movie_id in ["2009_2", "2020_5"]:
        row = data.loc[
            data["movie_record_id"].astype(str) == movie_id
        ]

        if row.empty:
            print(movie_id, "NOT FOUND")
            continue

        text_col = (
            "dialogue_for_bertopic"
            if "dialogue_for_bertopic" in data.columns
            else "clean_dialogue"
        )

        text = str(row.iloc[0][text_col])
        print(
            movie_id,
            "|", row.iloc[0]["movie_title"],
            "| words:", len(text.split()),
            "| preview:", text[:150]
        )

print("\nDiagnostic only: no files were modified.")

In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

SUBTITLE_FOLDER = (
    ROOT
    / "01_data"
    / "processed"
    / "subtitle_corpus_v2"
    / "incoming_manual_2017_2025"
    / "Extracted_Subtitles_2017_2025"
)

print("Folder exists:", SUBTITLE_FOLDER.exists())
print("Folder:", SUBTITLE_FOLDER)

if SUBTITLE_FOLDER.exists():
    files = sorted(
        p for p in SUBTITLE_FOLDER.rglob("*")
        if p.is_file()
    )

    print("Total files:", len(files))

    for file in files[:30]:
        print(file.name)

    if len(files) > 30:
        print(f"...and {len(files) - 30} more files")
else:
    print("Please check the mounted Drive path.")

In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

SUBTITLE_FOLDER = (
    ROOT
    / "01_data"
    / "processed"
    / "subtitle_corpus_v2"
    / "incoming_manual_2017_2025"
    / "Extracted_Subtitles_2017_2025"
)

files = sorted(
    p for p in SUBTITLE_FOLDER.rglob("*")
    if p.is_file()
)

print("=" * 65)
print("SUBTITLE SOURCE FILE INVENTORY")
print("=" * 65)

keywords = [
    "STREET", "DANCER", "MAGADHEERA",
    "KRRISH", "READY", "SAIYAARA", "COOLIE"
]

for keyword in keywords:
    matches = [
        p for p in files
        if keyword in p.name.upper()
    ]

    print(f"\n{keyword}: {len(matches)} match(es)")

    for path in matches:
        print(" ", path.name)

print("\nInventory complete.")

In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

print("Checking Drive mount:", ROOT.exists())

# Search within the project for the exact filename
matches = list(
    ROOT.rglob("2020_5_STREET_DANCER_3D_HINDI.srt.srt")
)

print("Exact matches found:", len(matches))

for i, path in enumerate(matches, start=1):
    print(f"\nMatch {i}:")
    print(path)
    print("File exists:", path.is_file())

# If the exact filename differs, search for related files
if not matches:
    print("\nSearching for related Street Dancer files...")
    for path in ROOT.rglob("*STREET*DANCER*"):
        if path.is_file():
            print(path)

In [ ]:
from pathlib import Path
import re

SRT_PATH = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/"
    "01_data/processed/subtitle_corpus_v2/"
    "incoming_manual_2017_2025/Extracted_Subtitles_2017_2025/"
    "street-dancer-3d_english-2168063/"
    "2020_5_STREET_DANCER_3D_HINDI.srt.srt"
)

assert SRT_PATH.is_file(), f"File not found: {SRT_PATH}"

raw = SRT_PATH.read_bytes()

if raw.startswith((b"\xff\xfe", b"\xfe\xff")):
    subtitle_text = raw.decode("utf-16")
elif raw.startswith(b"\xef\xbb\xbf"):
    subtitle_text = raw.decode("utf-8-sig")
else:
    subtitle_text = raw.decode("utf-8", errors="replace")

dialogue_lines = []
timestamp_pattern = re.compile(
    r"^\s*\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->"
)

for line in subtitle_text.replace("\r\n", "\n").replace(
    "\r", "\n"
).split("\n"):
    line = line.strip()

    if (
        not line
        or re.fullmatch(r"\d+", line)
        or timestamp_pattern.match(line)
    ):
        continue

    line = re.sub(r"<[^>]+>", " ", line)
    line = re.sub(r"\{\\[^}]+\}", " ", line)
    line = re.sub(r"\s+", " ", line).strip()

    if not line:
        continue

    if re.search(
        r"(?i)copyright\s+from|"
        r"subtitles?\s+(?:by|provided)|"
        r"hope\s+you\s+(?:have\s+)?liked|"
        r"enjoyed\s+the\s+movie",
        line,
    ):
        continue

    dialogue_lines.append(line)

recovered_dialogue = " ".join(dialogue_lines).strip()
word_count = len(recovered_dialogue.split())

print("Source:", SRT_PATH.name)
print("Extracted lines:", len(dialogue_lines))
print("Recovered words:", word_count)
print("\nPreview:")
print(recovered_dialogue[:1000])

assert word_count >= 100, (
    f"Extraction looks incomplete: only {word_count} words."
)

print("\nPASS: extraction completed for review.")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
OUT = ROOT / "06_outputs" / "topic_modelling"

INPUT = OUT / "ICMB_130_BERTOPIC_DIALOGUE_RECOVERED_V2.csv"
MOVIE_ID = "2020_5"

# Reload the original review file
test_df = pd.read_csv(INPUT)

print("=" * 60)
print("1. COLUMN DIAGNOSTICS")
print("=" * 60)

print("Duplicate column names:",
      test_df.columns[test_df.columns.duplicated()].tolist())

target_columns = [
    (i, repr(col))
    for i, col in enumerate(test_df.columns)
    if "dialogue_for_bertopic" in str(col)
]

print("Dialogue-related columns:", target_columns)

print("\n" + "=" * 60)
print("2. ROW DIAGNOSTICS")
print("=" * 60)

mask = test_df["movie_record_id"].astype(str).eq(MOVIE_ID)
row_indices = test_df.index[mask].tolist()

print("Matching row indices:", row_indices)
print("Matching rows:", int(mask.sum()))

row_idx = row_indices[0]
col_idx = test_df.columns.get_loc("dialogue_for_bertopic")

print("Column index:", col_idx)
print("Original text:", repr(test_df.at[row_idx, "dialogue_for_bertopic"]))
print("Original words:",
      len(str(test_df.at[row_idx, "dialogue_for_bertopic"]).split()))

print("\n" + "=" * 60)
print("3. IN-MEMORY ASSIGNMENT TEST")
print("=" * 60)

# Use the already extracted dialogue from the successful extraction cell
print("Incoming object type:", type(recovered))
print("Incoming characters:", len(recovered))
print("Incoming words:", len(recovered.split()))

# Assign to the exact row and column
test_df.at[row_idx, "dialogue_for_bertopic"] = recovered

updated_text = test_df.at[row_idx, "dialogue_for_bertopic"]

print("Updated characters:", len(str(updated_text)))
print("Updated words:", len(str(updated_text).split()))
print("Updated preview:", repr(str(updated_text)[:200]))

assert len(str(updated_text).split()) == len(recovered.split()), (
    "The in-memory assignment failed; do not save this DataFrame."
)

print("\nPASS: the dialogue assignment works in memory.")
print("No files were modified by this diagnostic cell.")

In [ ]:
from pathlib import Path
import csv
import pandas as pd

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
OUT = ROOT / "06_outputs" / "topic_modelling"

INPUT = OUT / "ICMB_130_BERTOPIC_DIALOGUE_RECOVERED_V2.csv"
TEMP = OUT / "ICMB_130_BERTOPIC_DIALOGUE_FINAL_REVIEW_TEMP.csv"
OUTPUT = OUT / "ICMB_130_BERTOPIC_DIALOGUE_FINAL_REVIEW.csv"

MOVIE_ID = "2020_5"

# 1. Clean the extracted dialogue
clean_recovered = recovered.replace("\x00", " ")
clean_recovered = " ".join(clean_recovered.split())
expected_words = len(clean_recovered.split())

assert expected_words >= 100
print("Cleaned recovered words:", expected_words)
print("NUL characters remaining:", clean_recovered.count("\x00"))

# 2. Reload the original review file and update exactly one row
df = pd.read_csv(INPUT, encoding="utf-8")
mask = df["movie_record_id"].astype(str).eq(MOVIE_ID)

assert len(df) == 130
assert mask.sum() == 1
assert df.columns.tolist().count("dialogue_for_bertopic") == 1

row_idx = df.index[mask][0]
df.at[row_idx, "dialogue_for_bertopic"] = clean_recovered
df.at[row_idx, "bertopic_word_count"] = expected_words
df.at[row_idx, "dialogue_status"] = "recovered_pending_review"

# Confirm assignment before saving
assert len(df.at[row_idx, "dialogue_for_bertopic"].split()) == expected_words

# 3. Write a temporary CSV with explicit quoting
df.to_csv(
    TEMP,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
)

# 4. Verify with pandas
check = pd.read_csv(TEMP, encoding="utf-8")
saved_row = check.loc[
    check["movie_record_id"].astype(str).eq(MOVIE_ID)
].iloc[0]
pandas_words = len(str(saved_row["dialogue_for_bertopic"]).split())

print("Pandas re-read word count:", pandas_words)

# 5. Independently verify with Python's CSV reader
with TEMP.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    matching_rows = [
        row for row in reader
        if row["movie_record_id"] == MOVIE_ID
    ]

assert len(matching_rows) == 1
csv_words = len(matching_rows[0]["dialogue_for_bertopic"].split())

print("Python CSV reader word count:", csv_words)

# Do not publish unless both readers preserve the full dialogue
assert pandas_words == expected_words, (
    f"Pandas read {pandas_words} words; expected {expected_words}."
)
assert csv_words == expected_words, (
    f"CSV reader read {csv_words} words; expected {expected_words}."
)

# 6. Publish the verified file
TEMP.replace(OUTPUT)

# 7. Final independent verification
final_check = pd.read_csv(OUTPUT, encoding="utf-8")
final_row = final_check.loc[
    final_check["movie_record_id"].astype(str).eq(MOVIE_ID)
].iloc[0]
final_words = len(str(final_row["dialogue_for_bertopic"]).split())

assert len(final_check) == 130
assert final_check["movie_record_id"].nunique() == 130
assert final_words == expected_words

print("\nPASS: final review file saved and verified.")
print("Final Street Dancer 3D word count:", final_words)
print("Final file:", OUTPUT)
print("Original subtitle source and NMF outputs remain unchanged.")

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

CORPUS_PATH = (
    ROOT / "06_outputs" / "topic_modelling"
    / "ICMB_130_BERTOPIC_DIALOGUE_FINAL_REVIEW.csv"
)

df = pd.read_csv(CORPUS_PATH)

required = [
    "movie_record_id",
    "movie_title",
    "year",
    "dialogue_for_bertopic",
]

missing_columns = [
    col for col in required if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

df["dialogue_for_bertopic"] = (
    df["dialogue_for_bertopic"].fillna("").astype(str)
)

df["actual_word_count"] = df["dialogue_for_bertopic"].map(
    lambda text: len(text.split())
)

print("=" * 65)
print("ICMB-130 — FINAL DIALOGUE CORPUS VALIDATION")
print("=" * 65)

print("Dataset rows:", len(df))
print("Unique movie IDs:", df["movie_record_id"].nunique())
print("Duplicate movie IDs:", df["movie_record_id"].duplicated().sum())
print("Missing dialogue:", (df["dialogue_for_bertopic"].str.strip() == "").sum())
print("Documents under 100 words:", (df["actual_word_count"] < 100).sum())
print("Documents under 500 words:", (df["actual_word_count"] < 500).sum())
print("Total dialogue words:", f"{df['actual_word_count'].sum():,}")
print("Minimum document words:", df["actual_word_count"].min())
print("Median document words:", int(df["actual_word_count"].median()))
print("Maximum document words:", df["actual_word_count"].max())

# Inspect short documents
short_docs = df.loc[
    df["actual_word_count"] < 500,
    ["movie_record_id", "movie_title", "year", "actual_word_count"]
].sort_values("actual_word_count")

print("\nDocuments under 500 words:")
if short_docs.empty:
    print("None")
else:
    print(short_docs.to_string(index=False))

# Check for leftover SRT timestamps
timestamp_pattern = (
    r"\b\d{2}:\d{2}:\d{2}[,.]\d{3}\s*-->"
)

timestamp_count = df["dialogue_for_bertopic"].str.contains(
    timestamp_pattern, regex=True
).sum()

print("\nDocuments containing SRT timestamps:", timestamp_count)

# Confirm the recovered document
street = df.loc[
    df["movie_record_id"].astype(str).eq("2020_5")
]

assert len(street) == 1
assert int(street["actual_word_count"].iloc[0]) == 11331

print("\nStreet Dancer 3D recovery: PASS")
print("Validation complete.")

In [ ]:
from pathlib import Path
import pandas as pd
import re

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

OUT = ROOT / "06_outputs" / "topic_modelling"

INPUT = OUT / "ICMB_130_BERTOPIC_DIALOGUE_FINAL_REVIEW.csv"
OUTPUT = OUT / "ICMB_130_BERTOPIC_CHUNKS_300W.csv"

CHUNK_SIZE = 300
OVERLAP = 50
STEP_SIZE = CHUNK_SIZE - OVERLAP

df = pd.read_csv(INPUT)

required_columns = [
    "movie_record_id",
    "movie_title",
    "year",
    "dialogue_for_bertopic",
]

missing = [c for c in required_columns if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if len(df) != 130 or df["movie_record_id"].nunique() != 130:
    raise ValueError("Expected 130 unique movie records.")

chunks = []

for _, row in df.iterrows():
    text = str(row["dialogue_for_bertopic"])
    text = text.replace("\x00", " ")
    words = re.findall(r"\S+", text)

    if len(words) < 100:
        raise ValueError(
            f"Unexpectedly short dialogue: {row['movie_record_id']}"
        )

    chunk_number = 0

    for start in range(0, len(words), STEP_SIZE):
        chunk_words = words[start:start + CHUNK_SIZE]

        if not chunk_words:
            continue

        # Avoid creating a tiny trailing chunk.
        if len(chunk_words) < 100 and chunks:
            # Merge with the preceding chunk from this same film.
            if (
                chunks[-1]["movie_record_id"]
                == str(row["movie_record_id"])
            ):
                chunks[-1]["text"] += " " + " ".join(chunk_words)
                chunks[-1]["word_count"] = len(
                    chunks[-1]["text"].split()
                )
                break

        chunk_number += 1

        chunks.append({
            "movie_record_id": str(row["movie_record_id"]),
            "movie_title": row["movie_title"],
            "year": int(row["year"]),
            "chunk_id": (
                f"{row['movie_record_id']}_chunk_{chunk_number:03d}"
            ),
            "chunk_number": chunk_number,
            "word_count": len(chunk_words),
            "text": " ".join(chunk_words),
        })

        if start + CHUNK_SIZE >= len(words):
            break

chunk_df = pd.DataFrame(chunks)

if chunk_df.empty:
    raise ValueError("No chunks were generated.")

if chunk_df["chunk_id"].duplicated().any():
    raise ValueError("Duplicate chunk IDs detected.")

if chunk_df["text"].isna().any():
    raise ValueError("Missing chunk text detected.")

chunk_df.to_csv(OUTPUT, index=False)

# Reopen and verify the saved file.
check = pd.read_csv(OUTPUT)

print("=" * 65)
print("ICMB-130 — BERTopic CHUNK PREPARATION")
print("=" * 65)
print("Source films:", df["movie_record_id"].nunique())
print("Saved chunks:", len(check))
print("Films represented:", check["movie_record_id"].nunique())
print("Duplicate chunk IDs:", check["chunk_id"].duplicated().sum())
print("Missing chunk text:", check["text"].isna().sum())
print("Minimum chunk words:", check["word_count"].min())
print("Median chunk words:", int(check["word_count"].median()))
print("Maximum chunk words:", check["word_count"].max())
print("Output file:", OUTPUT)

print("\nChunks per film — summary:")
print(
    check.groupby("movie_record_id").size()
    .describe()
    .round(2)
    .to_string()
)

assert check["movie_record_id"].nunique() == 130
assert check["chunk_id"].is_unique
assert check["text"].notna().all()
assert (check["word_count"] >= 100).all()

print("\nPASS: chunk dataset saved and verified.")

In [ ]:
import sys
import importlib.metadata as metadata
import importlib.util

print("=" * 60)
print("BERTOPIC ENVIRONMENT CHECK")
print("=" * 60)

print("Python version:", sys.version.split()[0])

packages = [
    "bertopic",
    "sentence-transformers",
    "transformers",
    "torch",
    "umap-learn",
    "hdbscan",
    "scikit-learn",
    "pandas",
    "numpy",
]

for package in packages:
    try:
        version = metadata.version(package)
        print(f"{package}: {version}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

try:
    import torch
    print("\nPyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("\nPyTorch check failed:", repr(exc))

print("\nEnvironment check complete.")

In [ ]:
%pip install bertopic

In [ ]:
import sys
import importlib.metadata as metadata
import bertopic
import sentence_transformers
import torch
import umap
import hdbscan
import sklearn
import numpy as np

print("=" * 55)
print("BERTOPIC ENVIRONMENT — FINAL CHECK")
print("=" * 55)

print("Python:", sys.version.split()[0])

for package in [
    "bertopic",
    "sentence-transformers",
    "transformers",
    "torch",
    "umap-learn",
    "hdbscan",
    "scikit-learn",
    "pandas",
    "numpy",
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: version unavailable")

print("\nImports:")
print("BERTopic:", bertopic.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("PyTorch:", torch.__version__)
print("UMAP:", umap.__version__)
print("HDBSCAN import: PASS")
print("Scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("CUDA available:", torch.cuda.is_available())

print("\nPASS: required packages imported successfully.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

CHUNK_PATH = (
    ROOT / "06_outputs" / "topic_modelling"
    / "ICMB_130_BERTOPIC_CHUNKS_300W.csv"
)

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

chunks = pd.read_csv(CHUNK_PATH)

assert len(chunks) == 5888
assert chunks["text"].notna().all()

# Small sample first; no full-corpus embeddings yet.
sample = chunks.sample(n=32, random_state=42)
sample_texts = sample["text"].astype(str).tolist()

print("Loading model:", MODEL_NAME)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

model = SentenceTransformer(
    MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

embeddings = model.encode(
    sample_texts,
    batch_size=8,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\nEmbedding test results")
print("Sample chunks:", len(sample_texts))
print("Embedding shape:", embeddings.shape)
print("All values finite:", bool(np.isfinite(embeddings).all()))
print("Embedding norms (first 5):",
      np.round(np.linalg.norm(embeddings, axis=1)[:5], 4))

assert embeddings.shape[0] == 32
assert np.isfinite(embeddings).all()

print("\nPASS: multilingual embedding test completed.")

In [ ]:
print("Model:", MODEL_NAME)
print("Maximum input sequence length:", model.max_seq_length)

tokenizer = model.tokenizer

for i in range(3):
    text = sample_texts[i]
    tokens = tokenizer(
        text,
        truncation=False,
        add_special_tokens=True
    )["input_ids"]

    print(
        f"Sample {i + 1}: {len(tokens)} tokens before truncation"
    )

print("\nInput-length diagnostic complete.")

In [ ]:
from pathlib import Path
import pandas as pd
import re

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

OUT = ROOT / "06_outputs" / "topic_modelling"

INPUT = OUT / "ICMB_130_BERTOPIC_DIALOGUE_FINAL_REVIEW.csv"
OUTPUT = OUT / "ICMB_130_BERTOPIC_CHUNKS_80W.csv"

CHUNK_SIZE = 80
OVERLAP = 15
STEP_SIZE = CHUNK_SIZE - OVERLAP

df = pd.read_csv(INPUT)

chunks = []

for _, row in df.iterrows():
    text = str(row["dialogue_for_bertopic"])
    text = text.replace("\x00", " ")
    words = re.findall(r"\S+", text)

    chunk_number = 0

    for start in range(0, len(words), STEP_SIZE):
        chunk_words = words[start:start + CHUNK_SIZE]

        if not chunk_words:
            continue

        # Skip a very small trailing chunk; the previous chunk
        # already contains overlapping context.
        if len(chunk_words) < 20:
            break

        chunk_number += 1

        chunks.append({
            "movie_record_id": str(row["movie_record_id"]),
            "movie_title": row["movie_title"],
            "year": int(row["year"]),
            "chunk_id": (
                f"{row['movie_record_id']}_chunk_{chunk_number:03d}"
            ),
            "chunk_number": chunk_number,
            "word_count": len(chunk_words),
            "text": " ".join(chunk_words),
        })

        if start + CHUNK_SIZE >= len(words):
            break

chunk_df = pd.DataFrame(chunks)
chunk_df.to_csv(OUTPUT, index=False)

# Reopen and verify
check = pd.read_csv(OUTPUT)

print("Films represented:", check["movie_record_id"].nunique())
print("Total chunks:", len(check))
print("Duplicate chunk IDs:", check["chunk_id"].duplicated().sum())
print("Missing text:", check["text"].isna().sum())
print("Minimum words:", check["word_count"].min())
print("Median words:", int(check["word_count"].median()))
print("Maximum words:", check["word_count"].max())
print("Output:", OUTPUT)

assert check["movie_record_id"].nunique() == 130
assert check["chunk_id"].is_unique
assert check["text"].notna().all()

print("\nPASS: smaller chunks saved and verified.")

In [ ]:
import pandas as pd

check = pd.read_csv(OUTPUT)
sample80 = check.sample(n=32, random_state=42)

token_counts = [
    len(
        model.tokenizer(
            text,
            truncation=False,
            add_special_tokens=True
        )["input_ids"]
    )
    for text in sample80["text"].astype(str)
]

print("Model token limit:", model.max_seq_length)
print("Sampled chunks:", len(token_counts))
print("Maximum sampled tokens:", max(token_counts))
print("Median sampled tokens:", int(pd.Series(token_counts).median()))
print("Chunks over model limit:", sum(
    count > model.max_seq_length for count in token_counts
))

In [ ]:
from pathlib import Path
import pandas as pd
import re

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)

OUT = ROOT / "06_outputs" / "topic_modelling"

INPUT = OUT / "ICMB_130_BERTOPIC_DIALOGUE_FINAL_REVIEW.csv"
OUTPUT = OUT / "ICMB_130_BERTOPIC_CHUNKS_55W.csv"

CHUNK_SIZE = 55
OVERLAP = 10
STEP_SIZE = CHUNK_SIZE - OVERLAP

df = pd.read_csv(INPUT)
chunks = []

for _, row in df.iterrows():
    text = str(row["dialogue_for_bertopic"]).replace("\x00", " ")
    words = re.findall(r"\S+", text)
    chunk_number = 0

    for start in range(0, len(words), STEP_SIZE):
        chunk_words = words[start:start + CHUNK_SIZE]

        if len(chunk_words) < 15:
            break

        chunk_number += 1
        chunks.append({
            "movie_record_id": str(row["movie_record_id"]),
            "movie_title": row["movie_title"],
            "year": int(row["year"]),
            "chunk_id": f"{row['movie_record_id']}_chunk_{chunk_number:03d}",
            "chunk_number": chunk_number,
            "word_count": len(chunk_words),
            "text": " ".join(chunk_words),
        })

        if start + CHUNK_SIZE >= len(words):
            break

chunk_df = pd.DataFrame(chunks)
chunk_df.to_csv(OUTPUT, index=False)

check = pd.read_csv(OUTPUT)

print("Films represented:", check["movie_record_id"].nunique())
print("Total chunks:", len(check))
print("Duplicate chunk IDs:", check["chunk_id"].duplicated().sum())
print("Minimum words:", check["word_count"].min())
print("Median words:", int(check["word_count"].median()))
print("Maximum words:", check["word_count"].max())
print("Output:", OUTPUT)

assert check["movie_record_id"].nunique() == 130
assert check["chunk_id"].is_unique
assert check["text"].notna().all()

print("\nPASS: 55-word chunks saved.")

In [ ]:
import pandas as pd

check = pd.read_csv(OUTPUT)
sample55 = check.sample(n=100, random_state=42)

token_counts = [
    len(model.tokenizer(
        text,
        truncation=False,
        add_special_tokens=True
    )["input_ids"])
    for text in sample55["text"].astype(str)
]

print("Model token limit:", model.max_seq_length)
print("Sampled chunks:", len(token_counts))
print("Maximum sampled tokens:", max(token_counts))
print("Median sampled tokens:", int(pd.Series(token_counts).median()))
print("Chunks over model limit:", sum(
    count > model.max_seq_length for count in token_counts
))

In [ ]:
import pandas as pd

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
CHUNK_PATH = (
    ROOT / "06_outputs" / "topic_modelling"
    / "ICMB_130_BERTOPIC_CHUNKS_55W.csv"
)

check = pd.read_csv(CHUNK_PATH)

# Reuse token_counts from the previous validation cell.
check["token_count"] = token_counts

outliers = (
    check.loc[
        check["token_count"] > model.max_seq_length,
        [
            "movie_record_id",
            "movie_title",
            "year",
            "chunk_id",
            "word_count",
            "token_count",
            "text",
        ],
    ]
    .sort_values("token_count", ascending=False)
)

print("Outlier chunks:", len(outliers))
print("\nToken count distribution for outliers:")
print(outliers["token_count"].describe().round(1).to_string())

print("\nTop 10 longest-tokenized chunks:")
for _, row in outliers.head(10).iterrows():
    print("\n" + "-" * 60)
    print(
        row["chunk_id"],
        "| words:", row["word_count"],
        "| tokens:", row["token_count"],
    )
    print(str(row["text"])[:400])

In [ ]:
import pandas as pd
import re

# Uses `check` and `token_counts` from the previous validation cell.
check["token_count"] = token_counts

outliers = check[check["token_count"] > model.max_seq_length].copy()

def artifact_flags(text):
    text = str(text)
    return pd.Series({
        "has_spaced_letters": bool(
            re.search(r"\b(?:[A-Za-z]\s+){3,}[A-Za-z]\b", text)
        ),
        "has_long_punctuation": bool(
            re.search(r"[<>]{8,}|[!?.,]{8,}", text)
        ),
        "has_repeated_chars": bool(
            re.search(r"(.)\1{4,}", text, flags=re.IGNORECASE)
        ),
        "has_credit_text": bool(
            re.search(r"(?i)copyright|subtitle|transcribed|telegram", text)
        ),
    })

flags = outliers["text"].apply(artifact_flags)
outliers = pd.concat([outliers, flags], axis=1)

print("Overlength chunks:", len(outliers))
print("\nOverlength chunks by film:")
print(
    outliers.groupby(["movie_record_id", "movie_title"])
    .size()
    .sort_values(ascending=False)
    .head(15)
    .to_string()
)

print("\nArtifact flag counts:")
print(flags.sum().to_string())

print("\nExamples with punctuation or repeated-character artifacts:")
for _, row in outliers[
    outliers["has_long_punctuation"] | outliers["has_repeated_chars"]
].head(8).iterrows():
    print(
        f"\n{row['chunk_id']} | {row['token_count']} tokens"
    )
    print(str(row["text"])[:300])

In [ ]:
import re
import pandas as pd

def clean_for_embedding(text):
    text = str(text).replace("\x00", " ")

    # Remove obvious runs of decorative punctuation.
    text = re.sub(r"[<>]{5,}", " ", text)
    text = re.sub(r"[!?.,]{6,}", " ", text)

    # Collapse extreme character repetitions, e.g. "WWWWWWhhheeemnn".
    # This is only a model-input normalization, not a source correction.
    text = re.sub(
        r"([A-Za-z])\1{3,}",
        r"\1",
        text,
        flags=re.IGNORECASE
    )

    # Normalize whitespace.
    text = re.sub(r"\s+", " ", text).strip()
    return text


examples = check.loc[
    check["token_count"] > model.max_seq_length
].sort_values("token_count", ascending=False).head(12).copy()

print("BEFORE / AFTER TOKEN COUNTS")
print("=" * 65)

for _, row in examples.iterrows():
    cleaned = clean_for_embedding(row["text"])

    before = len(
        model.tokenizer(
            row["text"],
            truncation=False,
            add_special_tokens=True
        )["input_ids"]
    )

    after = len(
        model.tokenizer(
            cleaned,
            truncation=False,
            add_special_tokens=True
        )["input_ids"]
    )

    print(
        f"{row['chunk_id']}: "
        f"{before} -> {after} tokens"
    )
    print("Cleaned preview:", cleaned[:180])
    print("-" * 65)

In [ ]:
from pathlib import Path
import pandas as pd
import re

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
OUT = ROOT / "06_outputs" / "topic_modelling"

INPUT = OUT / "ICMB_130_BERTOPIC_CHUNKS_55W.csv"
OUTPUT = OUT / "ICMB_130_BERTOPIC_TOKEN_SAFE_CHUNKS.csv"

MAX_TOKENS = model.max_seq_length
SAFE_LIMIT = 110

source = pd.read_csv(INPUT)
safe_chunks = []

def clean_obvious_artifacts(text):
    text = str(text).replace("\x00", " ")

    # Remove long decorative punctuation runs.
    text = re.sub(r"[<>]{5,}", " ", text)
    text = re.sub(r"[!?.,]{6,}", " ", text)

    # Remove the known spaced-letter copyright boilerplate.
    copyright_pattern = (
        r"(?i)c\s*o\s*p\s*y\s*r\s*i\s*g\s*h\s*t\s+"
        r"f\s*r\s*o\s*m\s+"
        r"s\s*i\s*m\s*e\s*d\s*i\s*a"
    )
    text = re.sub(copyright_pattern, " ", text)

    return re.sub(r"\s+", " ", text).strip()

def token_count(text):
    return len(
        model.tokenizer(
            text,
            add_special_tokens=True,
            truncation=False,
        )["input_ids"]
    )

def split_token_safe(text, safe_limit=SAFE_LIMIT):
    words = text.split()
    pieces = []
    current = []

    for word in words:
        candidate = " ".join(current + [word])

        if current and token_count(candidate) > safe_limit:
            pieces.append(" ".join(current))
            current = [word]
        else:
            current.append(word)

        if len(current) == 1 and token_count(current[0]) > safe_limit:
            raise ValueError(
                f"Single word exceeds safe limit: {current[0][:80]!r}"
            )

    if current:
        pieces.append(" ".join(current))

    return pieces

for _, row in source.iterrows():
    cleaned = clean_obvious_artifacts(row["text"])
    pieces = split_token_safe(cleaned)

    for part_number, piece in enumerate(pieces, start=1):
        count = token_count(piece)

        if count > MAX_TOKENS:
            raise ValueError(
                f"Token limit exceeded for {row['chunk_id']}: {count}"
            )

        safe_chunks.append({
            "movie_record_id": str(row["movie_record_id"]),
            "movie_title": row["movie_title"],
            "year": int(row["year"]),
            "source_chunk_id": row["chunk_id"],
            "part_number": part_number,
            "token_count": count,
            "text": piece,
        })

safe_df = pd.DataFrame(safe_chunks)
safe_df.to_csv(OUTPUT, index=False)

# Reopen and verify persistence.
check_safe = pd.read_csv(OUTPUT)

print("=" * 60)
print("TOKEN-SAFE BERTopic INPUT")
print("=" * 60)
print("Source chunks:", len(source))
print("Token-safe pieces:", len(check_safe))
print("Films represented:", check_safe["movie_record_id"].nunique())
print("Maximum tokens:", check_safe["token_count"].max())
print(
    "Pieces over model limit:",
    (check_safe["token_count"] > MAX_TOKENS).sum()
)
print("Missing text:", check_safe["text"].isna().sum())
print("Output:", OUTPUT)

assert check_safe["movie_record_id"].nunique() == 130
assert (check_safe["token_count"] <= MAX_TOKENS).all()
assert check_safe["text"].notna().all()

print("\nPASS: all saved embedding inputs fit the model limit.")

In [ ]:
import pandas as pd
from pathlib import Path
import re

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
INPUT = (
    ROOT / "06_outputs" / "topic_modelling"
    / "ICMB_130_BERTOPIC_TOKEN_SAFE_CHUNKS.csv"
)

df = pd.read_csv(INPUT)

punctuation_only = df["text"].astype(str).map(
    lambda text: bool(re.fullmatch(r"[\W_]+", text.strip()))
)

print("Punctuation-only pieces:", int(punctuation_only.sum()))

print(
    df.loc[
        punctuation_only,
        [
            "movie_record_id",
            "movie_title",
            "source_chunk_id",
            "part_number",
            "token_count",
            "text",
        ],
    ].to_string(index=False)
)

In [ ]:
from pathlib import Path
import pandas as pd
import re

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
OUT = ROOT / "06_outputs" / "topic_modelling"

INPUT = OUT / "ICMB_130_BERTOPIC_TOKEN_SAFE_CHUNKS.csv"
OUTPUT = OUT / "ICMB_130_BERTOPIC_EMBEDDING_INPUT.csv"

df = pd.read_csv(INPUT)
text = df["text"].fillna("").astype(str)

# Unicode-aware: retain any text containing a letter or digit
# from any script; exclude punctuation/music-only pieces.
has_letter_or_digit = text.map(
    lambda value: bool(re.search(r"[\w]", value, flags=re.UNICODE))
)

excluded = df.loc[~has_letter_or_digit].copy()
final_df = df.loc[has_letter_or_digit].copy()

final_df.to_csv(OUTPUT, index=False)

# Verify saved output
check = pd.read_csv(OUTPUT)

print("=" * 60)
print("FINAL BERTopic EMBEDDING INPUT — CORRECTED")
print("=" * 60)
print("Original pieces:", len(df))
print("Excluded non-dialogue pieces:", len(excluded))
print("Final pieces:", len(check))
print("Films represented:", check["movie_record_id"].nunique())
print("Missing text:", check["text"].isna().sum())
print("Maximum stored token count:", check["token_count"].max())
print("Pieces over 128 tokens:", (check["token_count"] > 128).sum())
print("Output:", OUTPUT)

print("\nExcluded pieces:")
print(
    excluded[
        ["movie_record_id", "movie_title", "source_chunk_id", "text"]
    ].to_string(index=False)
)

assert len(excluded) == 2
assert check["movie_record_id"].nunique() == 130
assert check["text"].notna().all()
assert check["text"].astype(str).str.strip().ne("").all()
assert (check["token_count"] <= 128).all()

print("\nPASS: final embedding input saved and verified.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import time

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
OUT = ROOT / "06_outputs" / "topic_modelling"

INPUT = OUT / "ICMB_130_BERTOPIC_EMBEDDING_INPUT.csv"
CHECKPOINT_DIR = OUT / "bertopic_embedding_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = CHECKPOINT_DIR / "embedding_input_manifest.csv"
CONFIG_PATH = CHECKPOINT_DIR / "embedding_config.json"

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
BATCH_ROWS = 256
ENCODE_BATCH_SIZE = 32

df = pd.read_csv(INPUT)

required = [
    "movie_record_id", "movie_title", "year",
    "source_chunk_id", "part_number", "text"
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

if df["text"].isna().any() or df["text"].astype(str).str.strip().eq("").any():
    raise ValueError("Empty text found in embedding input.")

if df["movie_record_id"].nunique() != 130:
    raise ValueError("Expected all 130 films.")

# Verify token lengths using the loaded model's tokenizer.
token_counts = [
    len(model.tokenizer(
        text,
        add_special_tokens=True,
        truncation=False
    )["input_ids"])
    for text in df["text"].astype(str)
]

if max(token_counts) > model.max_seq_length:
    raise ValueError(
        f"Input exceeds model limit: {max(token_counts)} tokens."
    )

# The manifest fixes the row order used for every embedding batch.
manifest = df[
    ["movie_record_id", "movie_title", "year",
     "source_chunk_id", "part_number"]
].copy()
manifest.insert(0, "embedding_row", np.arange(len(manifest)))

if MANIFEST_PATH.exists():
    old_manifest = pd.read_csv(MANIFEST_PATH)
    if not old_manifest.equals(manifest):
        raise ValueError(
            "Existing checkpoint manifest differs from current input. "
            "Use a new checkpoint folder rather than mixing embeddings."
        )
else:
    manifest.to_csv(MANIFEST_PATH, index=False)

config = {
    "model_name": MODEL_NAME,
    "max_seq_length": int(model.max_seq_length),
    "embedding_dimension_expected": 384,
    "input_rows": len(df),
    "batch_rows": BATCH_ROWS,
    "encode_batch_size": ENCODE_BATCH_SIZE,
}

if CONFIG_PATH.exists():
    old_config = json.loads(CONFIG_PATH.read_text())
    for key in [
        "model_name", "max_seq_length", "input_rows", "batch_rows"
    ]:
        if old_config.get(key) != config.get(key):
            raise ValueError(
                f"Checkpoint configuration mismatch for {key}. "
                "Do not mix incompatible batches."
            )
else:
    CONFIG_PATH.write_text(json.dumps(config, indent=2))

total_batches = (len(df) + BATCH_ROWS - 1) // BATCH_ROWS
started = time.time()

print("=" * 65)
print("CHECKPOINTED EMBEDDING RUN")
print("=" * 65)
print("Input rows:", len(df))
print("Batch rows:", BATCH_ROWS)
print("Total batches:", total_batches)
print("Device:", str(model.device))
print("Checkpoint folder:", CHECKPOINT_DIR)

for batch_num, start in enumerate(range(0, len(df), BATCH_ROWS)):
    end = min(start + BATCH_ROWS, len(df))
    batch_path = CHECKPOINT_DIR / f"batch_{batch_num:04d}.npy"
    expected_shape = (end - start, 384)

    # Resume only from a checkpoint with the expected shape.
    if batch_path.exists():
        try:
            saved = np.load(batch_path, mmap_mode="r")
            if saved.shape == expected_shape and np.isfinite(saved).all():
                print(
                    f"Batch {batch_num + 1}/{total_batches}: "
                    f"already verified; skipping."
                )
                continue
        except Exception:
            pass

        print(f"Batch {batch_num + 1}: replacing invalid checkpoint.")

    batch_texts = df["text"].iloc[start:end].astype(str).tolist()

    embeddings = model.encode(
        batch_texts,
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    embeddings = np.asarray(embeddings, dtype=np.float32)

    if embeddings.shape != expected_shape:
        raise ValueError(
            f"Unexpected embedding shape in batch {batch_num}: "
            f"{embeddings.shape}; expected {expected_shape}"
        )

    if not np.isfinite(embeddings).all():
        raise ValueError(f"Non-finite embedding in batch {batch_num}.")

    temp_path = batch_path.with_suffix(".tmp.npy")
    np.save(temp_path, embeddings)

    # Verify the saved checkpoint before publishing it.
    verify = np.load(temp_path, mmap_mode="r")
    if verify.shape != expected_shape or not np.isfinite(verify).all():
        raise ValueError(f"Checkpoint verification failed: {batch_num}")

    temp_path.replace(batch_path)

    elapsed = (time.time() - started) / 60
    print(
        f"Batch {batch_num + 1}/{total_batches} saved "
        f"(rows {start}–{end - 1}); elapsed {elapsed:.1f} min."
    )

print("\nEmbedding batch loop finished.")
print("Run the verification cell below before fitting BERTopic.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
OUT = ROOT / "06_outputs" / "topic_modelling"
CHECKPOINT_DIR = OUT / "bertopic_embedding_checkpoints"

df = pd.read_csv(
    OUT / "ICMB_130_BERTOPIC_EMBEDDING_INPUT.csv"
)

BATCH_ROWS = 256
total_batches = (len(df) + BATCH_ROWS - 1) // BATCH_ROWS

verified_rows = 0
errors = []

for batch_num, start in enumerate(range(0, len(df), BATCH_ROWS)):
    end = min(start + BATCH_ROWS, len(df))
    path = CHECKPOINT_DIR / f"batch_{batch_num:04d}.npy"

    if not path.exists():
        errors.append(f"Missing batch {batch_num}: {path.name}")
        continue

    arr = np.load(path, mmap_mode="r")
    expected_shape = (end - start, 384)

    if arr.shape != expected_shape:
        errors.append(
            f"{path.name}: shape {arr.shape}, expected {expected_shape}"
        )
        continue

    if not np.isfinite(arr).all():
        errors.append(f"{path.name}: non-finite values")

    verified_rows += arr.shape[0]

print("Input rows:", len(df))
print("Expected batches:", total_batches)
print("Verified embedding rows:", verified_rows)
print("Errors:", len(errors))

for error in errors[:20]:
    print("-", error)

if errors or verified_rows != len(df):
    raise ValueError(
        "Embedding checkpoints are incomplete or invalid. "
        "Fix the reported issue before continuing."
    )

print("\nPASS: all embedding checkpoints are verified.")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import gc
import time

from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

ROOT = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics"
)
OUT = ROOT / "06_outputs" / "topic_modelling"
INPUT = OUT / "ICMB_130_BERTOPIC_EMBEDDING_INPUT.csv"
CHECKPOINT_DIR = OUT / "bertopic_embedding_checkpoints"

ASSIGNMENTS_PATH = OUT / "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
TOPICS_PATH = OUT / "BERTOPIC_TOPIC_INFO.csv"
TERMS_PATH = OUT / "BERTOPIC_TOPIC_TERMS.csv"

# 1. Load the exact input rows used to create the embeddings.
df = pd.read_csv(INPUT)
docs = df["text"].astype(str).tolist()

assert len(docs) == 33800
assert df["movie_record_id"].nunique() == 130

# 2. Reassemble the verified embedding batches in row order.
BATCH_ROWS = 256
embedding_batches = []

for batch_num, start in enumerate(range(0, len(df), BATCH_ROWS)):
    end = min(start + BATCH_ROWS, len(df))
    batch_path = CHECKPOINT_DIR / f"batch_{batch_num:04d}.npy"

    if not batch_path.exists():
        raise FileNotFoundError(f"Missing embedding batch: {batch_path}")

    arr = np.load(batch_path)

    expected_shape = (end - start, 384)
    if arr.shape != expected_shape:
        raise ValueError(
            f"Batch {batch_num} has shape {arr.shape}; "
            f"expected {expected_shape}"
        )

    if not np.isfinite(arr).all():
        raise ValueError(f"Invalid values in batch {batch_num}")

    embedding_batches.append(arr)

embeddings = np.concatenate(embedding_batches, axis=0)
del embedding_batches
gc.collect()

assert embeddings.shape == (len(df), 384)
assert np.isfinite(embeddings).all()

print("Documents:", len(docs))
print("Embedding matrix:", embeddings.shape)

# 3. Configure the baseline topic model.
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
    low_memory=True,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.80,
    max_features=30000,
)

topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True,
)

# 4. Fit using precomputed embeddings.
start_time = time.time()

topics, probabilities = topic_model.fit_transform(
    docs,
    embeddings=embeddings,
)

elapsed_minutes = (time.time() - start_time) / 60

# 5. Save chunk-level topic assignments.
assignments = df.copy()
assignments["topic"] = topics
assignments.to_csv(ASSIGNMENTS_PATH, index=False)

# 6. Save topic summary and top terms.
topic_info = topic_model.get_topic_info()
topic_info.to_csv(TOPICS_PATH, index=False)

term_rows = []

for topic_id in topic_info["Topic"].tolist():
    if topic_id == -1:
        continue

    terms = topic_model.get_topic(topic_id) or []
    for rank, (term, score) in enumerate(terms, start=1):
        term_rows.append({
            "topic": topic_id,
            "term_rank": rank,
            "term": term,
            "c_tf_idf_score": score,
        })

terms_df = pd.DataFrame(term_rows)
terms_df.to_csv(TERMS_PATH, index=False)

print("\n" + "=" * 65)
print("BERTOPIC BASELINE FIT COMPLETE")
print("=" * 65)
print("Elapsed minutes:", round(elapsed_minutes, 2))
print("Input chunks:", len(assignments))
print("Unique assigned topic IDs:", assignments["topic"].nunique())
print("Discovered non-outlier topics:", (set(topics) - {-1}).__len__())
print("Outlier chunks:", int((assignments["topic"] == -1).sum()))
print("Outlier percentage:", round((assignments["topic"] == -1).mean() * 100, 2))
print("Topic summary:", TOPICS_PATH)
print("Topic terms:", TERMS_PATH)
print("Chunk assignments:", ASSIGNMENTS_PATH)

print("\nTop topic summary:")
print(topic_info.head(15).to_string(index=False))

print("\nPASS: BERTopic baseline outputs saved.")

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

# ------------------------------------------------------------
# 1. LOAD SAVED BERTOPIC RESULTS
# ------------------------------------------------------------
OUTPUT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"
)

assignments_path = OUTPUT_DIR / "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
topic_info_path = OUTPUT_DIR / "BERTOPIC_TOPIC_INFO.csv"

assignments_df = pd.read_csv(assignments_path)
topic_info_df = pd.read_csv(topic_info_path)

print("=" * 70)
print("BERTOPIC BASELINE — CLUSTERING DIAGNOSTICS")
print("=" * 70)

print("\nAssignment file shape:", assignments_df.shape)
print("Assignment columns:", assignments_df.columns.tolist())

# Identify the topic assignment column
topic_col = next(
    (c for c in assignments_df.columns if c.lower() == "topic"),
    None
)

if topic_col is None:
    raise ValueError(
        "Could not identify the topic column. "
        "Review the assignment columns printed above."
    )

# Identify available film-level metadata columns
film_id_col = next(
    (c for c in assignments_df.columns
     if c.lower() == "movie_record_id"),
    None
)
title_col = next(
    (c for c in assignments_df.columns
     if c.lower() in ["movie_title", "title"]),
    None
)
year_col = next(
    (c for c in assignments_df.columns
     if c.lower() == "year"),
    None
)

if film_id_col is None:
    raise ValueError(
        "movie_record_id is missing from the assignment file."
    )

# ------------------------------------------------------------
# 2. OVERALL TOPIC SIZE DISTRIBUTION
# ------------------------------------------------------------
topic_counts = (
    assignments_df[topic_col]
    .value_counts()
    .rename_axis("topic_id")
    .reset_index(name="chunk_count")
)

topic_counts["percentage_of_all_chunks"] = (
    100 * topic_counts["chunk_count"] / len(assignments_df)
)

outlier_count = int((assignments_df[topic_col] == -1).sum())
outlier_pct = 100 * outlier_count / len(assignments_df)

non_outlier_sizes = topic_counts.loc[
    topic_counts["topic_id"] != -1, "chunk_count"
]

print("\n--- Overall clustering ---")
print("Total chunks:", len(assignments_df))
print("Outlier chunks:", outlier_count)
print(f"Outlier percentage: {outlier_pct:.2f}%")
print("Non-outlier topics:", assignments_df.loc[
    assignments_df[topic_col] != -1, topic_col
].nunique())
print("Smallest non-outlier topic:", int(non_outlier_sizes.min()))
print("Median non-outlier topic size:", int(non_outlier_sizes.median()))
print("Largest non-outlier topic:", int(non_outlier_sizes.max()))

print("\n--- 15 smallest non-outlier topics ---")
display(
    topic_counts[topic_counts["topic_id"] != -1]
    .sort_values("chunk_count")
    .head(15)
    .round(3)
)

# ------------------------------------------------------------
# 3. OUTLIER RATE BY FILM
# ------------------------------------------------------------
group_cols = [film_id_col]
if title_col:
    group_cols.append(title_col)
if year_col:
    group_cols.append(year_col)

film_diagnostics = (
    assignments_df
    .assign(is_outlier=assignments_df[topic_col].eq(-1))
    .groupby(group_cols, dropna=False)
    .agg(
        total_chunks=(topic_col, "size"),
        outlier_chunks=("is_outlier", "sum"),
        assigned_topics=(topic_col, lambda x: x[x != -1].nunique())
    )
    .reset_index()
)

film_diagnostics["outlier_pct"] = (
    100 * film_diagnostics["outlier_chunks"]
    / film_diagnostics["total_chunks"]
)

print("\n--- 15 films with the highest outlier rate ---")
display(
    film_diagnostics
    .sort_values(["outlier_pct", "total_chunks"], ascending=[False, False])
    .head(15)
    .round(2)
)

print("\n--- 15 films with the lowest outlier rate ---")
display(
    film_diagnostics
    .sort_values(["outlier_pct", "total_chunks"], ascending=[True, False])
    .head(15)
    .round(2)
)

# ------------------------------------------------------------
# 4. OUTLIER RATE BY YEAR
# ------------------------------------------------------------
if year_col:
    year_diagnostics = (
        assignments_df
        .assign(is_outlier=assignments_df[topic_col].eq(-1))
        .groupby(year_col, dropna=False)
        .agg(
            total_chunks=(topic_col, "size"),
            outlier_chunks=("is_outlier", "sum"),
            distinct_assigned_topics=(
                topic_col,
                lambda x: x[x != -1].nunique()
            )
        )
        .reset_index()
    )

    year_diagnostics["outlier_pct"] = (
        100 * year_diagnostics["outlier_chunks"]
        / year_diagnostics["total_chunks"]
    )

    print("\n--- Outlier rate by year ---")
    display(year_diagnostics.round(2))

print("\n--- Topic summary file columns ---")
print(topic_info_df.columns.tolist())

print("\nDiagnostics complete. No model parameters or source files changed.")


In [ ]:

import pandas as pd
import numpy as np
import time
from pathlib import Path
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

# ------------------------------------------------------------
# BERTOPIC EXPERIMENT 2 — MORE INCLUSIVE CLUSTERING
# Reuse the saved embeddings; do not re-embed.
# ------------------------------------------------------------
OUTPUT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"
)
INPUT_PATH = OUTPUT_DIR / "ICMB_130_BERTOPIC_EMBEDDING_INPUT.csv"
CHECKPOINT_DIR = OUTPUT_DIR / "bertopic_embedding_checkpoints"

chunks_df = pd.read_csv(INPUT_PATH)
docs = chunks_df["text"].fillna("").astype(str).tolist()

# Reassemble saved embedding batches in original input order
BATCH_ROWS = 256
N_ROWS = len(chunks_df)
EMBEDDING_DIM = 384

batch_files = sorted(CHECKPOINT_DIR.glob("batch_*.npy"))
expected_batches = int(np.ceil(N_ROWS / BATCH_ROWS))

if len(batch_files) != expected_batches:
    raise ValueError(
        f"Expected {expected_batches} embedding batches, "
        f"found {len(batch_files)}."
    )

embedding_parts = [np.load(path) for path in batch_files]
embeddings = np.vstack(embedding_parts)

if embeddings.shape != (N_ROWS, EMBEDDING_DIM):
    raise ValueError(f"Unexpected embedding shape: {embeddings.shape}")

if not np.isfinite(embeddings).all():
    raise ValueError("Embedding matrix contains non-finite values.")

print("Documents:", len(docs))
print("Embeddings:", embeddings.shape)

# ------------------------------------------------------------
# MODEL CONFIGURATION
# ------------------------------------------------------------
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
    low_memory=True
)

# More inclusive than the baseline: min_cluster_size 30 vs 50;
# min_samples 5 vs 10.
hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.80,
    max_features=30000
)

topic_model_v2 = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=False,
    verbose=True
)

# ------------------------------------------------------------
# FIT USING SAVED EMBEDDINGS
# ------------------------------------------------------------
start_time = time.time()

topics_v2, probabilities_v2 = topic_model_v2.fit_transform(
    docs,
    embeddings=embeddings
)

elapsed_minutes = (time.time() - start_time) / 60

chunks_v2 = chunks_df.copy()
chunks_v2["topic"] = topics_v2

topic_info_v2 = topic_model_v2.get_topic_info()

terms_rows = []
for topic_id in topic_model_v2.get_topics():
    if topic_id == -1:
        continue
    terms = topic_model_v2.get_topic(topic_id) or []
    for rank, (term, score) in enumerate(terms, start=1):
        terms_rows.append({
            "topic_id": topic_id,
            "term_rank": rank,
            "term": term,
            "term_score": score
        })

terms_v2 = pd.DataFrame(terms_rows)

# ------------------------------------------------------------
# SAVE AS SEPARATE EXPERIMENT 2 FILES
# ------------------------------------------------------------
assignments_path = OUTPUT_DIR / "BERTOPIC_V2_CHUNK_TOPIC_ASSIGNMENTS.csv"
info_path = OUTPUT_DIR / "BERTOPIC_V2_TOPIC_INFO.csv"
terms_path = OUTPUT_DIR / "BERTOPIC_V2_TOPIC_TERMS.csv"

chunks_v2.to_csv(assignments_path, index=False)
topic_info_v2.to_csv(info_path, index=False)
terms_v2.to_csv(terms_path, index=False)

# ------------------------------------------------------------
# REPORT RESULTS
# ------------------------------------------------------------
outlier_count_v2 = int(np.sum(np.array(topics_v2) == -1))
outlier_pct_v2 = 100 * outlier_count_v2 / len(topics_v2)
non_outlier_topics_v2 = len(set(topics_v2) - {-1})

print("\n" + "=" * 68)
print("BERTOPIC EXPERIMENT 2 COMPLETE")
print("=" * 68)
print(f"Elapsed minutes: {elapsed_minutes:.2f}")
print(f"Input chunks: {len(chunks_v2)}")
print(f"Non-outlier topics: {non_outlier_topics_v2}")
print(f"Outlier chunks: {outlier_count_v2}")
print(f"Outlier percentage: {outlier_pct_v2:.2f}%")

print("\nTop topic summary:")
display(
    topic_info_v2.head(16)[
        ["Topic", "Count", "Name", "Representation"]
    ]
)

print("\nSaved files:")
print(assignments_path)
print(info_path)
print(terms_path)


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"
)

baseline = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
)
experiment2 = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_V2_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

def topic_diagnostics(df, experiment_name):
    counts = df["topic"].value_counts()
    outliers = int(counts.get(-1, 0))
    assigned = len(df) - outliers
    topic_sizes = counts[counts.index != -1]

    summary = {
        "experiment": experiment_name,
        "total_chunks": len(df),
        "outlier_chunks": outliers,
        "outlier_pct": round(100 * outliers / len(df), 2),
        "assigned_chunks": assigned,
        "assigned_pct": round(100 * assigned / len(df), 2),
        "non_outlier_topics": len(topic_sizes),
        "smallest_topic": int(topic_sizes.min()),
        "median_topic_size": float(topic_sizes.median()),
        "largest_topic": int(topic_sizes.max()),
        "topics_with_under_100_chunks": int((topic_sizes < 100).sum()),
        "topics_with_100_or_more_chunks": int((topic_sizes >= 100).sum())
    }

    return summary, topic_sizes

baseline_summary, baseline_sizes = topic_diagnostics(
    baseline, "Baseline"
)
v2_summary, v2_sizes = topic_diagnostics(
    experiment2, "Experiment 2"
)

comparison = pd.DataFrame([baseline_summary, v2_summary])

print("=" * 70)
print("BERTOPIC EXPERIMENT COMPARISON")
print("=" * 70)
display(comparison)

# ------------------------------------------------------------
# SMALL TOPICS IN EXPERIMENT 2
# ------------------------------------------------------------
info_v2 = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_V2_TOPIC_INFO.csv"
)

small_topic_ids = (
    experiment2.loc[experiment2["topic"] != -1, "topic"]
    .value_counts()
    .loc[lambda x: x < 100]
    .index
)

small_topics = info_v2[
    info_v2["Topic"].isin(small_topic_ids)
][["Topic", "Count", "Name", "Representation"]].sort_values("Count")

print("\n--- Experiment 2 topics with fewer than 100 chunks ---")
if small_topics.empty:
    print("None")
else:
    display(small_topics)

# ------------------------------------------------------------
# TOPIC TERMS FOR A FEW LARGER CLUSTERS
# ------------------------------------------------------------
print("\n--- Experiment 2: largest non-outlier topics ---")
largest_topics = (
    info_v2[info_v2["Topic"] != -1]
    .sort_values("Count", ascending=False)
    .head(20)
)
display(largest_topics[["Topic", "Count", "Name", "Representation"]])

# Save comparison table for reproducibility
comparison_path = OUTPUT_DIR / "BERTOPIC_EXPERIMENT_COMPARISON.csv"
comparison.to_csv(comparison_path, index=False)

print("\nComparison saved to:", comparison_path)


In [ ]:

import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"
)

baseline = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
)
experiment2 = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_V2_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

# Identify the same chunks across both experiments
key_cols = ["movie_record_id", "source_chunk_id", "part_number"]

baseline_outliers = baseline.loc[
    baseline["topic"] == -1, key_cols + ["movie_title", "year", "text"]
].copy()

v2_outliers = experiment2.loc[
    experiment2["topic"] == -1, key_cols
].copy()

baseline_outliers["baseline_outlier"] = True
v2_outliers["v2_outlier"] = True

comparison = baseline[
    key_cols + ["movie_title", "year", "text", "topic"]
].rename(columns={"topic": "baseline_topic"}).merge(
    experiment2[key_cols + ["topic"]].rename(
        columns={"topic": "v2_topic"}
    ),
    on=key_cols,
    how="inner",
    validate="one_to_one"
)

print("=" * 68)
print("OUTLIER OVERLAP AND TEXT REVIEW")
print("=" * 68)

print("Chunks matched:", len(comparison))
print("Outliers in baseline:", int((comparison["baseline_topic"] == -1).sum()))
print("Outliers in Experiment 2:", int((comparison["v2_topic"] == -1).sum()))

both_outliers = (
    (comparison["baseline_topic"] == -1)
    & (comparison["v2_topic"] == -1)
)
baseline_only = (
    (comparison["baseline_topic"] == -1)
    & (comparison["v2_topic"] != -1)
)
v2_only = (
    (comparison["baseline_topic"] != -1)
    & (comparison["v2_topic"] == -1)
)

print("Outliers in both experiments:", int(both_outliers.sum()))
print("Baseline-only outliers:", int(baseline_only.sum()))
print("Experiment-2-only outliers:", int(v2_only.sum()))

print("\n--- 12 examples that are outliers in both experiments ---")
sample_both = comparison.loc[
    both_outliers, ["movie_record_id", "movie_title", "year", "text"]
].sample(
    n=min(12, int(both_outliers.sum())),
    random_state=42
)

for _, row in sample_both.iterrows():
    print(f"\n{row['movie_record_id']} | {row['movie_title']} ({row['year']})")
    print(str(row["text"])[:700])

print("\n--- 8 examples assigned by baseline but outliers in Experiment 2 ---")
sample_v2_only = comparison.loc[
    v2_only, ["movie_record_id", "movie_title", "year", "text"]
].sample(
    n=min(8, int(v2_only.sum())),
    random_state=42
)

for _, row in sample_v2_only.iterrows():
    print(f"\n{row['movie_record_id']} | {row['movie_title']} ({row['year']})")
    print(str(row["text"])[:500])

print("\n--- 8 examples assigned by Experiment 2 but outliers in baseline ---")
sample_baseline_only = comparison.loc[
    baseline_only, ["movie_record_id", "movie_title", "year", "text"]
].sample(
    n=min(8, int(baseline_only.sum())),
    random_state=42
)

for _, row in sample_baseline_only.iterrows():
    print(f"\n{row['movie_record_id']} | {row['movie_title']} ({row['year']})")
    print(str(row["text"])[:500])


In [ ]:

import pandas as pd
import numpy as np
import re
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/gdrive2/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"
)

baseline = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
)
experiment2 = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_V2_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

# Join the two experiments using the unique chunk identity.
key_cols = ["movie_record_id", "source_chunk_id", "part_number"]

df = baseline[
    key_cols + ["movie_title", "year", "text", "topic"]
].rename(columns={"topic": "baseline_topic"})

df = df.merge(
    experiment2[key_cols + ["topic"]].rename(
        columns={"topic": "v2_topic"}
    ),
    on=key_cols,
    how="inner",
    validate="one_to_one"
)

# Count letters from common scripts.
DEVANAGARI = re.compile(r"[\u0900-\u097F]")
TAMIL = re.compile(r"[\u0B80-\u0BFF]")
TELUGU = re.compile(r"[\u0C00-\u0C7F]")
KANNADA = re.compile(r"[\u0C80-\u0CFF]")
LATIN = re.compile(r"[A-Za-z]")

def classify_script(text):
    text = str(text)
    counts = {
        "Devanagari": len(DEVANAGARI.findall(text)),
        "Tamil": len(TAMIL.findall(text)),
        "Telugu": len(TELUGU.findall(text)),
        "Kannada": len(KANNADA.findall(text)),
        "Latin": len(LATIN.findall(text))
    }

    total = sum(counts.values())
    if total == 0:
        return "No recognized letters"

    dominant_script = max(counts, key=counts.get)
    if counts[dominant_script] / total < 0.50:
        return "Mixed / other"

    return dominant_script

df["dominant_script"] = df["text"].apply(classify_script)
df["baseline_outlier"] = df["baseline_topic"].eq(-1)
df["v2_outlier"] = df["v2_topic"].eq(-1)
df["both_outlier"] = df["baseline_outlier"] & df["v2_outlier"]

print("=" * 68)
print("OUTLIER RATE BY DOMINANT SCRIPT")
print("=" * 68)

script_summary = (
    df.groupby("dominant_script")
    .agg(
        chunks=("text", "size"),
        baseline_outliers=("baseline_outlier", "sum"),
        experiment2_outliers=("v2_outlier", "sum"),
        outliers_in_both=("both_outlier", "sum")
    )
    .reset_index()
)

script_summary["baseline_outlier_pct"] = (
    100 * script_summary["baseline_outliers"] / script_summary["chunks"]
)
script_summary["experiment2_outlier_pct"] = (
    100 * script_summary["experiment2_outliers"] / script_summary["chunks"]
)
script_summary["both_outlier_pct"] = (
    100 * script_summary["outliers_in_both"] / script_summary["chunks"]
)

display(
    script_summary.sort_values("chunks", ascending=False).round(2)
)

print("\n--- Film-level script distribution ---")
film_script_summary = (
    df.groupby(["movie_record_id", "movie_title", "year", "dominant_script"])
    .size()
    .reset_index(name="chunk_count")
)

display(
    film_script_summary.sort_values("chunk_count", ascending=False).head(20)
)

print("\nScript diagnostic complete. No source files or models changed.")


In [ ]:

from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount("/content/drive", force_remount=True)

print("\nGoogle Drive mounted:", os.path.exists("/content/drive/MyDrive"))

# Locate project folder
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

print("Project exists:", PROJECT_ROOT.exists())

if PROJECT_ROOT.exists():
    print("\nProject folders:")
    for item in sorted(PROJECT_ROOT.iterdir()):
        if item.is_dir():
            print("📁", item.name)
else:
    print("\nProject folder not found at the expected location.")
    print("Check the exact folder name and Drive location.")


In [ ]:

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"

required_files = [
    "ICMB_130_BERTOPIC_EMBEDDING_INPUT.csv",
    "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv",
    "BERTOPIC_TOPIC_INFO.csv",
    "BERTOPIC_TOPIC_TERMS.csv",
    "BERTOPIC_V2_CHUNK_TOPIC_ASSIGNMENTS.csv",
    "BERTOPIC_V2_TOPIC_INFO.csv",
    "BERTOPIC_V2_TOPIC_TERMS.csv",
    "BERTOPIC_EXPERIMENT_COMPARISON.csv"
]

print("=" * 65)
print("BERTOPIC PROJECT FILE VERIFICATION")
print("=" * 65)

print("Output directory exists:", OUTPUT_DIR.exists())

for filename in required_files:
    path = OUTPUT_DIR / filename
    if path.exists():
        print(f"FOUND  | {filename} | {path.stat().st_size:,} bytes")
    else:
        print(f"MISSING | {filename}")


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)
OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"

# Load both completed experiments
baseline = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

experiment2 = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_V2_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

# Load topic information
baseline_info = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_TOPIC_INFO.csv"
)

experiment2_info = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_V2_TOPIC_INFO.csv"
)

# Load embedding input only if needed for future analysis
embedding_input = pd.read_csv(
    OUTPUT_DIR / "ICMB_130_BERTOPIC_EMBEDDING_INPUT.csv"
)

print("=" * 65)
print("BERTOPIC WORKSPACE RESTORED")
print("=" * 65)

print("\nBaseline assignments:", baseline.shape)
print("Experiment 2 assignments:", experiment2.shape)
print("Baseline topic information:", baseline_info.shape)
print("Experiment 2 topic information:", experiment2_info.shape)
print("Embedding input:", embedding_input.shape)

print("\nBaseline unique topics:", baseline["topic"].nunique())
print("Experiment 2 unique topics:", experiment2["topic"].nunique())

print("\nBaseline outliers:", (baseline["topic"] == -1).sum())
print("Experiment 2 outliers:", (experiment2["topic"] == -1).sum())

print("\nAll previous BERTopic results loaded successfully.")


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# RESTORE PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/IndianCinemaTrendAnalytics")
OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"

# ============================================================
# LOAD BASELINE AND EXPERIMENT 2 DATA
# ============================================================

baseline = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

experiment2 = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_V2_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

embedding_input = pd.read_csv(
    OUTPUT_DIR / "ICMB_130_BERTOPIC_EMBEDDING_INPUT.csv"
)

# ============================================================
# VERIFY IDENTIFIERS AND COLUMNS
# ============================================================

print("=" * 65)
print("BERTopic MOVIE-LEVEL ANALYSIS — DATA VALIDATION")
print("=" * 65)

print("\nBaseline columns:")
print(baseline.columns.tolist())

print("\nExperiment 2 columns:")
print(experiment2.columns.tolist())

print("\nEmbedding input columns:")
print(embedding_input.columns.tolist())

print("\nBaseline shape:", baseline.shape)
print("Experiment 2 shape:", experiment2.shape)
print("Embedding input shape:", embedding_input.shape)

print("\nBaseline sample:")
display(baseline.head(3))

print("\nEmbedding input sample:")
display(embedding_input.head(3))

In [ ]:
# ============================================================
# BERTopic — CORRECTED MOVIE-LEVEL TOPIC DISTRIBUTION
# Baseline vs Experiment 2
# ============================================================

def movie_topic_summary(df, model_name):

    df = df.copy()

    # Ensure topic is numeric
    df["topic"] = pd.to_numeric(df["topic"])

    # One row per movie
    movie_summary = (
        df.groupby(
            ["movie_record_id", "movie_title", "year"],
            as_index=False
        )
        .agg(total_chunks=("topic", "size"))
    )

    # Outlier counts per movie
    outliers = (
        df.assign(is_outlier=df["topic"].eq(-1))
        .groupby("movie_record_id")["is_outlier"]
        .sum()
        .rename("outlier_chunks")
    )

    # Assigned chunks and distinct topics
    assigned = df[df["topic"] != -1].copy()

    assigned_stats = (
        assigned.groupby("movie_record_id")
        .agg(
            assigned_chunks=("topic", "size"),
            distinct_topics=("topic", "nunique")
        )
    )

    # Dominant topic per movie
    dominant = (
        assigned.groupby(["movie_record_id", "topic"])
        .size()
        .reset_index(name="dominant_topic_chunks")
        .sort_values(
            ["movie_record_id", "dominant_topic_chunks", "topic"],
            ascending=[True, False, True]
        )
        .drop_duplicates("movie_record_id")
        .rename(columns={"topic": "dominant_topic"})
    )

    # Merge all statistics using movie_record_id
    movie_summary = movie_summary.merge(
        outliers.reset_index(),
        on="movie_record_id",
        how="left"
    )

    movie_summary = movie_summary.merge(
        assigned_stats.reset_index(),
        on="movie_record_id",
        how="left"
    )

    movie_summary = movie_summary.merge(
        dominant,
        on="movie_record_id",
        how="left"
    )

    # Fill movies without assigned topics
    count_columns = [
        "outlier_chunks",
        "assigned_chunks",
        "distinct_topics"
    ]

    movie_summary[count_columns] = (
        movie_summary[count_columns].fillna(0).astype(int)
    )

    # Percentages
    movie_summary["outlier_percentage"] = (
        movie_summary["outlier_chunks"] /
        movie_summary["total_chunks"] * 100
    ).round(2)

    movie_summary["assigned_percentage"] = (
        movie_summary["assigned_chunks"] /
        movie_summary["total_chunks"] * 100
    ).round(2)

    movie_summary["model"] = model_name

    return movie_summary


# Generate corrected summaries
baseline_movie_summary = movie_topic_summary(
    baseline, "Baseline"
)

experiment2_movie_summary = movie_topic_summary(
    experiment2, "Experiment 2"
)

# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)
print("CORRECTED MOVIE-LEVEL SUMMARY VALIDATION")
print("=" * 70)

print("\nBaseline shape:", baseline_movie_summary.shape)
print("Experiment 2 shape:", experiment2_movie_summary.shape)

print("\nBaseline unique movies:",
      baseline_movie_summary["movie_record_id"].nunique())

print("Experiment 2 unique movies:",
      experiment2_movie_summary["movie_record_id"].nunique())

print("\nBaseline total chunks:",
      baseline_movie_summary["total_chunks"].sum())

print("Experiment 2 total chunks:",
      experiment2_movie_summary["total_chunks"].sum())

print("\nBaseline outlier chunks:",
      baseline_movie_summary["outlier_chunks"].sum())

print("Experiment 2 outlier chunks:",
      experiment2_movie_summary["outlier_chunks"].sum())

print("\nBaseline preview:")
display(baseline_movie_summary.head(10))

print("\nExperiment 2 preview:")
display(experiment2_movie_summary.head(10))

# ============================================================
# SAVE CORRECTED FILES
# ============================================================

baseline_movie_summary.to_csv(
    OUTPUT_DIR / "BERTOPIC_BASELINE_MOVIE_LEVEL_SUMMARY.csv",
    index=False
)

experiment2_movie_summary.to_csv(
    OUTPUT_DIR / "BERTOPIC_V2_MOVIE_LEVEL_SUMMARY.csv",
    index=False
)

print("\nCorrected summaries saved successfully.")

In [ ]:
# ============================================================
# BERTopic — MOVIE-LEVEL COVERAGE COMPARISON
# Baseline vs Experiment 2
# ============================================================

# Merge the two movie-level summaries
movie_comparison = baseline_movie_summary.merge(
    experiment2_movie_summary[
        [
            "movie_record_id",
            "outlier_chunks",
            "assigned_chunks",
            "distinct_topics",
            "outlier_percentage",
            "assigned_percentage"
        ]
    ],
    on="movie_record_id",
    suffixes=("_baseline", "_experiment2"),
    how="inner"
)

# Calculate differences
movie_comparison["outlier_change_pp"] = (
    movie_comparison["outlier_percentage_experiment2"] -
    movie_comparison["outlier_percentage_baseline"]
).round(2)

movie_comparison["assigned_change_pp"] = (
    movie_comparison["assigned_percentage_experiment2"] -
    movie_comparison["assigned_percentage_baseline"]
).round(2)

movie_comparison["topic_count_change"] = (
    movie_comparison["distinct_topics_experiment2"] -
    movie_comparison["distinct_topics_baseline"]
)

print("=" * 75)
print("MOVIE-LEVEL COVERAGE COMPARISON")
print("=" * 75)

print("\nMovies compared:", len(movie_comparison))

print("\nAverage outlier percentage:")
print("Baseline:",
      round(movie_comparison["outlier_percentage_baseline"].mean(), 2), "%")
print("Experiment 2:",
      round(movie_comparison["outlier_percentage_experiment2"].mean(), 2), "%")

print("\nMovies with reduced outlier percentage:",
      (movie_comparison["outlier_change_pp"] < 0).sum())

print("Movies with increased outlier percentage:",
      (movie_comparison["outlier_change_pp"] > 0).sum())

print("Movies with unchanged outlier percentage:",
      (movie_comparison["outlier_change_pp"] == 0).sum())

# Highest outlier percentage — baseline
print("\n" + "=" * 75)
print("TOP 10 MOVIES — HIGHEST BASELINE OUTLIER PERCENTAGE")
print("=" * 75)

display(
    movie_comparison.sort_values(
        "outlier_percentage_baseline",
        ascending=False
    )[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "outlier_percentage_baseline",
            "outlier_percentage_experiment2",
            "outlier_change_pp"
        ]
    ].head(10)
)

# Lowest outlier percentage — baseline
print("\n" + "=" * 75)
print("TOP 10 MOVIES — LOWEST BASELINE OUTLIER PERCENTAGE")
print("=" * 75)

display(
    movie_comparison.sort_values(
        "outlier_percentage_baseline",
        ascending=True
    )[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "outlier_percentage_baseline",
            "outlier_percentage_experiment2",
            "outlier_change_pp"
        ]
    ].head(10)
)

# Save comparison
movie_comparison.to_csv(
    OUTPUT_DIR / "BERTOPIC_MOVIE_LEVEL_COVERAGE_COMPARISON.csv",
    index=False
)

print("\nMovie-level coverage comparison saved successfully.")

In [ ]:
# ============================================================
# BERTopic — YEAR-WISE COVERAGE COMPARISON
# Baseline vs Experiment 2
# ============================================================

year_comparison = (
    movie_comparison
    .groupby("year", as_index=False)
    .agg(
        total_movies=("movie_record_id", "nunique"),

        baseline_outlier_percentage=(
            "outlier_percentage_baseline", "mean"
        ),

        experiment2_outlier_percentage=(
            "outlier_percentage_experiment2", "mean"
        ),

        baseline_assigned_percentage=(
            "assigned_percentage_baseline", "mean"
        ),

        experiment2_assigned_percentage=(
            "assigned_percentage_experiment2", "mean"
        )
    )
)

year_comparison["outlier_change_pp"] = (
    year_comparison["experiment2_outlier_percentage"] -
    year_comparison["baseline_outlier_percentage"]
).round(2)

year_comparison["assigned_change_pp"] = (
    year_comparison["experiment2_assigned_percentage"] -
    year_comparison["baseline_assigned_percentage"]
).round(2)

percentage_columns = [
    "baseline_outlier_percentage",
    "experiment2_outlier_percentage",
    "baseline_assigned_percentage",
    "experiment2_assigned_percentage"
]

year_comparison[percentage_columns] = (
    year_comparison[percentage_columns].round(2)
)

print("=" * 80)
print("YEAR-WISE BERTopic COVERAGE COMPARISON")
print("=" * 80)

display(year_comparison)

print("\nAverage outlier percentage across years:")
print(
    "Baseline:",
    round(year_comparison["baseline_outlier_percentage"].mean(), 2)
)

print(
    "Experiment 2:",
    round(year_comparison["experiment2_outlier_percentage"].mean(), 2)
)

# Save
year_comparison.to_csv(
    OUTPUT_DIR / "BERTOPIC_YEAR_LEVEL_COVERAGE_COMPARISON.csv",
    index=False
)

print("\nYear-wise comparison saved successfully.")

In [ ]:
# ============================================================
# BERTopic — YEAR-WISE OUTLIER TREND
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(15, 6))

plt.plot(
    year_comparison["year"],
    year_comparison["baseline_outlier_percentage"],
    marker="o",
    linewidth=2,
    label="Baseline"
)

plt.plot(
    year_comparison["year"],
    year_comparison["experiment2_outlier_percentage"],
    marker="s",
    linewidth=2,
    label="Experiment 2"
)

plt.title(
    "Year-wise BERTopic Outlier Percentage (2000–2025)",
    fontsize=14
)

plt.xlabel("Year")
plt.ylabel("Average Outlier Percentage (%)")

plt.xticks(
    year_comparison["year"],
    rotation=45
)

plt.ylim(0, 100)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()

# Save chart
plt.savefig(
    OUTPUT_DIR / "BERTOPIC_YEARWISE_OUTLIER_TREND.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# BERTopic — TOPIC INFORMATION INSPECTION
# Baseline vs Experiment 2
# ============================================================

baseline_topic_info = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_TOPIC_INFO.csv"
)

experiment2_topic_info = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_V2_TOPIC_INFO.csv"
)

baseline_topic_terms = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_TOPIC_TERMS.csv"
)

experiment2_topic_terms = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_V2_TOPIC_TERMS.csv"
)

print("=" * 75)
print("BASELINE TOPIC INFORMATION")
print("=" * 75)

print("Shape:", baseline_topic_info.shape)
print("Columns:", baseline_topic_info.columns.tolist())

display(baseline_topic_info.head(15))

print("\n" + "=" * 75)
print("EXPERIMENT 2 TOPIC INFORMATION")
print("=" * 75)

print("Shape:", experiment2_topic_info.shape)
print("Columns:", experiment2_topic_info.columns.tolist())

display(experiment2_topic_info.head(15))

print("\n" + "=" * 75)
print("BASELINE TOPIC TERMS")
print("=" * 75)

print("Shape:", baseline_topic_terms.shape)
print("Columns:", baseline_topic_terms.columns.tolist())

display(baseline_topic_terms.head(20))

print("\n" + "=" * 75)
print("EXPERIMENT 2 TOPIC TERMS")
print("=" * 75)

print("Shape:", experiment2_topic_terms.shape)
print("Columns:", experiment2_topic_terms.columns.tolist())

display(experiment2_topic_terms.head(20))

In [ ]:
# ============================================================
# BERTopic — REPRESENTATIVE TOPIC REVIEW
# Top 20 topics from each configuration
# ============================================================

import ast

def prepare_topic_review(topic_info, topic_terms, model_name):

    info = topic_info.copy()
    terms = topic_terms.copy()

    # Standardize topic ID columns
    info["Topic"] = pd.to_numeric(info["Topic"]).astype(int)

    topic_col = "topic" if "topic" in terms.columns else "topic_id"
    terms[topic_col] = pd.to_numeric(terms[topic_col]).astype(int)

    # Exclude outlier topic (-1)
    info = info[info["Topic"] != -1].copy()

    # Sort by topic size
    info = info.sort_values("Count", ascending=False).head(20)

    # Prepare keyword lists
    keyword_col = "term"

    term_lists = (
        terms[terms[topic_col] != -1]
        .sort_values("term_rank")
        .groupby(topic_col)[keyword_col]
        .apply(lambda x: ", ".join(x.head(10).astype(str)))
        .rename("top_keywords")
    )

    info = info.merge(
        term_lists,
        left_on="Topic",
        right_index=True,
        how="left"
    )

    # Extract representative dialogue safely
    def extract_representative_docs(value):

        try:
            docs = ast.literal_eval(value) if isinstance(value, str) else value

            if isinstance(docs, list):
                return " || ".join(
                    str(doc).replace("\n", " ")[:350]
                    for doc in docs[:2]
                )

        except (ValueError, SyntaxError, TypeError):
            pass

        return str(value)[:500]

    info["representative_dialogue"] = (
        info["Representative_Docs"]
        .apply(extract_representative_docs)
    )

    info["model"] = model_name

    return info[
        [
            "model",
            "Topic",
            "Count",
            "Name",
            "top_keywords",
            "representative_dialogue"
        ]
    ]


# Generate topic review tables
baseline_topic_review = prepare_topic_review(
    baseline_topic_info,
    baseline_topic_terms,
    "Baseline"
)

experiment2_topic_review = prepare_topic_review(
    experiment2_topic_info,
    experiment2_topic_terms,
    "Experiment 2"
)

# Display baseline topics
print("=" * 90)
print("BASELINE — TOP 20 REPRESENTATIVE TOPICS")
print("=" * 90)

display(baseline_topic_review)

# Display Experiment 2 topics
print("\n" + "=" * 90)
print("EXPERIMENT 2 — TOP 20 REPRESENTATIVE TOPICS")
print("=" * 90)

display(experiment2_topic_review)

# Save review files
baseline_topic_review.to_csv(
    OUTPUT_DIR / "BERTOPIC_BASELINE_TOP20_TOPIC_REVIEW.csv",
    index=False
)

experiment2_topic_review.to_csv(
    OUTPUT_DIR / "BERTOPIC_V2_TOP20_TOPIC_REVIEW.csv",
    index=False
)

print("\nBoth topic review files saved successfully.")

In [ ]:
# ============================================================
# BERTopic — TOPIC DISTRIBUTION ACROSS MOVIES AND YEARS
# Baseline vs Experiment 2
# ============================================================

def topic_distribution_analysis(df, model_name):

    # Exclude unassigned chunks
    data = df[df["topic"] != -1].copy()

    # Topic-level totals
    topic_totals = (
        data.groupby("topic")
        .agg(
            total_chunks=("source_chunk_id", "count"),
            unique_movies=("movie_record_id", "nunique"),
            unique_years=("year", "nunique")
        )
        .reset_index()
    )

    # Movie contribution to each topic
    movie_contributions = (
        data.groupby(
            ["topic", "movie_record_id", "movie_title", "year"]
        )
        .size()
        .reset_index(name="movie_topic_chunks")
    )

    # Largest movie contribution
    largest_movie = (
        movie_contributions
        .sort_values(
            ["topic", "movie_topic_chunks"],
            ascending=[True, False]
        )
        .drop_duplicates("topic")
        .copy()
    )

    largest_movie = largest_movie.rename(
        columns={
            "movie_title": "largest_contributing_movie",
            "movie_topic_chunks": "largest_movie_chunks"
        }
    )

    largest_movie["largest_movie_share_pct"] = (
        largest_movie["largest_movie_chunks"] /
        largest_movie["topic"].map(
            topic_totals.set_index("topic")["total_chunks"]
        ) * 100
    ).round(2)

    # Five largest contributing movies
    top_movies = (
        movie_contributions
        .sort_values(
            ["topic", "movie_topic_chunks"],
            ascending=[True, False]
        )
        .groupby("topic")
        .head(5)
        .copy()
    )

    top_movies["movie_contribution"] = (
        top_movies["movie_title"] +
        " (" +
        top_movies["year"].astype(str) +
        "): " +
        top_movies["movie_topic_chunks"].astype(str)
    )

    top_movie_lists = (
        top_movies.groupby("topic")["movie_contribution"]
        .apply(lambda x: " | ".join(x))
        .rename("top_5_movies")
    )

    # Combine statistics
    result = topic_totals.merge(
        largest_movie[
            [
                "topic",
                "largest_contributing_movie",
                "largest_movie_chunks",
                "largest_movie_share_pct"
            ]
        ],
        on="topic",
        how="left"
    )

    result = result.join(
        top_movie_lists,
        on="topic"
    )

    result["model"] = model_name

    return result.sort_values(
        "total_chunks",
        ascending=False
    )


# Run both analyses
baseline_topic_distribution = topic_distribution_analysis(
    baseline, "Baseline"
)

experiment2_topic_distribution = topic_distribution_analysis(
    experiment2, "Experiment 2"
)

# Display baseline
print("=" * 90)
print("BASELINE — TOPIC DISTRIBUTION ACROSS MOVIES")
print("=" * 90)

display(baseline_topic_distribution.head(20))

# Display Experiment 2
print("\n" + "=" * 90)
print("EXPERIMENT 2 — TOPIC DISTRIBUTION ACROSS MOVIES")
print("=" * 90)

display(experiment2_topic_distribution.head(20))

# Save results
baseline_topic_distribution.to_csv(
    OUTPUT_DIR / "BERTOPIC_BASELINE_TOPIC_MOVIE_DISTRIBUTION.csv",
    index=False
)

experiment2_topic_distribution.to_csv(
    OUTPUT_DIR / "BERTOPIC_V2_TOPIC_MOVIE_DISTRIBUTION.csv",
    index=False
)

print("\nBoth topic distribution files saved successfully.")

In [ ]:
# ============================================================
# BERTopic — TOPIC DISTRIBUTION RELIABILITY DIAGNOSTIC
# ============================================================

def classify_topic_distribution(df):

    df = df.copy()

    def classify(row):

        if (
            row["unique_movies"] <= 5
            or row["largest_movie_share_pct"] >= 50
        ):
            return "Film-concentrated"

        elif (
            row["unique_movies"] >= 20
            and row["largest_movie_share_pct"] <= 20
        ):
            return "Broadly distributed"

        else:
            return "Intermediate"

    df["distribution_category"] = df.apply(
        classify,
        axis=1
    )

    return df


# Apply to both models
baseline_topic_diagnostic = classify_topic_distribution(
    baseline_topic_distribution
)

experiment2_topic_diagnostic = classify_topic_distribution(
    experiment2_topic_distribution
)


# ============================================================
# CATEGORY COUNTS
# ============================================================

print("=" * 80)
print("BASELINE — TOPIC DISTRIBUTION CATEGORIES")
print("=" * 80)

display(
    baseline_topic_diagnostic[
        "distribution_category"
    ].value_counts().rename_axis("category").reset_index(
        name="topic_count"
    )
)

print("\n" + "=" * 80)
print("EXPERIMENT 2 — TOPIC DISTRIBUTION CATEGORIES")
print("=" * 80)

display(
    experiment2_topic_diagnostic[
        "distribution_category"
    ].value_counts().rename_axis("category").reset_index(
        name="topic_count"
    )
)


# ============================================================
# FILM-CONCENTRATED TOPICS
# ============================================================

print("\nBASELINE — FILM-CONCENTRATED TOPICS")

display(
    baseline_topic_diagnostic[
        baseline_topic_diagnostic["distribution_category"]
        == "Film-concentrated"
    ].sort_values(
        "total_chunks",
        ascending=False
    )
)

print("\nEXPERIMENT 2 — FILM-CONCENTRATED TOPICS")

display(
    experiment2_topic_diagnostic[
        experiment2_topic_diagnostic["distribution_category"]
        == "Film-concentrated"
    ].sort_values(
        "total_chunks",
        ascending=False
    )
)


# ============================================================
# SAVE
# ============================================================

baseline_topic_diagnostic.to_csv(
    OUTPUT_DIR / "BERTOPIC_BASELINE_TOPIC_RELIABILITY_DIAGNOSTIC.csv",
    index=False
)

experiment2_topic_diagnostic.to_csv(
    OUTPUT_DIR / "BERTOPIC_V2_TOPIC_RELIABILITY_DIAGNOSTIC.csv",
    index=False
)

print("\nBoth topic reliability diagnostic files saved.")

In [ ]:
# ============================================================
# RESTORE BERTopic TOPIC INVENTORY INPUTS
# ============================================================

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

OUTPUT_DIR = PROJECT_ROOT / "06_outputs" / "topic_modelling"

# Topic information
baseline_topic_info = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_TOPIC_INFO.csv"
)

experiment2_topic_info = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_V2_TOPIC_INFO.csv"
)

# Topic terms
baseline_topic_terms = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_TOPIC_TERMS.csv"
)

experiment2_topic_terms = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_V2_TOPIC_TERMS.csv"
)

# Topic distribution diagnostics
baseline_topic_diagnostic = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_BASELINE_TOPIC_RELIABILITY_DIAGNOSTIC.csv"
)

experiment2_topic_diagnostic = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_V2_TOPIC_RELIABILITY_DIAGNOSTIC.csv"
)

print("=" * 70)
print("BERTopic INVENTORY INPUTS RESTORED")
print("=" * 70)

for name, df in [
    ("Baseline topic info", baseline_topic_info),
    ("Experiment 2 topic info", experiment2_topic_info),
    ("Baseline topic terms", baseline_topic_terms),
    ("Experiment 2 topic terms", experiment2_topic_terms),
    ("Baseline diagnostic", baseline_topic_diagnostic),
    ("Experiment 2 diagnostic", experiment2_topic_diagnostic)
]:
    print(f"{name}: {df.shape}")

print("\nAll six input files loaded successfully.")

In [ ]:
print("=" * 70)
print("VERIFYING TOPIC INVENTORY SCHEMAS")
print("=" * 70)

datasets = {
    "Baseline Topic Info": baseline_topic_info,
    "Experiment 2 Topic Info": experiment2_topic_info,
    "Baseline Topic Terms": baseline_topic_terms,
    "Experiment 2 Topic Terms": experiment2_topic_terms,
    "Baseline Diagnostic": baseline_topic_diagnostic,
    "Experiment 2 Diagnostic": experiment2_topic_diagnostic
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 60)
    print("Columns:", df.columns.tolist())
    print(df.head(2).to_string(index=False))

In [ ]:
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 1. OUTPUT DIRECTORY
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"
)

# ------------------------------------------------------------
# 2. FUNCTION TO BUILD COMPLETE TOPIC INVENTORY
# ------------------------------------------------------------

def build_complete_topic_inventory(
    topic_info,
    topic_terms,
    topic_diagnostic,
    model_name
):

    # Exclude outlier topic (-1)
    info = topic_info[
        topic_info["Topic"] != -1
    ].copy()

    # Standardize topic ID
    info["Topic"] = info["Topic"].astype(int)

    # Prepare top 10 keywords for each topic
    terms = topic_terms.copy()

    terms["topic"] = (
        terms["topic"].astype(int)
        if "topic" in terms.columns
        else terms["topic_id"].astype(int)
    )

    terms = terms.sort_values(
        ["topic", "term_rank"]
    )

    top_keywords = (
        terms.groupby("topic")["term"]
        .apply(lambda x: ", ".join(x.head(10).astype(str)))
        .reset_index(name="Top_10_Keywords")
    )

    # Prepare diagnostic information
    diagnostic = topic_diagnostic.copy()
    diagnostic["topic"] = diagnostic["topic"].astype(int)

    diagnostic = diagnostic.drop(
        columns=["model"],
        errors="ignore"
    )

    # Merge topic information and keywords
    inventory = info.merge(
        top_keywords,
        left_on="Topic",
        right_on="topic",
        how="left"
    )

    inventory = inventory.drop(
        columns=["topic"],
        errors="ignore"
    )

    # Merge distribution diagnostics
    inventory = inventory.merge(
        diagnostic,
        left_on="Topic",
        right_on="topic",
        how="left",
        validate="one_to_one"
    )

    inventory = inventory.drop(
        columns=["topic"],
        errors="ignore"
    )

    # Add model identifier
    inventory.insert(0, "Model", model_name)

    # Organize columns
    column_order = [
        "Model",
        "Topic",
        "Name",
        "Count",
        "Representation",
        "Top_10_Keywords",
        "Representative_Docs",
        "total_chunks",
        "unique_movies",
        "unique_years",
        "largest_contributing_movie",
        "largest_movie_chunks",
        "largest_movie_share_pct",
        "top_5_movies",
        "distribution_category"
    ]

    inventory = inventory[
        [col for col in column_order if col in inventory.columns]
    ]

    return inventory


# ------------------------------------------------------------
# 3. BUILD BOTH INVENTORIES
# ------------------------------------------------------------

baseline_inventory = build_complete_topic_inventory(
    baseline_topic_info,
    baseline_topic_terms,
    baseline_topic_diagnostic,
    "Baseline"
)

experiment2_inventory = build_complete_topic_inventory(
    experiment2_topic_info,
    experiment2_topic_terms,
    experiment2_topic_diagnostic,
    "Experiment 2"
)

# ------------------------------------------------------------
# 4. CREATE COMBINED INVENTORY
# ------------------------------------------------------------

combined_inventory = pd.concat(
    [baseline_inventory, experiment2_inventory],
    ignore_index=True
)

# ------------------------------------------------------------
# 5. SAVE CSV FILES
# ------------------------------------------------------------

baseline_inventory.to_csv(
    OUTPUT_DIR / "BERTOPIC_BASELINE_COMPLETE_TOPIC_INVENTORY.csv",
    index=False
)

experiment2_inventory.to_csv(
    OUTPUT_DIR / "BERTOPIC_V2_COMPLETE_TOPIC_INVENTORY.csv",
    index=False
)

combined_inventory.to_csv(
    OUTPUT_DIR / "BERTOPIC_COMBINED_COMPLETE_TOPIC_INVENTORY.csv",
    index=False
)

# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

print("=" * 75)
print("COMPLETE BERTopic INVENTORY — VALIDATION")
print("=" * 75)

print("\nBaseline inventory:", baseline_inventory.shape)
print("Experiment 2 inventory:", experiment2_inventory.shape)
print("Combined inventory:", combined_inventory.shape)

print("\nExpected topic counts:")
print("Baseline: 62")
print("Experiment 2: 109")
print("Combined: 171")

print("\nActual topic counts:")
print("Baseline:", baseline_inventory["Topic"].nunique())
print("Experiment 2:", experiment2_inventory["Topic"].nunique())

print("\nMissing values:")
print(
    combined_inventory.isnull().sum()[
        combined_inventory.isnull().sum() > 0
    ]
)

print("\nSaved files:")
for filename in [
    "BERTOPIC_BASELINE_COMPLETE_TOPIC_INVENTORY.csv",
    "BERTOPIC_V2_COMPLETE_TOPIC_INVENTORY.csv",
    "BERTOPIC_COMBINED_COMPLETE_TOPIC_INVENTORY.csv"
]:
    path = OUTPUT_DIR / filename
    print(f"{filename}: {path.exists()}")

print("\n" + "=" * 75)
print("BASELINE INVENTORY — SAMPLE")
print("=" * 75)

display(
    baseline_inventory[
        ["Topic", "Name", "Count", "Top_10_Keywords",
         "unique_movies", "distribution_category"]
    ].head(10)
)

print("\n" + "=" * 75)
print("EXPERIMENT 2 INVENTORY — SAMPLE")
print("=" * 75)

display(
    experiment2_inventory[
        ["Topic", "Name", "Count", "Top_10_Keywords",
         "unique_movies", "distribution_category"]
    ].head(10)
)

In [ ]:
print("=" * 75)
print("TOPIC INVENTORY READINESS CHECK")
print("=" * 75)

print("\nBaseline topic inventory:")
print(baseline_inventory[
    ["Topic", "Name", "Top_10_Keywords", "Count"]
].head(15).to_string(index=False))

print("\nExperiment 2 topic inventory:")
print(experiment2_inventory[
    ["Topic", "Name", "Top_10_Keywords", "Count"]
].head(15).to_string(index=False))

print("\nTopic count comparison:")
print("Baseline:", len(baseline_inventory))
print("Experiment 2:", len(experiment2_inventory))

print("\nInventory files are ready for semantic comparison.")

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ------------------------------------------------------------
# 1. REBUILD MATCHING INPUTS
# ------------------------------------------------------------

baseline_match = baseline_inventory[
    ["Topic", "Name", "Top_10_Keywords", "Count"]
].copy()

experiment2_match = experiment2_inventory[
    ["Topic", "Name", "Top_10_Keywords", "Count"]
].copy()

baseline_match["keyword_text"] = (
    baseline_match["Top_10_Keywords"].fillna("")
)

experiment2_match["keyword_text"] = (
    experiment2_match["Top_10_Keywords"].fillna("")
)

# ------------------------------------------------------------
# 2. TF-IDF AND COSINE SIMILARITY
# ------------------------------------------------------------

all_keyword_text = (
    baseline_match["keyword_text"].tolist()
    + experiment2_match["keyword_text"].tolist()
)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    token_pattern=r"(?u)\b\w+\b",
    lowercase=True
)

tfidf_matrix = vectorizer.fit_transform(all_keyword_text)

baseline_vectors = tfidf_matrix[:len(baseline_match)]
experiment2_vectors = tfidf_matrix[len(baseline_match):]

similarity_matrix = cosine_similarity(
    baseline_vectors,
    experiment2_vectors
)

# ------------------------------------------------------------
# 3. TOP 3 MATCHES
# ------------------------------------------------------------

match_rows = []

for i in range(len(baseline_match)):
    top_indices = similarity_matrix[i].argsort()[::-1][:3]

    for rank, j in enumerate(top_indices, start=1):
        base_row = baseline_match.iloc[i]
        exp_row = experiment2_match.iloc[j]

        match_rows.append({
            "Baseline_Topic": int(base_row["Topic"]),
            "Baseline_Name": base_row["Name"],
            "Baseline_Keywords": base_row["Top_10_Keywords"],
            "Baseline_Count": int(base_row["Count"]),
            "Match_Rank": rank,
            "Experiment2_Topic": int(exp_row["Topic"]),
            "Experiment2_Name": exp_row["Name"],
            "Experiment2_Keywords": exp_row["Top_10_Keywords"],
            "Experiment2_Count": int(exp_row["Count"]),
            "Keyword_Cosine_Similarity": round(
                float(similarity_matrix[i, j]), 4
            )
        })

topic_matches = pd.DataFrame(match_rows)

# ------------------------------------------------------------
# 4. RECIPROCAL STRONGEST MATCHES
# ------------------------------------------------------------

baseline_best = similarity_matrix.argmax(axis=1)
experiment2_best = similarity_matrix.argmax(axis=0)

reciprocal_pairs = []

for i, j in enumerate(baseline_best):
    if experiment2_best[j] == i:
        reciprocal_pairs.append({
            "Baseline_Topic": int(baseline_match.iloc[i]["Topic"]),
            "Baseline_Name": baseline_match.iloc[i]["Name"],
            "Experiment2_Topic": int(experiment2_match.iloc[j]["Topic"]),
            "Experiment2_Name": experiment2_match.iloc[j]["Name"],
            "Keyword_Cosine_Similarity": round(
                float(similarity_matrix[i, j]), 4
            )
        })

reciprocal_matches = pd.DataFrame(reciprocal_pairs)

# ------------------------------------------------------------
# 5. SAVE RESULTS
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"
)

topic_matches.to_csv(
    OUTPUT_DIR / "BERTOPIC_TOPIC_SIMILARITY_TOP3_MATCHES.csv",
    index=False
)

reciprocal_matches.to_csv(
    OUTPUT_DIR / "BERTOPIC_RECIPROCAL_TOPIC_MATCHES.csv",
    index=False
)

# ------------------------------------------------------------
# 6. VALIDATION
# ------------------------------------------------------------

print("=" * 70)
print("BERTopic TOPIC SIMILARITY ANALYSIS")
print("=" * 70)

print("Baseline topics:", len(baseline_match))
print("Experiment 2 topics:", len(experiment2_match))
print("Pairwise comparisons:", similarity_matrix.size)
print("Top 3 match records:", len(topic_matches))
print("Reciprocal matches:", len(reciprocal_matches))

print("\nTOP 3 MATCHES — FIRST 15 BASELINE TOPICS")

display(
    topic_matches[
        topic_matches["Baseline_Topic"].isin(
            baseline_match["Topic"].head(15)
        )
    ][[
        "Baseline_Topic",
        "Baseline_Name",
        "Match_Rank",
        "Experiment2_Topic",
        "Experiment2_Name",
        "Keyword_Cosine_Similarity"
    ]]
)

print("\nRECIPROCAL STRONGEST MATCHES")

display(
    reciprocal_matches.sort_values(
        "Keyword_Cosine_Similarity",
        ascending=False
    )
)

print("\nBoth output files saved successfully.")

In [ ]:
# ------------------------------------------------------------
# 1. CLASSIFY RECIPROCAL MATCH STRENGTH
# ------------------------------------------------------------

reciprocal_review = reciprocal_matches.copy()

def classify_similarity(score):
    if score >= 0.50:
        return "Strong"
    elif score >= 0.30:
        return "Moderate"
    elif score >= 0.15:
        return "Weak"
    else:
        return "Very weak"

reciprocal_review["Similarity_Category"] = (
    reciprocal_review["Keyword_Cosine_Similarity"]
    .apply(classify_similarity)
)

# ------------------------------------------------------------
# 2. SUMMARY
# ------------------------------------------------------------

print("=" * 75)
print("RECIPROCAL TOPIC MATCH STRENGTH")
print("=" * 75)

category_summary = (
    reciprocal_review["Similarity_Category"]
    .value_counts()
    .reindex(
        ["Strong", "Moderate", "Weak", "Very weak"],
        fill_value=0
    )
    .rename_axis("Similarity_Category")
    .reset_index(name="Number_of_Matches")
)

category_summary["Percentage"] = (
    category_summary["Number_of_Matches"]
    / len(reciprocal_review) * 100
).round(2)

display(category_summary)

# ------------------------------------------------------------
# 3. LOWEST-SIMILARITY RECIPROCAL MATCHES
# ------------------------------------------------------------

print("\n10 LOWEST-SIMILARITY RECIPROCAL MATCHES")

display(
    reciprocal_review.sort_values(
        "Keyword_Cosine_Similarity",
        ascending=True
    ).head(10)
)

# ------------------------------------------------------------
# 4. HIGHEST-SIMILARITY RECIPROCAL MATCHES
# ------------------------------------------------------------

print("\n10 HIGHEST-SIMILARITY RECIPROCAL MATCHES")

display(
    reciprocal_review.sort_values(
        "Keyword_Cosine_Similarity",
        ascending=False
    ).head(10)
)

# ------------------------------------------------------------
# 5. SAVE REVIEW FILE
# ------------------------------------------------------------

reciprocal_review.to_csv(
    OUTPUT_DIR / "BERTOPIC_RECIPROCAL_MATCH_STRENGTH_REVIEW.csv",
    index=False
)

print("\nSaved: BERTOPIC_RECIPROCAL_MATCH_STRENGTH_REVIEW.csv")

In [ ]:
# ------------------------------------------------------------
# 1. IDENTIFY MULTIPLE RELATED EXPERIMENT 2 TOPICS
# ------------------------------------------------------------

fragmentation_candidates = topic_matches[
    topic_matches["Keyword_Cosine_Similarity"] >= 0.30
].copy()

# Count related Experiment 2 topics for each baseline topic
fragmentation_summary = (
    fragmentation_candidates
    .groupby(
        ["Baseline_Topic", "Baseline_Name"]
    )
    .agg(
        Related_Experiment2_Topics=(
            "Experiment2_Topic", "nunique"
        ),
        Related_Experiment2_IDs=(
            "Experiment2_Topic",
            lambda x: ", ".join(map(str, x))
        ),
        Related_Experiment2_Names=(
            "Experiment2_Name",
            lambda x: " | ".join(x)
        ),
        Similarity_Scores=(
            "Keyword_Cosine_Similarity",
            lambda x: ", ".join(f"{v:.4f}" for v in x)
        )
    )
    .reset_index()
)

# Candidate fragmentation: at least two related clusters
fragmentation_summary["Potential_Fragmentation"] = (
    fragmentation_summary["Related_Experiment2_Topics"] >= 2
)

fragmentation_summary = fragmentation_summary.sort_values(
    ["Related_Experiment2_Topics", "Baseline_Topic"],
    ascending=[False, True]
)

# ------------------------------------------------------------
# 2. DISPLAY RESULTS
# ------------------------------------------------------------

print("=" * 75)
print("BERTopic POTENTIAL TOPIC FRAGMENTATION ANALYSIS")
print("=" * 75)

print("\nBaseline topics with multiple related Experiment 2 topics:",
      fragmentation_summary["Potential_Fragmentation"].sum())

print("\nFRAGMENTATION CANDIDATES")

display(
    fragmentation_summary[
        fragmentation_summary["Potential_Fragmentation"]
    ]
)

# ------------------------------------------------------------
# 3. SAVE RESULTS
# ------------------------------------------------------------

fragmentation_summary.to_csv(
    OUTPUT_DIR / "BERTOPIC_POTENTIAL_TOPIC_FRAGMENTATION.csv",
    index=False
)

print("\nSaved: BERTOPIC_POTENTIAL_TOPIC_FRAGMENTATION.csv")

In [ ]:
# ------------------------------------------------------------
# CANDIDATE FRAGMENTATION — DETAILED EVIDENCE REVIEW
# ------------------------------------------------------------

candidate_pairs = [
    (0, [0, 1]),
    (1, [1, 0]),
    (3, [2, 48]),
    (22, [49, 57])
]

def inspect_topic(inventory, topic_id, model_name):

    row = inventory.loc[
        inventory["Topic"] == topic_id
    ]

    if row.empty:
        print(f"Topic {topic_id} not found in {model_name}")
        return

    row = row.iloc[0]

    print(f"\n{model_name} — Topic {topic_id}")
    print("-" * 70)

    print("Topic name:", row["Name"])
    print("Chunk count:", row["Count"])
    print("Top keywords:", row["Top_10_Keywords"])
    print("Unique movies:", row["unique_movies"])
    print("Unique years:", row["unique_years"])
    print("Largest contributing movie:",
          row["largest_contributing_movie"])
    print("Largest movie share (%):",
          row["largest_movie_share_pct"])
    print("Distribution category:",
          row["distribution_category"])

    print("\nRepresentative dialogue:")

    docs = row["Representative_Docs"]

    if isinstance(docs, str):
        try:
            import ast
            docs = ast.literal_eval(docs)
        except Exception:
            docs = [docs]

    if isinstance(docs, list):
        for i, doc in enumerate(docs[:2], start=1):
            print(f"\nExcerpt {i}:")
            print(str(doc)[:700])
    else:
        print("No representative dialogue available.")


# ------------------------------------------------------------
# REVIEW ALL CANDIDATE TOPICS
# ------------------------------------------------------------

for baseline_id, experiment2_ids in candidate_pairs:

    print("\n" + "=" * 85)
    print(f"BASELINE TOPIC {baseline_id} — FRAGMENTATION REVIEW")
    print("=" * 85)

    inspect_topic(
        baseline_inventory,
        baseline_id,
        "BASELINE"
    )

    for exp_id in experiment2_ids:
        inspect_topic(
            experiment2_inventory,
            exp_id,
            "EXPERIMENT 2"
        )

print("\n" + "=" * 85)
print("FRAGMENTATION EVIDENCE REVIEW COMPLETED")
print("=" * 85)

In [ ]:
# ============================================================
# BERTopic — TOPIC CONCENTRATION COMPARISON
# ============================================================

import pandas as pd
import numpy as np

def calculate_concentration_summary(inventory, model_name):

    df = inventory.copy()

    summary = {
        "Model": model_name,
        "Total_Topics": len(df),

        "Mean_Movie_Coverage": round(
            df["unique_movies"].mean(), 2
        ),

        "Median_Movie_Coverage": round(
            df["unique_movies"].median(), 2
        ),

        "Mean_Largest_Movie_Share_Pct": round(
            df["largest_movie_share_pct"].mean(), 2
        ),

        "Topics_Covering_1_to_5_Movies": int(
            (df["unique_movies"] <= 5).sum()
        ),

        "Topics_Covering_10_or_More_Movies": int(
            (df["unique_movies"] >= 10).sum()
        ),

        "Topics_Covering_20_or_More_Movies": int(
            (df["unique_movies"] >= 20).sum()
        ),

        "Topics_With_Dominant_Movie_Above_50_Pct": int(
            (df["largest_movie_share_pct"] >= 50).sum()
        ),

        "Broadly_Distributed_Topics": int(
            (df["distribution_category"] ==
             "Broadly distributed").sum()
        ),

        "Film_Concentrated_Topics": int(
            (df["distribution_category"] ==
             "Film-concentrated").sum()
        )
    }

    return summary


# ------------------------------------------------------------
# BUILD COMPARISON
# ------------------------------------------------------------

concentration_comparison = pd.DataFrame([

    calculate_concentration_summary(
        baseline_inventory,
        "Baseline"
    ),

    calculate_concentration_summary(
        experiment2_inventory,
        "Experiment 2"
    )

])

print("=" * 85)
print("BERTopic TOPIC CONCENTRATION COMPARISON")
print("=" * 85)

display(concentration_comparison)

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

concentration_comparison.to_csv(
    OUTPUT_DIR / "BERTOPIC_TOPIC_CONCENTRATION_COMPARISON.csv",
    index=False
)

print("\nSaved: BERTOPIC_TOPIC_CONCENTRATION_COMPARISON.csv")

In [ ]:
# ============================================================
# ICMB-130 — CONSOLIDATED BERTopic MODEL COMPARISON
# ============================================================

import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"
)

# ------------------------------------------------------------
# 1. LOAD PREVIOUSLY SAVED COMPARISON RESULTS
# ------------------------------------------------------------

coverage = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_MOVIE_LEVEL_COVERAGE_COMPARISON.csv"
)

year_coverage = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_YEAR_LEVEL_COVERAGE_COMPARISON.csv"
)

# ------------------------------------------------------------
# 2. CALCULATE COVERAGE METRICS
# ------------------------------------------------------------

coverage_columns = coverage.columns.tolist()
year_columns = year_coverage.columns.tolist()

print("Movie coverage columns:", coverage_columns)
print("Year coverage columns:", year_columns)

# ------------------------------------------------------------
# 3. SUMMARIZE TOPIC DISTRIBUTION
# ------------------------------------------------------------

def summarize_model(inventory, model_name):

    total = len(inventory)

    return {
        "Model": model_name,
        "Total_Topics": total,
        "Broad_Topic_Count": int(
            (inventory["distribution_category"] ==
             "Broadly distributed").sum()
        ),
        "Broad_Topic_Percentage": round(
            (inventory["distribution_category"] ==
             "Broadly distributed").mean() * 100, 2
        ),
        "Intermediate_Topic_Count": int(
            (inventory["distribution_category"] ==
             "Intermediate").sum()
        ),
        "Intermediate_Topic_Percentage": round(
            (inventory["distribution_category"] ==
             "Intermediate").mean() * 100, 2
        ),
        "Film_Concentrated_Count": int(
            (inventory["distribution_category"] ==
             "Film-concentrated").sum()
        ),
        "Film_Concentrated_Percentage": round(
            (inventory["distribution_category"] ==
             "Film-concentrated").mean() * 100, 2
        ),
        "Mean_Movie_Coverage": round(
            inventory["unique_movies"].mean(), 2
        ),
        "Mean_Dominant_Movie_Share": round(
            inventory["largest_movie_share_pct"].mean(), 2
        )
    }


model_comparison = pd.DataFrame([
    summarize_model(baseline_inventory, "Baseline"),
    summarize_model(experiment2_inventory, "Experiment 2")
])

# ------------------------------------------------------------
# 4. DISPLAY COMPARISON
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("ICMB-130 — CONSOLIDATED TOPIC DISTRIBUTION COMPARISON")
print("=" * 85)

display(model_comparison)

# ------------------------------------------------------------
# 5. SAVE
# ------------------------------------------------------------

model_comparison.to_csv(
    OUTPUT_DIR / "ICMB_130_BERTOPIC_CONSOLIDATED_MODEL_COMPARISON.csv",
    index=False
)

print("\nSaved:")
print("ICMB_130_BERTOPIC_CONSOLIDATED_MODEL_COMPARISON.csv")

In [ ]:
import os
import pandas as pd

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

files_to_check = [
    "BERTOPIC_BASELINE_TOPIC_REVIEW.csv",
    "BERTOPIC_V2_TOPIC_REVIEW.csv",
    "BERTOPIC_BASELINE_COMPLETE_TOPIC_INVENTORY.csv",
    "BERTOPIC_V2_COMPLETE_TOPIC_INVENTORY.csv",
    "BERTOPIC_TOPIC_TERMS.csv",
    "BERTOPIC_V2_TOPIC_TERMS.csv"
]

print("=" * 75)
print("ICMB-130 — TOPIC QUALITY EVALUATION: INPUT CHECK")
print("=" * 75)

for filename in files_to_check:
    path = os.path.join(OUTPUT_DIR, filename)
    print(f"{filename}: {'FOUND' if os.path.exists(path) else 'NOT FOUND'}")

In [ ]:
import os
import pandas as pd

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

# Load topic inventories
baseline_inventory = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "BERTOPIC_BASELINE_COMPLETE_TOPIC_INVENTORY.csv"
    )
)

experiment2_inventory = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "BERTOPIC_V2_COMPLETE_TOPIC_INVENTORY.csv"
    )
)

# Load topic terms
baseline_terms = pd.read_csv(
    os.path.join(OUTPUT_DIR, "BERTOPIC_TOPIC_TERMS.csv")
)

experiment2_terms = pd.read_csv(
    os.path.join(OUTPUT_DIR, "BERTOPIC_V2_TOPIC_TERMS.csv")
)

print("=" * 75)
print("ICMB-130 — TOPIC QUALITY EVALUATION")
print("=" * 75)

print("\nBASELINE INVENTORY")
print("Shape:", baseline_inventory.shape)
print("Columns:", baseline_inventory.columns.tolist())

print("\nEXPERIMENT 2 INVENTORY")
print("Shape:", experiment2_inventory.shape)
print("Columns:", experiment2_inventory.columns.tolist())

print("\nBASELINE TOPIC TERMS")
print("Shape:", baseline_terms.shape)
print("Columns:", baseline_terms.columns.tolist())

print("\nEXPERIMENT 2 TOPIC TERMS")
print("Shape:", experiment2_terms.shape)
print("Columns:", experiment2_terms.columns.tolist())

print("\nBASELINE TERMS — SAMPLE")
display(baseline_terms.head(10))

print("\nEXPERIMENT 2 TERMS — SAMPLE")
display(experiment2_terms.head(10))

In [ ]:
import os
import pandas as pd
import numpy as np

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

# Load original chunk-level text
chunk_path = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_BERTOPIC_EMBEDDING_INPUT.csv"
)

chunk_df = pd.read_csv(chunk_path)

print("=" * 75)
print("ICMB-130 — COHERENCE EVALUATION: CORPUS CHECK")
print("=" * 75)

print("Chunk dataset shape:", chunk_df.shape)
print("\nAvailable columns:")
print(chunk_df.columns.tolist())

display(chunk_df.head(3))

In [ ]:
import re
import nltk
import numpy as np
import pandas as pd

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer

nltk.download("stopwords")

# English stopwords
stop_words = set(stopwords.words("english"))

# Prepare dialogue text
coherence_corpus = (
    chunk_df["text"]
    .fillna("")
    .astype(str)
    .tolist()
)

# Tokenization and preprocessing
def coherence_tokenizer(text):
    tokens = re.findall(r"\b[a-zA-Z]{2,}\b", text.lower())
    return [
        word for word in tokens
        if word not in stop_words
    ]

# Build document-word matrix
coherence_vectorizer = CountVectorizer(
    tokenizer=coherence_tokenizer,
    token_pattern=None,
    lowercase=False,
    binary=True,
    min_df=2
)

coherence_matrix = coherence_vectorizer.fit_transform(
    coherence_corpus
)

coherence_vocabulary = set(
    coherence_vectorizer.get_feature_names_out()
)

print("=" * 75)
print("ICMB-130 — NPMI CORPUS PREPARATION")
print("=" * 75)

print("Total dialogue chunks:", len(coherence_corpus))
print("Document-word matrix:", coherence_matrix.shape)
print("Vocabulary size:", len(coherence_vocabulary))
print("Average vocabulary words per chunk:",
      round(coherence_matrix.sum() / len(coherence_corpus), 2))

print("\nSample processed tokens:")
print(coherence_tokenizer(coherence_corpus[0])[:30])

In [ ]:
import numpy as np
import pandas as pd
import os

from itertools import combinations
from math import log

# Convert document-word matrix to binary co-occurrence matrix
binary_matrix = coherence_matrix.astype(bool).astype(np.int32)

# Document frequency of each word
word_document_frequency = np.asarray(
    binary_matrix.sum(axis=0)
).ravel()

# Word-to-column mapping
word_to_index = {
    word: index
    for index, word in enumerate(
        coherence_vectorizer.get_feature_names_out()
    )
}

TOTAL_DOCUMENTS = binary_matrix.shape[0]

def calculate_npmi(word1, word2):

    if word1 not in word_to_index or word2 not in word_to_index:
        return np.nan

    idx1 = word_to_index[word1]
    idx2 = word_to_index[word2]

    freq1 = word_document_frequency[idx1]
    freq2 = word_document_frequency[idx2]

    cooccurrence = int(
        binary_matrix[:, idx1]
        .multiply(binary_matrix[:, idx2])
        .sum()
    )

    if cooccurrence == 0:
        return -1.0

    p1 = freq1 / TOTAL_DOCUMENTS
    p2 = freq2 / TOTAL_DOCUMENTS
    p12 = cooccurrence / TOTAL_DOCUMENTS

    pmi = log(p12 / (p1 * p2))
    npmi = pmi / (-log(p12))

    return npmi


def evaluate_topic_coherence(topic_terms, topic_column, word_column):

    results = []

    for topic_id, group in topic_terms.groupby(topic_column):

        words = (
            group.sort_values("term_rank")[word_column]
            .astype(str)
            .head(10)
            .tolist()
        )

        scores = [
            calculate_npmi(w1, w2)
            for w1, w2 in combinations(words, 2)
        ]

        valid_scores = [
            score for score in scores
            if not np.isnan(score)
        ]

        results.append({
            "Topic": topic_id,
            "Top_Word_Count": len(words),
            "Valid_Pairs": len(valid_scores),
            "Total_Pairs": len(scores),
            "NPMI_Coherence": (
                np.mean(valid_scores)
                if valid_scores else np.nan
            )
        })

    return pd.DataFrame(results)


# Evaluate both models
baseline_coherence = evaluate_topic_coherence(
    baseline_terms,
    topic_column="topic",
    word_column="term"
)

experiment2_coherence = evaluate_topic_coherence(
    experiment2_terms,
    topic_column="topic_id",
    word_column="term"
)

print("=" * 75)
print("ICMB-130 — NPMI TOPIC COHERENCE RESULTS")
print("=" * 75)

print("\nBASELINE")
print("Topics evaluated:", len(baseline_coherence))
display(baseline_coherence.head(10))

print("\nEXPERIMENT 2")
print("Topics evaluated:", len(experiment2_coherence))
display(experiment2_coherence.head(10))

In [ ]:
def diagnose_coherence_coverage(
    topic_terms,
    coherence_results,
    topic_column,
    word_column,
    model_name
):
    rows = []

    for topic_id, group in topic_terms.groupby(topic_column):

        words = (
            group.sort_values("term_rank")[word_column]
            .astype(str)
            .head(10)
            .tolist()
        )

        # Count keywords present in the corpus vocabulary
        present_words = [
            word for word in words
            if word in coherence_vocabulary
        ]

        missing_words = [
            word for word in words
            if word not in coherence_vocabulary
        ]

        result_row = coherence_results[
            coherence_results["Topic"] == topic_id
        ]

        valid_pairs = int(result_row["Valid_Pairs"].iloc[0])
        total_pairs = int(result_row["Total_Pairs"].iloc[0])
        npmi_score = float(
            result_row["NPMI_Coherence"].iloc[0]
        )

        rows.append({
            "Model": model_name,
            "Topic": topic_id,
            "Keywords_Present": len(present_words),
            "Keywords_Missing": len(missing_words),
            "Missing_Keywords": ", ".join(missing_words),
            "Valid_Pairs": valid_pairs,
            "Total_Pairs": total_pairs,
            "Pair_Coverage_Pct": round(
                valid_pairs / total_pairs * 100, 2
            ),
            "NPMI_Coherence": npmi_score
        })

    return pd.DataFrame(rows)


baseline_diagnostic = diagnose_coherence_coverage(
    baseline_terms,
    baseline_coherence,
    "topic",
    "term",
    "Baseline"
)

experiment2_diagnostic = diagnose_coherence_coverage(
    experiment2_terms,
    experiment2_coherence,
    "topic_id",
    "term",
    "Experiment 2"
)

combined_diagnostic = pd.concat(
    [baseline_diagnostic, experiment2_diagnostic],
    ignore_index=True
)

# Model-level summary
coverage_summary = (
    combined_diagnostic
    .groupby("Model")
    .agg(
        Total_Topics=("Topic", "count"),
        Mean_Keyword_Coverage=("Keywords_Present", "mean"),
        Mean_Valid_Pair_Coverage=("Pair_Coverage_Pct", "mean"),
        Topics_Below_50_Pct_Coverage=(
            "Pair_Coverage_Pct",
            lambda x: (x < 50).sum()
        ),
        Topics_Below_80_Pct_Coverage=(
            "Pair_Coverage_Pct",
            lambda x: (x < 80).sum()
        ),
        Mean_NPMI=("NPMI_Coherence", "mean"),
        Median_NPMI=("NPMI_Coherence", "median")
    )
    .round(3)
    .reset_index()
)

print("=" * 75)
print("ICMB-130 — NPMI COVERAGE DIAGNOSTICS")
print("=" * 75)

display(coverage_summary)

print("\nBASELINE — 10 LOWEST COHERENCE TOPICS")
display(
    baseline_diagnostic
    .sort_values("NPMI_Coherence")
    .head(10)
)

print("\nEXPERIMENT 2 — 10 LOWEST COHERENCE TOPICS")
display(
    experiment2_diagnostic
    .sort_values("NPMI_Coherence")
    .head(10)
)

In [ ]:
# Inspect the lowest-coherence topics with their original keywords

def inspect_low_coherence_topics(
    diagnostic_df,
    inventory_df,
    terms_df,
    topic_column,
    term_topic_column,
    model_name,
    n=10
):
    lowest_topics = (
        diagnostic_df
        .sort_values("NPMI_Coherence")
        .head(n)["Topic"]
        .tolist()
    )

    records = []

    for topic_id in lowest_topics:

        inventory_row = inventory_df[
            inventory_df["Topic"] == topic_id
        ]

        term_rows = (
            terms_df[terms_df[term_topic_column] == topic_id]
            .sort_values("term_rank")
        )

        if inventory_row.empty:
            continue

        row = inventory_row.iloc[0]

        records.append({
            "Model": model_name,
            "Topic": topic_id,
            "Topic_Name": row["Name"],
            "Top_10_Keywords": ", ".join(
                term_rows["term"].head(10).astype(str)
            ),
            "Representative_Docs": row["Representative_Docs"],
            "Movie_Coverage": row["unique_movies"],
            "Largest_Movie": row["largest_contributing_movie"]
        })

    return pd.DataFrame(records)


baseline_low_topics = inspect_low_coherence_topics(
    baseline_diagnostic,
    baseline_inventory,
    baseline_terms,
    "Topic",
    "topic",
    "Baseline"
)

experiment2_low_topics = inspect_low_coherence_topics(
    experiment2_diagnostic,
    experiment2_inventory,
    experiment2_terms,
    "Topic",
    "topic_id",
    "Experiment 2"
)

print("=" * 75)
print("BASELINE — LOW COHERENCE TOPIC REVIEW")
print("=" * 75)

display(baseline_low_topics)

print("\n" + "=" * 75)
print("EXPERIMENT 2 — LOW COHERENCE TOPIC REVIEW")
print("=" * 75)

display(experiment2_low_topics)

In [ ]:
import os
import pandas as pd

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

assignment_files = {
    "Baseline": "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv",
    "Experiment 2": "BERTOPIC_V2_CHUNK_TOPIC_ASSIGNMENTS.csv"
}

for model_name, filename in assignment_files.items():

    path = os.path.join(OUTPUT_DIR, filename)

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    if os.path.exists(path):
        df = pd.read_csv(path)

        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(3))
    else:
        print("FILE NOT FOUND:", filename)

In [ ]:
# Load both assignment datasets

baseline_assignments = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
    )
)

experiment2_assignments = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "BERTOPIC_V2_CHUNK_TOPIC_ASSIGNMENTS.csv"
    )
)

# Check unique chunk IDs
baseline_ids = set(baseline_assignments["source_chunk_id"])
experiment2_ids = set(experiment2_assignments["source_chunk_id"])
original_ids = set(chunk_df["source_chunk_id"])

print("=" * 75)
print("ICMB-130 — CHUNK ALIGNMENT VALIDATION")
print("=" * 75)

print("Baseline rows:", len(baseline_assignments))
print("Experiment 2 rows:", len(experiment2_assignments))
print("Original corpus rows:", len(chunk_df))

print("\nUnique chunk IDs:")
print("Baseline:", baseline_assignments["source_chunk_id"].nunique())
print("Experiment 2:", experiment2_assignments["source_chunk_id"].nunique())
print("Original:", chunk_df["source_chunk_id"].nunique())

print("\nChunk ID comparison:")
print("Baseline vs Experiment 2 identical:", baseline_ids == experiment2_ids)
print("Baseline vs Original identical:", baseline_ids == original_ids)
print("Experiment 2 vs Original identical:", experiment2_ids == original_ids)

# Check duplicate chunk IDs
print("\nDuplicate chunk IDs:")
print("Baseline:", baseline_assignments["source_chunk_id"].duplicated().sum())
print("Experiment 2:", experiment2_assignments["source_chunk_id"].duplicated().sum())

# Verify text alignment by chunk ID
baseline_text = baseline_assignments.set_index("source_chunk_id")["text"].sort_index()
experiment2_text = experiment2_assignments.set_index("source_chunk_id")["text"].sort_index()

print("\nText alignment:")
print(
    "Baseline and Experiment 2 text identical:",
    baseline_text.equals(experiment2_text)
)

In [ ]:
# Identify duplicated chunk IDs in the original corpus

duplicate_chunks = chunk_df[
    chunk_df["source_chunk_id"].duplicated(keep=False)
].sort_values("source_chunk_id")

print("=" * 75)
print("ICMB-130 — DUPLICATE CHUNK INVESTIGATION")
print("=" * 75)

print("Rows involved in duplicate IDs:", len(duplicate_chunks))
print(
    "Unique duplicated chunk IDs:",
    duplicate_chunks["source_chunk_id"].nunique()
)

print("\nFirst 15 duplicated records:")

display(
    duplicate_chunks[
        [
            "movie_record_id",
            "movie_title",
            "source_chunk_id",
            "part_number",
            "token_count",
            "text"
        ]
    ].head(15)
)

# Check whether duplicate IDs have identical text
duplicate_text_check = (
    duplicate_chunks
    .groupby("source_chunk_id")
    .agg(
        occurrence_count=("text", "size"),
        unique_text_versions=("text", "nunique"),
        unique_movies=("movie_record_id", "nunique")
    )
    .reset_index()
)

print("\nDuplicate ID classification:")
display(
    duplicate_text_check[
        "unique_text_versions"
    ].value_counts().rename_axis(
        "unique_text_versions"
    ).reset_index(name="chunk_id_count")
)

print("\nDuplicate IDs associated with multiple movies:",
      (duplicate_text_check["unique_movies"] > 1).sum())

In [ ]:
# Create unique segment identifiers without modifying original IDs

for df in [
    chunk_df,
    baseline_assignments,
    experiment2_assignments
]:
    df["segment_id"] = (
        df["movie_record_id"].astype(str)
        + "_"
        + df["source_chunk_id"].astype(str)
        + "_part_"
        + df["part_number"].astype(str)
    )

print("=" * 75)
print("ICMB-130 — UNIQUE SEGMENT ID VALIDATION")
print("=" * 75)

print("Original corpus:")
print("Total rows:", len(chunk_df))
print("Unique segment IDs:", chunk_df["segment_id"].nunique())
print("Duplicate segment IDs:", chunk_df["segment_id"].duplicated().sum())

print("\nBaseline assignments:")
print("Total rows:", len(baseline_assignments))
print("Unique segment IDs:", baseline_assignments["segment_id"].nunique())
print("Duplicate segment IDs:", baseline_assignments["segment_id"].duplicated().sum())

print("\nExperiment 2 assignments:")
print("Total rows:", len(experiment2_assignments))
print("Unique segment IDs:", experiment2_assignments["segment_id"].nunique())
print("Duplicate segment IDs:", experiment2_assignments["segment_id"].duplicated().sum())

# Verify composite IDs and text alignment
baseline_check = (
    baseline_assignments
    .set_index("segment_id")["text"]
    .sort_index()
)

experiment2_check = (
    experiment2_assignments
    .set_index("segment_id")["text"]
    .sort_index()
)

original_check = (
    chunk_df
    .set_index("segment_id")["text"]
    .sort_index()
)

print("\nAlignment validation:")
print("Baseline vs Original:", baseline_check.equals(original_check))
print("Experiment 2 vs Original:", experiment2_check.equals(original_check))

In [ ]:
# ============================================================
# ICMB-130 — PREPARE TOPIC-SPECIFIC NPMI INPUTS
# ============================================================

import pandas as pd

# Exclude outlier chunks
baseline_topic_data = baseline_assignments[
    baseline_assignments["topic"] != -1
].copy()

experiment2_topic_data = experiment2_assignments[
    experiment2_assignments["topic"] != -1
].copy()

# Confirm unique segment-level records
assert baseline_topic_data["segment_id"].is_unique
assert experiment2_topic_data["segment_id"].is_unique

# Group dialogue segments by their assigned topic
baseline_topic_docs = (
    baseline_topic_data
    .groupby("topic")["text"]
    .apply(list)
)

experiment2_topic_docs = (
    experiment2_topic_data
    .groupby("topic")["text"]
    .apply(list)
)

print("=" * 70)
print("ICMB-130 — TOPIC-SPECIFIC NPMI INPUT VALIDATION")
print("=" * 70)

print("\nBASELINE")
print("Number of topics:", len(baseline_topic_docs))
print("Assigned segments:", len(baseline_topic_data))
print("Total grouped segments:", sum(len(x) for x in baseline_topic_docs))

print("\nEXPERIMENT 2")
print("Number of topics:", len(experiment2_topic_docs))
print("Assigned segments:", len(experiment2_topic_data))
print("Total grouped segments:", sum(len(x) for x in experiment2_topic_docs))

print("\nSEGMENTS PER TOPIC — BASELINE")
print(baseline_topic_docs.apply(len).describe())

print("\nSEGMENTS PER TOPIC — EXPERIMENT 2")
print(experiment2_topic_docs.apply(len).describe())

print("\nValidation complete.")

In [ ]:
# ============================================================
# ICMB-130 — PREPARE UNIGRAM AND BIGRAM FEATURES
# ============================================================

import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

# Consistent English tokenizer
def coherence_tokenizer(text):
    tokens = re.findall(r"\b[a-zA-Z]{2,}\b", str(text).lower())
    return [
        token for token in tokens
        if token not in ENGLISH_STOP_WORDS
    ]

# Build unigram + bigram binary document-term matrices
coherence_vectorizer = CountVectorizer(
    tokenizer=coherence_tokenizer,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    binary=True,
    min_df=1
)

# Fit on the complete original corpus to establish a common feature vocabulary
coherence_matrix = coherence_vectorizer.fit_transform(
    chunk_df["text"].fillna("").astype(str)
)

coherence_vocabulary = set(coherence_vectorizer.get_feature_names_out())

print("=" * 70)
print("ICMB-130 — COHERENCE FEATURE PREPARATION")
print("=" * 70)

print("Original dialogue segments:", len(chunk_df))
print("Feature matrix shape:", coherence_matrix.shape)
print("Unique unigram + bigram features:", len(coherence_vocabulary))

print("\nSample features:")
print(list(coherence_vectorizer.get_feature_names_out()[:30]))

print("\nFeature preparation completed.")

In [ ]:
# ============================================================
# ICMB-130 — REVISED COHERENCE FEATURE PREPARATION
# Aligned with original BERTopic vectorizer
# ============================================================

from sklearn.feature_extraction.text import CountVectorizer

coherence_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.80,
    max_features=30000,
    binary=True
)

coherence_matrix = coherence_vectorizer.fit_transform(
    chunk_df["text"].fillna("").astype(str)
)

coherence_vocabulary = set(
    coherence_vectorizer.get_feature_names_out()
)

print("=" * 70)
print("ICMB-130 — REVISED COHERENCE FEATURE VALIDATION")
print("=" * 70)

print("Original dialogue segments:", len(chunk_df))
print("Feature matrix shape:", coherence_matrix.shape)
print("Unique features:", len(coherence_vocabulary))

print("\nSample features:")
print(list(coherence_vectorizer.get_feature_names_out()[:30]))

print("\nVectorizer configuration:")
print("N-gram range:", coherence_vectorizer.ngram_range)
print("Minimum document frequency:", coherence_vectorizer.min_df)
print("Maximum document frequency:", coherence_vectorizer.max_df)
print("Maximum features:", coherence_vectorizer.max_features)

print("\nRevised feature preparation completed.")

In [ ]:
# ============================================================
# ICMB-130 — TOPIC KEYWORD COVERAGE VALIDATION
# ============================================================

import os
import pandas as pd

# Define the output directory explicitly
output_dir = (
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

# Load topic keyword files
baseline_terms = pd.read_csv(
    os.path.join(output_dir, "BERTOPIC_TOPIC_TERMS.csv")
)

experiment2_terms = pd.read_csv(
    os.path.join(output_dir, "BERTOPIC_V2_TOPIC_TERMS.csv")
)

# Normalize vocabulary
coherence_vocabulary = {
    str(term).strip().lower()
    for term in coherence_vocabulary
}

def evaluate_keyword_coverage(terms_df, topic_col):
    results = []

    for topic_id, group in terms_df.groupby(topic_col):

        top_terms = (
            group.sort_values("term_rank")["term"]
            .head(10)
            .astype(str)
            .str.strip()
            .str.lower()
            .tolist()
        )

        present_terms = [
            term for term in top_terms
            if term in coherence_vocabulary
        ]

        missing_terms = [
            term for term in top_terms
            if term not in coherence_vocabulary
        ]

        results.append({
            "topic_id": topic_id,
            "total_keywords": len(top_terms),
            "keywords_present": len(present_terms),
            "keywords_missing": len(missing_terms),
            "coverage_pct": len(present_terms) / len(top_terms) * 100,
            "missing_terms": ", ".join(missing_terms)
        })

    return pd.DataFrame(results)


baseline_keyword_coverage = evaluate_keyword_coverage(
    baseline_terms, "topic"
)

experiment2_keyword_coverage = evaluate_keyword_coverage(
    experiment2_terms, "topic_id"
)

print("=" * 70)
print("ICMB-130 — TOPIC KEYWORD COVERAGE")
print("=" * 70)

for name, df in [
    ("BASELINE", baseline_keyword_coverage),
    ("EXPERIMENT 2", experiment2_keyword_coverage)
]:
    print(f"\n{name}")
    print("Total topics:", len(df))
    print("Average keywords present:",
          round(df["keywords_present"].mean(), 2), "/ 10")
    print("Average keyword coverage:",
          round(df["coverage_pct"].mean(), 2), "%")
    print("Topics with all 10 keywords:",
          (df["keywords_present"] == 10).sum())
    print("Topics with fewer than 5 keywords:",
          (df["keywords_present"] < 5).sum())

print("\nBASELINE — LOWEST COVERAGE TOPICS")
print(
    baseline_keyword_coverage
    .sort_values("coverage_pct")
    .head(10)
    .to_string(index=False)
)

print("\nEXPERIMENT 2 — LOWEST COVERAGE TOPICS")
print(
    experiment2_keyword_coverage
    .sort_values("coverage_pct")
    .head(10)
    .to_string(index=False)
)

In [ ]:
# ============================================================
# ICMB-130 — INVESTIGATE MISSING TOPIC KEYWORDS
# ============================================================

print("=" * 70)
print("MISSING KEYWORD INVESTIGATION")
print("=" * 70)

print("\nBASELINE")
print("Total missing term values:", baseline_terms["term"].isna().sum())

print("\nEXPERIMENT 2")
print("Total missing term values:", experiment2_terms["term"].isna().sum())

print("\nEXPERIMENT 2 — TOPIC 91")
print(
    experiment2_terms[
        experiment2_terms["topic_id"] == 91
    ].sort_values("term_rank").to_string(index=False)
)

print("\nEXPERIMENT 2 — ALL TOPICS WITH MISSING TERMS")
missing_exp2 = (
    experiment2_terms[
        experiment2_terms["term"].isna()
    ]
    .groupby("topic_id")
    .size()
    .sort_values(ascending=False)
)

print(missing_exp2.to_string())

print("\nBASELINE — ALL TOPICS WITH MISSING TERMS")
missing_baseline = (
    baseline_terms[
        baseline_terms["term"].isna()
    ]
    .groupby("topic")
    .size()
    .sort_values(ascending=False)
)

print(missing_baseline.to_string())

In [ ]:
# ============================================================
# ICMB-130 — CLEAN TOPIC KEYWORD LISTS
# ============================================================

def prepare_clean_topic_keywords(terms_df, topic_col):

    clean_topic_keywords = {}

    for topic_id, group in terms_df.groupby(topic_col):

        # Remove empty keyword entries
        valid_group = group.dropna(subset=["term"]).copy()

        # Sort by original keyword rank
        valid_group = valid_group.sort_values("term_rank")

        # Retain up to 10 actual keywords
        keywords = (
            valid_group["term"]
            .astype(str)
            .str.strip()
            .str.lower()
            .tolist()
        )

        clean_topic_keywords[topic_id] = keywords[:10]

    return clean_topic_keywords


baseline_topic_keywords = prepare_clean_topic_keywords(
    baseline_terms, "topic"
)

experiment2_topic_keywords = prepare_clean_topic_keywords(
    experiment2_terms, "topic_id"
)

print("=" * 70)
print("ICMB-130 — CLEAN KEYWORD VALIDATION")
print("=" * 70)

for name, keyword_dict in [
    ("BASELINE", baseline_topic_keywords),
    ("EXPERIMENT 2", experiment2_topic_keywords)
]:

    keyword_counts = pd.Series(
        [len(words) for words in keyword_dict.values()]
    )

    print(f"\n{name}")
    print("Total topics:", len(keyword_dict))
    print("Average actual keywords:", round(keyword_counts.mean(), 2))
    print("Topics with 10 keywords:", (keyword_counts == 10).sum())
    print("Topics with fewer than 10 keywords:", (keyword_counts < 10).sum())

print("\nEXPERIMENT 2 — TOPIC 91")
print(experiment2_topic_keywords[91])

print("\nEXPERIMENT 2 — TOPIC 101")
print(experiment2_topic_keywords[101])

In [ ]:
# ============================================================
# ICMB-130 — TOPIC-SPECIFIC NPMI CALCULATION
# ============================================================

import numpy as np
import pandas as pd
from itertools import combinations
from scipy.sparse import csr_matrix

# Map segment IDs to row positions in the original feature matrix
segment_to_position = pd.Series(
    np.arange(len(chunk_df)),
    index=chunk_df["segment_id"]
)

def calculate_topic_npmi(assignments, topic_keywords, model_name):

    results = []

    for topic_id, keywords in topic_keywords.items():

        # Get segments assigned to this topic
        topic_segments = assignments.loc[
            assignments["topic"] == topic_id,
            "segment_id"
        ]

        row_positions = segment_to_position.loc[
            topic_segments
        ].to_numpy()

        # Restrict feature matrix to this topic's segments
        topic_matrix = coherence_matrix[row_positions]

        n_docs = topic_matrix.shape[0]

        # Identify available keyword features
        valid_keywords = [
            keyword for keyword in keywords
            if keyword in coherence_vocabulary
        ]

        missing_keywords = [
            keyword for keyword in keywords
            if keyword not in coherence_vocabulary
        ]

        total_pairs = len(list(combinations(valid_keywords, 2)))

        pair_scores = []
        zero_pairs = 0

        # Calculate NPMI for every valid keyword pair
        for word_a, word_b in combinations(valid_keywords, 2):

            idx_a = coherence_vectorizer.vocabulary_[word_a]
            idx_b = coherence_vectorizer.vocabulary_[word_b]

            count_a = int(topic_matrix[:, idx_a].sum())
            count_b = int(topic_matrix[:, idx_b].sum())

            cooccur = int(
                topic_matrix[:, idx_a]
                .multiply(topic_matrix[:, idx_b])
                .sum()
            )

            # Skip features absent from this topic's documents
            if count_a == 0 or count_b == 0:
                continue

            if cooccur == 0:
                # Conventional zero-co-occurrence NPMI convention
                npmi = -1.0
                zero_pairs += 1

            else:
                p_a = count_a / n_docs
                p_b = count_b / n_docs
                p_ab = cooccur / n_docs

                npmi = (
                    np.log(p_ab / (p_a * p_b))
                    / -np.log(p_ab)
                )

            pair_scores.append(float(npmi))

        valid_pairs = len(pair_scores)

        results.append({
            "model": model_name,
            "topic_id": topic_id,
            "topic_segments": n_docs,
            "total_keywords": len(keywords),
            "keywords_in_vocabulary": len(valid_keywords),
            "keywords_missing": len(missing_keywords),
            "valid_pairs": valid_pairs,
            "total_possible_pairs": total_pairs,
            "pair_coverage_pct": (
                valid_pairs / total_pairs * 100
                if total_pairs else np.nan
            ),
            "zero_cooccurrence_pairs": zero_pairs,
            "topic_specific_npmi": (
                np.mean(pair_scores)
                if pair_scores else np.nan
            ),
            "missing_keyword_list": ", ".join(missing_keywords)
        })

    return pd.DataFrame(results)


# Calculate scores separately for both models
baseline_npmi = calculate_topic_npmi(
    baseline_assignments,
    baseline_topic_keywords,
    "Baseline"
)

experiment2_npmi = calculate_topic_npmi(
    experiment2_assignments,
    experiment2_topic_keywords,
    "Experiment 2"
)

print("=" * 70)
print("ICMB-130 — TOPIC-SPECIFIC NPMI RESULTS")
print("=" * 70)

for name, result in [
    ("BASELINE", baseline_npmi),
    ("EXPERIMENT 2", experiment2_npmi)
]:

    print(f"\n{name}")
    print("Topics evaluated:", len(result))
    print("Mean topic-specific NPMI:",
          round(result["topic_specific_npmi"].mean(), 4))
    print("Median topic-specific NPMI:",
          round(result["topic_specific_npmi"].median(), 4))
    print("Topics without a valid score:",
          result["topic_specific_npmi"].isna().sum())
    print("Mean pair coverage:",
          round(result["pair_coverage_pct"].mean(), 2), "%")

print("\nBASELINE — FIRST 10 TOPICS")
print(baseline_npmi.head(10).to_string(index=False))

print("\nEXPERIMENT 2 — FIRST 10 TOPICS")
print(experiment2_npmi.head(10).to_string(index=False))

In [ ]:
# ============================================================
# ICMB-130 — NPMI RELIABILITY DIAGNOSTICS
# ============================================================

def create_npmi_diagnostics(npmi_df):

    df = npmi_df.copy()

    df["zero_pair_pct"] = (
        df["zero_cooccurrence_pairs"]
        / df["valid_pairs"].replace(0, np.nan)
    ) * 100

    df["support_category"] = np.select(
        [
            df["pair_coverage_pct"] < 50,
            df["pair_coverage_pct"] < 80,
            df["pair_coverage_pct"] < 100
        ],
        [
            "Low pair coverage",
            "Moderate pair coverage",
            "Partial pair coverage"
        ],
        default="Complete pair coverage"
    )

    return df


baseline_npmi_diagnostic = create_npmi_diagnostics(baseline_npmi)
experiment2_npmi_diagnostic = create_npmi_diagnostics(experiment2_npmi)

for name, df in [
    ("BASELINE", baseline_npmi_diagnostic),
    ("EXPERIMENT 2", experiment2_npmi_diagnostic)
]:

    print("\n" + "=" * 70)
    print(name, "— RELIABILITY SUMMARY")
    print("=" * 70)

    print("\nPair coverage categories:")
    print(df["support_category"].value_counts().to_string())

    print("\nTopics with pair coverage below 80%:",
          (df["pair_coverage_pct"] < 80).sum())

    print("Topics with zero-pair percentage above 50%:",
          (df["zero_pair_pct"] > 50).sum())

    print("Topics with NPMI below -0.50:",
          (df["topic_specific_npmi"] < -0.50).sum())

    print("\nLOWEST 10 NPMI TOPICS:")
    print(
        df.sort_values("topic_specific_npmi")[
            [
                "topic_id",
                "topic_segments",
                "keywords_in_vocabulary",
                "valid_pairs",
                "pair_coverage_pct",
                "zero_cooccurrence_pairs",
                "zero_pair_pct",
                "topic_specific_npmi",
                "missing_keyword_list"
            ]
        ].head(10).to_string(index=False)
    )

    print("\nHIGHEST 10 NPMI TOPICS:")
    print(
        df.sort_values("topic_specific_npmi", ascending=False)[
            [
                "topic_id",
                "topic_segments",
                "valid_pairs",
                "pair_coverage_pct",
                "zero_pair_pct",
                "topic_specific_npmi"
            ]
        ].head(10).to_string(index=False)
    )

In [ ]:
# ============================================================
# ICMB-130 — KEYWORD SUPPORT WITHIN EACH TOPIC
# ============================================================

from itertools import combinations
import numpy as np
import pandas as pd

def calculate_keyword_support(assignments, topic_keywords, model_name):

    results = []

    for topic_id, keywords in topic_keywords.items():

        topic_segments = assignments.loc[
            assignments["topic"] == topic_id, "segment_id"
        ]

        row_positions = segment_to_position.loc[
            topic_segments
        ].to_numpy()

        topic_matrix = coherence_matrix[row_positions]

        # Keywords available in the overall vocabulary
        vocabulary_keywords = [
            word for word in keywords
            if word in coherence_vocabulary
        ]

        # Count how many topic segments contain each keyword
        keyword_counts = {}

        for word in vocabulary_keywords:
            col_idx = coherence_vectorizer.vocabulary_[word]
            keyword_counts[word] = int(topic_matrix[:, col_idx].sum())

        # Retain only keywords actually occurring in this topic
        occurring_keywords = [
            word for word, count in keyword_counts.items()
            if count > 0
        ]

        absent_keywords = [
            word for word, count in keyword_counts.items()
            if count == 0
        ]

        # All pairs among keywords actually present in the topic
        topic_pairs = list(combinations(occurring_keywords, 2))

        zero_pairs = 0

        for word_a, word_b in topic_pairs:

            idx_a = coherence_vectorizer.vocabulary_[word_a]
            idx_b = coherence_vectorizer.vocabulary_[word_b]

            cooccur = int(
                topic_matrix[:, idx_a]
                .multiply(topic_matrix[:, idx_b])
                .sum()
            )

            if cooccur == 0:
                zero_pairs += 1

        results.append({
            "model": model_name,
            "topic_id": topic_id,
            "topic_segments": len(topic_segments),
            "keywords_requested": len(keywords),
            "keywords_in_vocabulary": len(vocabulary_keywords),
            "keywords_occurring_in_topic": len(occurring_keywords),
            "keywords_absent_from_topic": len(absent_keywords),
            "possible_pairs_with_topic_support": len(topic_pairs),
            "zero_cooccurrence_pairs": zero_pairs,
            "zero_pair_pct": (
                zero_pairs / len(topic_pairs) * 100
                if topic_pairs else np.nan
            ),
            "absent_keyword_list": ", ".join(absent_keywords)
        })

    return pd.DataFrame(results)


baseline_keyword_support = calculate_keyword_support(
    baseline_assignments,
    baseline_topic_keywords,
    "Baseline"
)

experiment2_keyword_support = calculate_keyword_support(
    experiment2_assignments,
    experiment2_topic_keywords,
    "Experiment 2"
)

for name, df in [
    ("BASELINE", baseline_keyword_support),
    ("EXPERIMENT 2", experiment2_keyword_support)
]:

    print("\n" + "=" * 70)
    print(name, "— TOPIC KEYWORD SUPPORT")
    print("=" * 70)

    print("Average keywords occurring in topic:",
          round(df["keywords_occurring_in_topic"].mean(), 2))

    print("Topics with all available keywords occurring:",
          (
              df["keywords_occurring_in_topic"]
              == df["keywords_in_vocabulary"]
          ).sum())

    print("Topics with fewer than 5 occurring keywords:",
          (df["keywords_occurring_in_topic"] < 5).sum())

    print("Topics with fewer than 5 supported keyword pairs:",
          (df["possible_pairs_with_topic_support"] < 5).sum())

    print("\nTopics with the fewest occurring keywords:")

    print(
        df.sort_values(
            ["keywords_occurring_in_topic", "topic_segments"]
        )[
            [
                "topic_id",
                "topic_segments",
                "keywords_in_vocabulary",
                "keywords_occurring_in_topic",
                "possible_pairs_with_topic_support",
                "zero_cooccurrence_pairs",
                "zero_pair_pct",
                "absent_keyword_list"
            ]
        ].head(10).to_string(index=False)
    )

In [ ]:
# ============================================================
# ICMB-130 — FINAL NPMI ROBUSTNESS COMPARISON
# ============================================================

def final_npmi_robustness(npmi_df, support_df, model_name):

    df = npmi_df.merge(
        support_df[
            [
                "topic_id",
                "keywords_occurring_in_topic",
                "possible_pairs_with_topic_support"
            ]
        ],
        on="topic_id",
        how="left",
        validate="one_to_one"
    )

    # Explicit minimum-support rule
    df["supported_topic"] = (
        (df["keywords_occurring_in_topic"] >= 5)
        & (df["valid_pairs"] >= 10)
    )

    eligible = df[df["supported_topic"]].copy()

    # Recover pair-level scores for pooled mean
    pair_scores_all = []

    for topic_id, keywords in (
        baseline_topic_keywords.items()
        if model_name == "Baseline"
        else experiment2_topic_keywords.items()
    ):

        assignments = (
            baseline_assignments
            if model_name == "Baseline"
            else experiment2_assignments
        )

        topic_segments = assignments.loc[
            assignments["topic"] == topic_id, "segment_id"
        ]

        positions = segment_to_position.loc[
            topic_segments
        ].to_numpy()

        matrix = coherence_matrix[positions]

        occurring = []

        for word in keywords:
            if word not in coherence_vocabulary:
                continue

            idx = coherence_vectorizer.vocabulary_[word]

            if int(matrix[:, idx].sum()) > 0:
                occurring.append(word)

        for word_a, word_b in combinations(occurring, 2):

            idx_a = coherence_vectorizer.vocabulary_[word_a]
            idx_b = coherence_vectorizer.vocabulary_[word_b]

            count_a = int(matrix[:, idx_a].sum())
            count_b = int(matrix[:, idx_b].sum())

            cooccur = int(
                matrix[:, idx_a].multiply(matrix[:, idx_b]).sum()
            )

            n_docs = matrix.shape[0]

            if cooccur == 0:
                score = -1.0
            else:
                p_a = count_a / n_docs
                p_b = count_b / n_docs
                p_ab = cooccur / n_docs

                score = (
                    np.log(p_ab / (p_a * p_b))
                    / -np.log(p_ab)
                )

            pair_scores_all.append({
                "topic_id": topic_id,
                "npmi": float(score)
            })

    pair_scores_df = pd.DataFrame(pair_scores_all)

    # Pooled pair score across all topics
    pooled_all = pair_scores_df["npmi"].mean()

    eligible_topic_ids = set(eligible["topic_id"])

    pooled_supported = pair_scores_df.loc[
        pair_scores_df["topic_id"].isin(eligible_topic_ids),
        "npmi"
    ].mean()

    return {
        "model": model_name,
        "total_topics": len(df),
        "supported_topics": len(eligible),
        "excluded_topics": len(df) - len(eligible),
        "all_topic_macro_npmi": df["topic_specific_npmi"].mean(),
        "supported_topic_macro_npmi": eligible["topic_specific_npmi"].mean(),
        "all_pair_pooled_npmi": pooled_all,
        "supported_pair_pooled_npmi": pooled_supported,
        "mean_supported_keywords": eligible[
            "keywords_occurring_in_topic"
        ].mean(),
        "supported_topic_details": df
    }


baseline_final = final_npmi_robustness(
    baseline_npmi,
    baseline_keyword_support,
    "Baseline"
)

experiment2_final = final_npmi_robustness(
    experiment2_npmi,
    experiment2_keyword_support,
    "Experiment 2"
)

comparison = pd.DataFrame([
    {k: v for k, v in baseline_final.items()
     if k != "supported_topic_details"},
    {k: v for k, v in experiment2_final.items()
     if k != "supported_topic_details"}
])

print("=" * 75)
print("ICMB-130 — FINAL NPMI ROBUSTNESS COMPARISON")
print("=" * 75)

print(comparison.round(4).to_string(index=False))

print("\nBASELINE — EXCLUDED TOPICS")
print(
    baseline_final["supported_topic_details"]
    .query("supported_topic == False")[
        [
            "topic_id",
            "topic_segments",
            "keywords_occurring_in_topic",
            "valid_pairs",
            "topic_specific_npmi"
        ]
    ].to_string(index=False)
)

print("\nEXPERIMENT 2 — EXCLUDED TOPICS")
print(
    experiment2_final["supported_topic_details"]
    .query("supported_topic == False")[
        [
            "topic_id",
            "topic_segments",
            "keywords_occurring_in_topic",
            "valid_pairs",
            "topic_specific_npmi"
        ]
    ].to_string(index=False)
)

In [ ]:
# ============================================================
# ICMB-130 — SAVE FINAL NPMI EVALUATION
# ============================================================

import os
import pandas as pd

output_dir = (
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

os.makedirs(output_dir, exist_ok=True)

# Save detailed topic-level scores
baseline_npmi_diagnostic.to_csv(
    os.path.join(
        output_dir,
        "BERTOPIC_BASELINE_TOPIC_SPECIFIC_NPMI.csv"
    ),
    index=False
)

experiment2_npmi_diagnostic.to_csv(
    os.path.join(
        output_dir,
        "BERTOPIC_V2_TOPIC_SPECIFIC_NPMI.csv"
    ),
    index=False
)

# Save keyword support diagnostics
baseline_keyword_support.to_csv(
    os.path.join(
        output_dir,
        "BERTOPIC_BASELINE_KEYWORD_SUPPORT.csv"
    ),
    index=False
)

experiment2_keyword_support.to_csv(
    os.path.join(
        output_dir,
        "BERTOPIC_V2_KEYWORD_SUPPORT.csv"
    ),
    index=False
)

# Save consolidated robustness comparison
comparison.to_csv(
    os.path.join(
        output_dir,
        "BERTOPIC_NPMI_ROBUSTNESS_COMPARISON.csv"
    ),
    index=False
)

print("=" * 70)
print("ICMB-130 — NPMI RESULTS SAVED")
print("=" * 70)

saved_files = [
    "BERTOPIC_BASELINE_TOPIC_SPECIFIC_NPMI.csv",
    "BERTOPIC_V2_TOPIC_SPECIFIC_NPMI.csv",
    "BERTOPIC_BASELINE_KEYWORD_SUPPORT.csv",
    "BERTOPIC_V2_KEYWORD_SUPPORT.csv",
    "BERTOPIC_NPMI_ROBUSTNESS_COMPARISON.csv"
]

for filename in saved_files:
    filepath = os.path.join(output_dir, filename)
    print(filename, "| Exists:", os.path.exists(filepath))

print("\nNPMI evaluation files saved successfully.")

In [ ]:
# ============================================================
# ICMB-130 — INSPECT REPRESENTATIVE DOCUMENTS
# ============================================================

import os
import pandas as pd

output_dir = (
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/"
    "06_outputs/topic_modelling"
)

files_to_inspect = {
    "Baseline Topic Info": "BERTOPIC_TOPIC_INFO.csv",
    "Experiment 2 Topic Info": "BERTOPIC_V2_TOPIC_INFO.csv",
    "Baseline Topic Terms": "BERTOPIC_TOPIC_TERMS.csv",
    "Experiment 2 Topic Terms": "BERTOPIC_V2_TOPIC_TERMS.csv"
}

print("=" * 75)
print("ICMB-130 — REPRESENTATIVE DOCUMENT INSPECTION")
print("=" * 75)

for name, filename in files_to_inspect.items():

    filepath = os.path.join(output_dir, filename)

    print(f"\n{name}")
    print("-" * 60)

    if os.path.exists(filepath):
        df = pd.read_csv(filepath)

        print("File exists:", True)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())

        if "Representative_Docs" in df.columns:
            print("\nRepresentative document sample:")

            sample = df[df["Topic"] != -1].head(3)

            for _, row in sample.iterrows():
                print(f"\nTopic: {row['Topic']}")
                print(str(row["Representative_Docs"])[:1000])

    else:
        print("File exists:", False)

print("\nInspection completed.")

In [ ]:
# ============================================================
# ICMB-130 — SONG CONTAMINATION INVESTIGATION
# STEP 1: INSPECT SONG MARKERS
# ============================================================

import re
import pandas as pd

# Inspect original dialogue corpus
song_patterns = {
    "Music symbols": r"[♪♫🎵🎶]",
    "Singing labels": r"\b(singing|sings|sang|singer)\b",
    "Song labels": r"\b(song|lyrics|chorus)\b",
    "Music labels": r"\b(background music|music playing|music starts)\b",
    "Bracketed music": r"\[(music|song|singing|sings)\]",
    "Repeated words": r"\b(\w+)(?:\s+\1){2,}\b"
}

print("=" * 75)
print("ICMB-130 — SONG MARKER INVESTIGATION")
print("=" * 75)

total_segments = len(chunk_df)

for label, pattern in song_patterns.items():

    matches = chunk_df["text"].fillna("").str.contains(
        pattern,
        case=False,
        regex=True
    )

    print(
        f"{label}: {matches.sum()} segments "
        f"({matches.mean() * 100:.2f}%)"
    )

print("\nTOTAL SEGMENTS:", total_segments)

# Display examples containing explicit music markers
explicit_pattern = (
    r"[♪♫🎵🎶]"
    r"|\b(singing|sings|song|lyrics|chorus)\b"
    r"|\[(music|song|singing|sings)\]"
)

examples = chunk_df[
    chunk_df["text"].fillna("").str.contains(
        explicit_pattern,
        case=False,
        regex=True
    )
]

print("\nSAMPLE MATCHING SEGMENTS")
print("-" * 75)

for _, row in examples.head(10).iterrows():

    print(
        f"\nMovie: {row['movie_title']} | "
        f"Year: {row['year']} | "
        f"Segment: {row['segment_id']}"
    )

    print(str(row["text"])[:700])

print("\nInvestigation completed.")

In [ ]:
# ============================================================
# ICMB-130 — SONG CONTAMINATION DISTRIBUTION
# ============================================================

import pandas as pd

# Detect explicit musical symbols
song_symbol_mask = chunk_df["text"].fillna("").str.contains(
    r"[♪♫🎵🎶]",
    regex=True
)

song_marker_df = chunk_df.copy()
song_marker_df["has_music_symbol"] = song_symbol_mask

# Movie-level distribution
movie_song_summary = (
    song_marker_df.groupby(
        ["movie_record_id", "movie_title", "year"]
    )
    .agg(
        total_segments=("segment_id", "count"),
        music_symbol_segments=("has_music_symbol", "sum")
    )
    .reset_index()
)

movie_song_summary["music_segment_pct"] = (
    movie_song_summary["music_symbol_segments"]
    / movie_song_summary["total_segments"] * 100
)

# Year-level distribution
year_song_summary = (
    movie_song_summary.groupby("year")
    .agg(
        total_movies=("movie_title", "count"),
        movies_with_music=("music_symbol_segments",
                           lambda x: (x > 0).sum()),
        total_segments=("total_segments", "sum"),
        music_symbol_segments=("music_symbol_segments", "sum")
    )
    .reset_index()
)

year_song_summary["music_segment_pct"] = (
    year_song_summary["music_symbol_segments"]
    / year_song_summary["total_segments"] * 100
)

print("=" * 75)
print("ICMB-130 — SONG CONTAMINATION DISTRIBUTION")
print("=" * 75)

print("\nOVERALL")
print("Total movies:", movie_song_summary.shape[0])
print("Movies containing explicit music symbols:",
      (movie_song_summary["music_symbol_segments"] > 0).sum())
print("Total segments:", song_marker_df.shape[0])
print("Segments containing music symbols:", song_symbol_mask.sum())
print("Overall percentage:", round(song_symbol_mask.mean() * 100, 2), "%")

print("\nTOP 15 MOVIES BY MUSIC-SYMBOL SEGMENT PERCENTAGE")
print(
    movie_song_summary.sort_values(
        "music_segment_pct", ascending=False
    ).head(15).to_string(index=False)
)

print("\nYEAR-WISE DISTRIBUTION")
print(year_song_summary.to_string(index=False))

In [ ]:
# ============================================================
# ICMB-130 — SONG SEGMENT CONTEXT INSPECTION
# ============================================================

import pandas as pd

# Films with the highest explicit music-marker percentages
priority_movies = [
    "Enthiran",
    "Coolie",
    "Kantara: A Legend – Chapter 1",
    "Sanju",
    "Kaho Naa... Pyaar Hai",
    "Vikram",
    "PK",
    "Saiyaara",
    "Kantara"
]

# Preserve original corpus order
inspection_df = chunk_df.reset_index(drop=True).copy()

inspection_df["has_music_symbol"] = (
    inspection_df["text"].fillna("").str.contains(
        r"[♪♫🎵🎶]",
        regex=True
    )
)

print("=" * 75)
print("ICMB-130 — SONG SEGMENT CONTEXT INSPECTION")
print("=" * 75)

for movie in priority_movies:

    movie_positions = inspection_df.index[
        inspection_df["movie_title"] == movie
    ].tolist()

    marked_positions = [
        pos for pos in movie_positions
        if inspection_df.loc[pos, "has_music_symbol"]
    ]

    if not marked_positions:
        continue

    print("\n" + "=" * 75)
    print("MOVIE:", movie)
    print("Marked segments:", len(marked_positions))
    print("=" * 75)

    # Inspect up to two marked segments per movie
    for pos in marked_positions[:2]:

        print("\n--- MARKED SEGMENT ---")
        print("Segment ID:", inspection_df.loc[pos, "segment_id"])
        print("Text:", str(inspection_df.loc[pos, "text"])[:500])

        # Inspect immediate previous segment from same movie
        if (
            pos > 0
            and inspection_df.loc[pos - 1, "movie_record_id"]
            == inspection_df.loc[pos, "movie_record_id"]
        ):
            print("\n--- PREVIOUS SEGMENT ---")
            print(str(inspection_df.loc[pos - 1, "text"])[:350])

        # Inspect immediate next segment from same movie
        if (
            pos + 1 < len(inspection_df)
            and inspection_df.loc[pos + 1, "movie_record_id"]
            == inspection_df.loc[pos, "movie_record_id"]
        ):
            print("\n--- NEXT SEGMENT ---")
            print(str(inspection_df.loc[pos + 1, "text"])[:350])

print("\nContext inspection completed.")

In [ ]:
# ============================================================
# ICMB-130 — MEASURE EXPLICIT LYRIC CONTENT
# ============================================================

import re
import pandas as pd

def extract_music_spans(text):
    text = str(text)

    pattern = r"[♪♫🎵🎶](.*?)[♪♫🎵🎶]"

    spans = re.findall(pattern, text, flags=re.DOTALL)

    # Remove empty or whitespace-only spans
    spans = [span.strip() for span in spans if span.strip()]

    return spans


music_analysis = chunk_df.copy()

music_analysis["music_spans"] = (
    music_analysis["text"].fillna("").apply(extract_music_spans)
)

music_analysis["marked_lyric_chars"] = (
    music_analysis["music_spans"]
    .apply(lambda spans: sum(len(span) for span in spans))
)

music_analysis["total_text_chars"] = (
    music_analysis["text"].fillna("").str.len()
)

music_analysis["marked_lyric_pct"] = (
    music_analysis["marked_lyric_chars"]
    / music_analysis["total_text_chars"].replace(0, pd.NA)
    * 100
)

music_analysis["music_span_count"] = (
    music_analysis["music_spans"].apply(len)
)

marked_segments = music_analysis[
    music_analysis["text"].fillna("").str.contains(
        r"[♪♫🎵🎶]", regex=True
    )
].copy()

print("=" * 75)
print("ICMB-130 — EXPLICIT LYRIC CONTENT ANALYSIS")
print("=" * 75)

print("\nMarked segments:", len(marked_segments))

print(
    "Segments with extractable text between music symbols:",
    (marked_segments["music_span_count"] > 0).sum()
)

print(
    "Segments with no extractable lyric spans:",
    (marked_segments["music_span_count"] == 0).sum()
)

print(
    "\nAverage percentage of text enclosed by music symbols:",
    round(marked_segments["marked_lyric_pct"].mean(), 2), "%"
)

print("\nLYRIC-CONTENT DISTRIBUTION")

print(
    pd.cut(
        marked_segments["marked_lyric_pct"],
        bins=[-1, 0, 25, 50, 75, 100],
        labels=[
            "0% — No extractable lyrics",
            "1–25%",
            "26–50%",
            "51–75%",
            "76–100%"
        ]
    ).value_counts().sort_index().to_string()
)

print("\nSAMPLE SEGMENTS WITH HIGH LYRIC CONTENT")

high_lyric_samples = marked_segments.sort_values(
    "marked_lyric_pct", ascending=False
).head(10)

for _, row in high_lyric_samples.iterrows():

    print(
        f"\nMovie: {row['movie_title']} | "
        f"Year: {row['year']} | "
        f"Segment: {row['segment_id']}"
    )

    print("Marked lyric percentage:",
          round(row["marked_lyric_pct"], 2), "%")

    print("Extracted spans:")
    print(row["music_spans"][:5])

print("\nAnalysis completed.")

In [ ]:
# ============================================================
# ICMB-130 — SEGMENT-LEVEL SONG / DIALOGUE CLASSIFICATION
# ============================================================

import numpy as np
import pandas as pd

# Preserve all original segment data
song_classification = music_analysis.copy()

# Classify each segment
def classify_segment(row):

    has_music_marker = bool(
        re.search(r"[♪♫🎵🎶]", str(row["text"]))
    )

    if not has_music_marker:
        return "no_explicit_music_marker"

    if row["marked_lyric_pct"] == 0:
        return "music_marker_only"

    if row["marked_lyric_pct"] > 50:
        return "song_dominated"

    return "mixed_song_dialogue"


song_classification["content_class"] = (
    song_classification.apply(classify_segment, axis=1)
)

# Summary by classification
classification_summary = (
    song_classification
    .groupby("content_class")
    .agg(
        segment_count=("text", "size"),
        unique_movies=("movie_record_id", "nunique"),
        average_text_length=("total_text_chars", "mean")
    )
    .reset_index()
)

classification_summary["segment_percentage"] = (
    classification_summary["segment_count"]
    / len(song_classification) * 100
)

# Display results
print("=" * 75)
print("ICMB-130 — SONG / DIALOGUE CLASSIFICATION SUMMARY")
print("=" * 75)

print(f"\nTotal segments: {len(song_classification):,}")

print(
    classification_summary
    .sort_values("segment_count", ascending=False)
    .round(2)
    .to_string(index=False)
)

print("\nCLASSIFICATION CHECK")
print(
    song_classification["content_class"]
    .value_counts()
    .to_string()
)

# Verify that no segments were lost
assert len(song_classification) == len(music_analysis)
assert song_classification["content_class"].notna().all()

print("\nAll segments classified successfully.")

In [ ]:
# ============================================================
# ICMB-130 — FILM-LEVEL SONG CONTENT DISTRIBUTION
# ============================================================

# Identify the available movie title column
title_candidates = [
    "dataset_title", "movie_title", "tmdb_title",
    "title", "movie"
]

title_col = next(
    (col for col in title_candidates if col in song_classification.columns),
    None
)

if title_col is None:
    print("Available columns:", song_classification.columns.tolist())
    raise ValueError("Movie title column not found.")

# Aggregate segment classifications by film
film_song_summary = (
    song_classification
    .groupby("movie_record_id")
    .agg(
        movie_title=(title_col, "first"),
        year=("year", "first"),
        total_segments=("text", "size"),
        song_dominated_segments=(
            "content_class",
            lambda x: (x == "song_dominated").sum()
        ),
        mixed_segments=(
            "content_class",
            lambda x: (x == "mixed_song_dialogue").sum()
        ),
        music_marker_only_segments=(
            "content_class",
            lambda x: (x == "music_marker_only").sum()
        )
    )
    .reset_index()
)

# Calculate percentages
film_song_summary["song_dominated_pct"] = (
    film_song_summary["song_dominated_segments"]
    / film_song_summary["total_segments"] * 100
)

film_song_summary["mixed_pct"] = (
    film_song_summary["mixed_segments"]
    / film_song_summary["total_segments"] * 100
)

film_song_summary["explicit_lyric_or_mixed_pct"] = (
    (
        film_song_summary["song_dominated_segments"]
        + film_song_summary["mixed_segments"]
    )
    / film_song_summary["total_segments"] * 100
)

# Rank by song-dominated and mixed content
film_song_summary = film_song_summary.sort_values(
    ["explicit_lyric_or_mixed_pct", "song_dominated_segments"],
    ascending=False
)

print("=" * 85)
print("ICMB-130 — FILM-LEVEL SONG CONTENT DISTRIBUTION")
print("=" * 85)

print("\nTOP 15 FILMS BY EXPLICIT LYRIC / MIXED SEGMENT SHARE")

display(
    film_song_summary[
        [
            "movie_title",
            "year",
            "total_segments",
            "song_dominated_segments",
            "mixed_segments",
            "music_marker_only_segments",
            "explicit_lyric_or_mixed_pct"
        ]
    ].head(15).round(2)
)

print("\nOVERALL CHECK")
print("Films represented:", film_song_summary["movie_record_id"].nunique())
print(
    "Films with song-dominated segments:",
    (film_song_summary["song_dominated_segments"] > 0).sum()
)
print(
    "Films with mixed segments:",
    (film_song_summary["mixed_segments"] > 0).sum()
)

# Save the film-level summary
output_file = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_FILM_LEVEL_SONG_CONTENT_SUMMARY.csv"
)

film_song_summary.to_csv(output_file, index=False)

print("\nSaved:", output_file)

In [ ]:
# ============================================================
# ICMB-130 — SONG CONTENT vs BERTopic TOPIC ASSIGNMENTS
# ============================================================

# Load the two existing model assignment files
baseline_path = os.path.join(
    OUTPUT_DIR, "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

experiment2_path = os.path.join(
    OUTPUT_DIR, "BERTOPIC_V2_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

baseline_assignments = pd.read_csv(baseline_path)
experiment2_assignments = pd.read_csv(experiment2_path)

# Create a consistent segment identifier
def create_segment_id(df):
    df = df.copy()

    df["segment_id"] = (
        df["movie_record_id"].astype(str) + "_" +
        df["source_chunk_id"].astype(str) + "_" +
        df["part_number"].astype(str)
    )

    return df

song_data = create_segment_id(song_classification)
baseline_data = create_segment_id(baseline_assignments)
experiment2_data = create_segment_id(experiment2_assignments)

# Check that segment identifiers are unique
assert song_data["segment_id"].is_unique
assert baseline_data["segment_id"].is_unique
assert experiment2_data["segment_id"].is_unique

# Identify topic column names
def find_topic_column(df):
    for col in ["topic", "Topic", "topic_id"]:
        if col in df.columns:
            return col
    raise ValueError("Topic column not found.")

baseline_topic_col = find_topic_column(baseline_data)
experiment2_topic_col = find_topic_column(experiment2_data)

# Keep only the required assignment fields
baseline_data = baseline_data[
    ["segment_id", baseline_topic_col]
].rename(columns={baseline_topic_col: "baseline_topic"})

experiment2_data = experiment2_data[
    ["segment_id", experiment2_topic_col]
].rename(columns={experiment2_topic_col: "experiment2_topic"})

# Merge topic assignments with song classification
topic_song_analysis = (
    song_data[
        ["segment_id", "movie_title", "year", "content_class"]
    ]
    .merge(baseline_data, on="segment_id", how="left", validate="one_to_one")
    .merge(experiment2_data, on="segment_id", how="left", validate="one_to_one")
)

# Verify complete alignment
assert len(topic_song_analysis) == 33800
assert topic_song_analysis["baseline_topic"].notna().all()
assert topic_song_analysis["experiment2_topic"].notna().all()

# Summarize song-related segments by topic
def summarize_song_by_topic(df, topic_col):

    result = (
        df.groupby(topic_col)
        .agg(
            total_segments=("segment_id", "size"),
            song_dominated=(
                "content_class",
                lambda x: (x == "song_dominated").sum()
            ),
            mixed_song_dialogue=(
                "content_class",
                lambda x: (x == "mixed_song_dialogue").sum()
            ),
            music_marker_only=(
                "content_class",
                lambda x: (x == "music_marker_only").sum()
            )
        )
        .reset_index()
    )

    result["song_dominated_pct"] = (
        result["song_dominated"] / result["total_segments"] * 100
    )

    result["song_or_mixed_pct"] = (
        (
            result["song_dominated"]
            + result["mixed_song_dialogue"]
        ) / result["total_segments"] * 100
    )

    return result.sort_values(
        ["song_or_mixed_pct", "song_dominated"],
        ascending=False
    )


baseline_topic_song = summarize_song_by_topic(
    topic_song_analysis, "baseline_topic"
)

experiment2_topic_song = summarize_song_by_topic(
    topic_song_analysis, "experiment2_topic"
)

# Display top affected topics, excluding outlier topic -1
print("=" * 85)
print("BASELINE — TOP TOPICS BY SONG / MIXED CONTENT")
print("=" * 85)

display(
    baseline_topic_song[
        baseline_topic_song["baseline_topic"] != -1
    ].head(15).round(2)
)

print("\n" + "=" * 85)
print("EXPERIMENT 2 — TOP TOPICS BY SONG / MIXED CONTENT")
print("=" * 85)

display(
    experiment2_topic_song[
        experiment2_topic_song["experiment2_topic"] != -1
    ].head(15).round(2)
)

# Overall assignment check
print("\n" + "=" * 85)
print("ALIGNMENT CHECK")
print("=" * 85)

print("Total aligned segments:", len(topic_song_analysis))
print("Baseline outliers:", (topic_song_analysis["baseline_topic"] == -1).sum())
print("Experiment 2 outliers:", (topic_song_analysis["experiment2_topic"] == -1).sum())

print("\nAnalysis completed.")

In [ ]:
# ============================================================
# ICMB-130 — CORRECTED SONG-INFLUENCED TOPIC INSPECTION
# ============================================================

def inspect_topic_corrected(
    topic_id, model_name, info_df, terms_df, topic_col
):

    print("\n" + "=" * 85)
    print(f"{model_name} — TOPIC {topic_id}")
    print("=" * 85)

    # Topic information
    topic_row = info_df[info_df["Topic"] == topic_id]

    if not topic_row.empty:
        print("\nTOPIC INFORMATION")
        print(topic_row.to_string(index=False))

    # Top keywords
    topic_keywords = (
        terms_df[terms_df[topic_col] == topic_id]
        .sort_values("term_rank")
        .head(15)
    )

    print("\nTOP 15 KEYWORDS")
    print(topic_keywords.to_string(index=False))

    # Select topic segments
    assignment_col = (
        "experiment2_topic"
        if model_name == "EXPERIMENT 2"
        else "baseline_topic"
    )

    topic_segments = topic_song_analysis[
        topic_song_analysis[assignment_col] == topic_id
    ].copy()

    print("\nCLASSIFICATION COUNTS")
    print(topic_segments["content_class"].value_counts().to_string())

    # Song-related film contribution
    affected = topic_segments[
        topic_segments["content_class"].isin(
            ["song_dominated", "mixed_song_dialogue"]
        )
    ]

    print("\nFILM CONTRIBUTION")

    if not affected.empty:
        display(
            affected.groupby("movie_title")
            .agg(
                song_mixed_segments=("segment_id", "size"),
                year=("year", "first")
            )
            .sort_values("song_mixed_segments", ascending=False)
        )

    # Retrieve original text safely
    print("\nSAMPLE SONG / MIXED SEGMENTS")

    text_lookup = song_data.set_index("segment_id")["text"]

    for _, row in affected.head(5).iterrows():

        segment_id = row["segment_id"]

        if segment_id not in text_lookup.index:
            print("Segment not found:", segment_id)
            continue

        original_text = text_lookup.loc[segment_id]

        print(f"\nMovie: {row['movie_title']} ({row['year']})")
        print(f"Classification: {row['content_class']}")
        print(f"Segment ID: {segment_id}")
        print("Text:")
        print(str(original_text)[:700])


# Baseline Topic 5
inspect_topic_corrected(
    5,
    "BASELINE",
    baseline_info,
    baseline_terms,
    "topic"
)

# Experiment 2 Topic 23
inspect_topic_corrected(
    23,
    "EXPERIMENT 2",
    experiment2_info,
    experiment2_terms,
    "topic_id"
)

In [ ]:
# ============================================================
# ICMB-130 — PREPARE SONG ROBUSTNESS-TEST DATA
# ============================================================

# Keep all explicitly identified song-related segments
song_flagged_segments = topic_song_analysis[
    topic_song_analysis["content_class"].isin([
        "song_dominated",
        "mixed_song_dialogue"
    ])
].copy()

# Add original text and marker-based lyric percentage
text_lookup = song_data.set_index("segment_id")

song_flagged_segments["original_text"] = (
    song_flagged_segments["segment_id"]
    .map(text_lookup["text"])
)

song_flagged_segments["marked_lyric_pct"] = (
    song_flagged_segments["segment_id"]
    .map(text_lookup["marked_lyric_pct"])
)

# Create two separate filtering lists
strict_exclusion = song_flagged_segments[
    song_flagged_segments["content_class"] == "song_dominated"
].copy()

broad_exclusion = song_flagged_segments.copy()

# Save audit files
strict_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_STRICT_SONG_EXCLUSION_SEGMENTS.csv"
)

broad_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BROAD_SONG_EXCLUSION_SEGMENTS.csv"
)

strict_exclusion.to_csv(strict_path, index=False)
broad_exclusion.to_csv(broad_path, index=False)

print("=" * 75)
print("ICMB-130 — ROBUSTNESS DATA PREPARATION")
print("=" * 75)

print("\nSTRICT FILTER")
print("Song-dominated segments:", len(strict_exclusion))

print("\nBROAD FILTER")
print("Song-dominated + mixed segments:", len(broad_exclusion))

print("\nOriginal corpus segments:", len(song_data))
print("Original corpus remains unchanged.")

print("\nSaved files:")
print(strict_path)
print(broad_path)

assert len(strict_exclusion) == 91
assert len(broad_exclusion) == 328
assert song_flagged_segments["original_text"].notna().all()

print("\nValidation passed.")

In [ ]:
# ============================================================
# ICMB-130 — CREATE FILTERED INPUTS USING SAVED EMBEDDINGS
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Define paths
# ------------------------------------------------------------

embedding_input_path = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_BERTOPIC_EMBEDDING_INPUT.csv"
)

checkpoint_dir = os.path.join(
    OUTPUT_DIR,
    "bertopic_embedding_checkpoints"
)

strict_exclusion_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_STRICT_SONG_EXCLUSION_SEGMENTS.csv"
)

broad_exclusion_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BROAD_SONG_EXCLUSION_SEGMENTS.csv"
)

# ------------------------------------------------------------
# 2. Load original embedding input
# ------------------------------------------------------------

embedding_input = pd.read_csv(embedding_input_path)

required_columns = [
    "movie_record_id",
    "source_chunk_id",
    "part_number"
]

missing_columns = [
    col for col in required_columns
    if col not in embedding_input.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

embedding_input["segment_id"] = (
    embedding_input["movie_record_id"].astype(str) + "_" +
    embedding_input["source_chunk_id"].astype(str) + "_" +
    embedding_input["part_number"].astype(str)
)

assert embedding_input["segment_id"].is_unique

# ------------------------------------------------------------
# 3. Load saved embedding checkpoints
# ------------------------------------------------------------

checkpoint_files = sorted(
    glob.glob(os.path.join(checkpoint_dir, "batch_*.npy"))
)

if not checkpoint_files:
    raise FileNotFoundError("No embedding checkpoints found.")

embedding_batches = [
    np.load(file, allow_pickle=False)
    for file in checkpoint_files
]

original_embeddings = np.concatenate(
    embedding_batches,
    axis=0
)

print("=" * 75)
print("ORIGINAL EMBEDDING ALIGNMENT CHECK")
print("=" * 75)

print("Input rows:", len(embedding_input))
print("Checkpoint batches:", len(checkpoint_files))
print("Combined embedding shape:", original_embeddings.shape)

assert len(embedding_input) == 33800
assert original_embeddings.shape == (33800, 384)

# ------------------------------------------------------------
# 4. Load exclusion lists
# ------------------------------------------------------------

strict_exclusion = pd.read_csv(strict_exclusion_path)
broad_exclusion = pd.read_csv(broad_exclusion_path)

strict_ids = set(strict_exclusion["segment_id"])
broad_ids = set(broad_exclusion["segment_id"])

assert len(strict_ids) == 91
assert len(broad_ids) == 328

# Verify that every excluded segment exists
assert strict_ids.issubset(set(embedding_input["segment_id"]))
assert broad_ids.issubset(set(embedding_input["segment_id"]))

# ------------------------------------------------------------
# 5. Create filtered datasets and embeddings
# ------------------------------------------------------------

def create_filtered_version(name, excluded_ids):

    keep_mask = ~embedding_input["segment_id"].isin(
        excluded_ids
    )

    filtered_input = (
        embedding_input.loc[keep_mask]
        .reset_index(drop=True)
        .copy()
    )

    filtered_embeddings = original_embeddings[
        keep_mask.to_numpy()
    ]

    # Save filtered input
    input_file = os.path.join(
        OUTPUT_DIR,
        f"ICMB_130_{name}_EMBEDDING_INPUT.csv"
    )

    embedding_file = os.path.join(
        OUTPUT_DIR,
        f"ICMB_130_{name}_EMBEDDINGS.npy"
    )

    filtered_input.to_csv(input_file, index=False)
    np.save(embedding_file, filtered_embeddings)

    print(f"\n{name} FILTER")
    print("Excluded segments:", len(excluded_ids))
    print("Remaining segments:", len(filtered_input))
    print("Filtered embedding shape:", filtered_embeddings.shape)
    print("Saved input:", input_file)
    print("Saved embeddings:", embedding_file)

    return filtered_input, filtered_embeddings


strict_input, strict_embeddings = create_filtered_version(
    "STRICT_SONG_FILTER",
    strict_ids
)

broad_input, broad_embeddings = create_filtered_version(
    "BROAD_SONG_FILTER",
    broad_ids
)

# ------------------------------------------------------------
# 6. Final validation
# ------------------------------------------------------------

assert len(strict_input) == 33709
assert len(broad_input) == 33472

assert strict_embeddings.shape == (33709, 384)
assert broad_embeddings.shape == (33472, 384)

assert strict_input["segment_id"].is_unique
assert broad_input["segment_id"].is_unique

print("\n" + "=" * 75)
print("FILTERED EMBEDDING VALIDATION PASSED")
print("=" * 75)

print("Original:", original_embeddings.shape)
print("Strict:", strict_embeddings.shape)
print("Broad:", broad_embeddings.shape)

print("\nOriginal embedding checkpoints remain unchanged.")

In [ ]:
# ============================================================
# ICMB-130 — FILTERED MODELLING INPUT INSPECTION
# ============================================================

print("ORIGINAL INPUT COLUMNS")
print(embedding_input.columns.tolist())

print("\nFIRST TWO ORIGINAL ROWS")
display(embedding_input.head(2))

print("\nTEXT COLUMN CANDIDATES")

for col in embedding_input.columns:
    if embedding_input[col].dtype == "object":
        sample = embedding_input[col].dropna().astype(str).head(2)

        if len(sample):
            print(f"\nColumn: {col}")
            for value in sample:
                print(value[:250])

print("\nFILTERED INPUT ROW COUNTS")
print("Strict:", len(strict_input))
print("Broad:", len(broad_input))

print("\nEMBEDDING SHAPES")
print("Strict:", strict_embeddings.shape)
print("Broad:", broad_embeddings.shape)

In [ ]:
# ============================================================
# ICMB-130 — BERTopic ENVIRONMENT CHECK
# ============================================================

import importlib.util

packages = [
    "bertopic",
    "umap",
    "hdbscan",
    "sentence_transformers",
    "sklearn",
    "numpy",
    "pandas"
]

print("=" * 65)
print("PACKAGE AVAILABILITY CHECK")
print("=" * 65)

for package in packages:
    status = (
        "INSTALLED"
        if importlib.util.find_spec(package)
        else "NOT INSTALLED"
    )
    print(f"{package:<25} {status}")

In [ ]:
import bertopic
from bertopic import BERTopic

print("BERTopic version:", bertopic.__version__)
print("BERTopic imported successfully.")

In [ ]:
# ============================================================
# ICMB-130 — RESTORE STRICT FILTER DATA
# ============================================================

import os
import numpy as np
import pandas as pd
import umap
import hdbscan

from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# Project paths
PROJECT_DIR = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "06_outputs",
    "topic_modelling"
)

# Check Drive availability
assert os.path.exists(OUTPUT_DIR), "Output directory not found."

# Load strict-filter input
strict_input_path = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_STRICT_SONG_FILTER_EMBEDDING_INPUT.csv"
)

strict_embedding_path = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_STRICT_SONG_FILTER_EMBEDDINGS.npy"
)

strict_input = pd.read_csv(strict_input_path)
strict_embeddings = np.load(strict_embedding_path)

# Prepare documents
strict_docs = strict_input["text"].fillna("").astype(str).tolist()

# Validate
assert len(strict_docs) == 33709
assert strict_embeddings.shape == (33709, 384)
assert strict_input["segment_id"].is_unique

print("=" * 70)
print("STRICT FILTER DATA RESTORED")
print("=" * 70)

print("Documents:", len(strict_docs))
print("Embeddings:", strict_embeddings.shape)
print("Unique segments:", strict_input["segment_id"].nunique())
print("BERTopic version:", bertopic.__version__)

print("\nData restoration and validation successful.")

In [ ]:
# ============================================================
# ICMB-130 — STRICT SONG FILTER
# BASELINE BERTopic MODEL
# ============================================================

# 1. UMAP
strict_umap = umap.UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

print("Fitting UMAP on strict-filter embeddings...")

strict_reduced_embeddings = strict_umap.fit_transform(
    strict_embeddings
)

print("UMAP completed:", strict_reduced_embeddings.shape)


# 2. HDBSCAN
strict_hdbscan = hdbscan.HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)


# 3. CountVectorizer
strict_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.80,
    max_features=30000
)


# 4. BERTopic
strict_baseline_model = BERTopic(
    embedding_model=None,
    umap_model=strict_umap,
    hdbscan_model=strict_hdbscan,
    vectorizer_model=strict_vectorizer,
    calculate_probabilities=False,
    verbose=True
)


# 5. Fit model using saved embeddings
print("\nFitting strict-filter baseline BERTopic model...")

strict_baseline_topics, strict_baseline_probs = (
    strict_baseline_model.fit_transform(
        strict_docs,
        embeddings=strict_embeddings
    )
)


# 6. Evaluate results
strict_baseline_topics = np.asarray(strict_baseline_topics)

total_segments = len(strict_baseline_topics)
outlier_count = int(np.sum(strict_baseline_topics == -1))
assigned_count = total_segments - outlier_count

topic_count = len(set(strict_baseline_topics) - {-1})

print("\n" + "=" * 75)
print("STRICT FILTER — BASELINE RESULTS")
print("=" * 75)

print("Total segments:", total_segments)
print("Number of topics:", topic_count)
print("Assigned segments:", assigned_count)
print("Outlier segments:", outlier_count)
print(f"Assigned percentage: {assigned_count / total_segments * 100:.2f}%")
print(f"Outlier percentage: {outlier_count / total_segments * 100:.2f}%")

In [ ]:
# ============================================================
# ICMB-130 — SAVE STRICT-FILTER BASELINE RESULTS
# ============================================================

# 1. Save segment-level topic assignments
strict_baseline_assignments = strict_input.copy()

strict_baseline_assignments["topic"] = strict_baseline_topics

assignments_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_STRICT_FILTER_BASELINE_ASSIGNMENTS.csv"
)

strict_baseline_assignments.to_csv(
    assignments_path,
    index=False
)

# 2. Save topic information
strict_baseline_topic_info = (
    strict_baseline_model.get_topic_info()
)

topic_info_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_STRICT_FILTER_BASELINE_TOPIC_INFO.csv"
)

strict_baseline_topic_info.to_csv(
    topic_info_path,
    index=False
)

# 3. Save topic keywords
strict_terms = []

for topic_id in strict_baseline_model.get_topics():

    if topic_id == -1:
        continue

    for rank, (term, score) in enumerate(
        strict_baseline_model.get_topic(topic_id), start=1
    ):
        strict_terms.append({
            "topic": topic_id,
            "term_rank": rank,
            "term": term,
            "c_tf_idf_score": score
        })

strict_baseline_topic_terms = pd.DataFrame(strict_terms)

terms_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_STRICT_FILTER_BASELINE_TOPIC_TERMS.csv"
)

strict_baseline_topic_terms.to_csv(
    terms_path,
    index=False
)

print("=" * 75)
print("STRICT-FILTER BASELINE RESULTS SAVED")
print("=" * 75)

print("Assignments:", strict_baseline_assignments.shape)
print("Topic information:", strict_baseline_topic_info.shape)
print("Topic terms:", strict_baseline_topic_terms.shape)

print("\nFiles saved:")
print(assignments_path)
print(topic_info_path)
print(terms_path)

In [ ]:
# ============================================================
# ICMB-130 — RESTORE PROJECT WORKSPACE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import numpy as np
import pandas as pd

PROJECT_DIR = (
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics"
)

NOTEBOOK_DIR = os.path.join(
    PROJECT_DIR, "02_notebooks"
)

OUTPUT_DIR = os.path.join(
    PROJECT_DIR, "06_outputs", "topic_modelling"
)

CHECKPOINT_DIR = os.path.join(
    OUTPUT_DIR, "bertopic_embedding_checkpoints"
)

print("=" * 75)
print("ICMB-130 — PROJECT WORKSPACE RESTORATION")
print("=" * 75)

print("Project directory exists:", os.path.exists(PROJECT_DIR))
print("Notebook directory exists:", os.path.exists(NOTEBOOK_DIR))
print("Output directory exists:", os.path.exists(OUTPUT_DIR))
print("Embedding checkpoints exist:", os.path.exists(CHECKPOINT_DIR))

assert os.path.exists(OUTPUT_DIR), "Topic modelling output directory missing."

print("\nProject workspace restored.")

In [ ]:
# ============================================================
# ICMB-130 — RESTORE MODELLING ENVIRONMENT AND SAVED DATA
# ============================================================

# 1. Import required libraries
import os
import glob
import numpy as np
import pandas as pd
import umap
import hdbscan
import bertopic

from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

print("BERTopic version:", bertopic.__version__)
print("Required packages imported successfully.")

# 2. Define file paths
files = {
    "strict_input": "ICMB_130_STRICT_SONG_FILTER_EMBEDDING_INPUT.csv",
    "strict_embeddings": "ICMB_130_STRICT_SONG_FILTER_EMBEDDINGS.npy",
    "broad_input": "ICMB_130_BROAD_SONG_FILTER_EMBEDDING_INPUT.csv",
    "broad_embeddings": "ICMB_130_BROAD_SONG_FILTER_EMBEDDINGS.npy",

    "strict_baseline_assignments": "BERTOPIC_STRICT_FILTER_BASELINE_ASSIGNMENTS.csv",
    "strict_baseline_info": "BERTOPIC_STRICT_FILTER_BASELINE_TOPIC_INFO.csv",
    "strict_baseline_terms": "BERTOPIC_STRICT_FILTER_BASELINE_TOPIC_TERMS.csv",

    "original_baseline_assignments": "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv",
    "original_exp2_assignments": "BERTOPIC_V2_CHUNK_TOPIC_ASSIGNMENTS.csv",
    "original_baseline_info": "BERTOPIC_TOPIC_INFO.csv",
    "original_exp2_info": "BERTOPIC_V2_TOPIC_INFO.csv"
}

paths = {
    key: os.path.join(OUTPUT_DIR, filename)
    for key, filename in files.items()
}

# 3. Verify all required files exist
missing_files = [
    key for key, path in paths.items()
    if not os.path.isfile(path)
]

print("\nFILE AVAILABILITY CHECK")

for key, path in paths.items():
    print(f"{key:<35} {'FOUND' if os.path.isfile(path) else 'MISSING'}")

if missing_files:
    raise FileNotFoundError(
        f"Required project files missing: {missing_files}"
    )

# 4. Load filtered datasets and embeddings
strict_input = pd.read_csv(paths["strict_input"])
strict_embeddings = np.load(paths["strict_embeddings"])

broad_input = pd.read_csv(paths["broad_input"])
broad_embeddings = np.load(paths["broad_embeddings"])

# 5. Load completed strict-baseline results
strict_baseline_assignments = pd.read_csv(
    paths["strict_baseline_assignments"]
)

strict_baseline_topic_info = pd.read_csv(
    paths["strict_baseline_info"]
)

strict_baseline_topic_terms = pd.read_csv(
    paths["strict_baseline_terms"]
)

# 6. Load original model assignments for comparison
baseline_assignments = pd.read_csv(
    paths["original_baseline_assignments"]
)

experiment2_assignments = pd.read_csv(
    paths["original_exp2_assignments"]
)

baseline_topic_info = pd.read_csv(
    paths["original_baseline_info"]
)

experiment2_topic_info = pd.read_csv(
    paths["original_exp2_info"]
)

# 7. Reconstruct document lists
strict_docs = (
    strict_input["text"].fillna("").astype(str).tolist()
)

broad_docs = (
    broad_input["text"].fillna("").astype(str).tolist()
)

# 8. Validate dimensions and alignment
assert len(strict_input) == 33709
assert strict_embeddings.shape == (33709, 384)

assert len(broad_input) == 33472
assert broad_embeddings.shape == (33472, 384)

assert len(strict_baseline_assignments) == 33709
assert len(baseline_assignments) == 33800
assert len(experiment2_assignments) == 33800

assert strict_input["segment_id"].is_unique
assert broad_input["segment_id"].is_unique

assert (
    strict_input["segment_id"].tolist()
    == strict_baseline_assignments["segment_id"].tolist()
)

# 9. Display restoration summary
print("\n" + "=" * 75)
print("ICMB-130 — DATA RESTORATION SUMMARY")
print("=" * 75)

print("Strict-filter documents:", len(strict_docs))
print("Strict embeddings:", strict_embeddings.shape)

print("\nBroad-filter documents:", len(broad_docs))
print("Broad embeddings:", broad_embeddings.shape)

print("\nStrict baseline assignments:",
      strict_baseline_assignments.shape)
print("Strict baseline topics:",
      len(set(strict_baseline_assignments["topic"]) - {-1}))

print("\nOriginal baseline assignments:",
      baseline_assignments.shape)
print("Original Experiment 2 assignments:",
      experiment2_assignments.shape)

print("\nAll required files loaded and validation passed.")

In [ ]:
%pip install bertopic==0.17.4

In [ ]:
# ============================================================
# ICMB-130 — STRICT-FILTER EXPERIMENT 2
# HDBSCAN: min_cluster_size=30, min_samples=5
# ============================================================

import os
import numpy as np
import pandas as pd
import umap
import hdbscan

from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

print("=" * 70)
print("STRICT-FILTER EXPERIMENT 2 — STARTING")
print("=" * 70)

# 1. Configure UMAP
strict_exp2_umap = umap.UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# 2. Configure HDBSCAN — Experiment 2 settings
strict_exp2_hdbscan = hdbscan.HDBSCAN(
    min_cluster_size=30,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

# 3. Configure CountVectorizer
strict_exp2_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.80,
    max_features=30000
)

# 4. Build BERTopic model
strict_exp2_model = BERTopic(
    embedding_model=None,
    umap_model=strict_exp2_umap,
    hdbscan_model=strict_exp2_hdbscan,
    vectorizer_model=strict_exp2_vectorizer,
    calculate_probabilities=False,
    verbose=True
)

# 5. Fit model using existing embeddings
print("\nFitting Strict-filter Experiment 2...")

strict_exp2_topics, strict_exp2_probabilities = (
    strict_exp2_model.fit_transform(
        strict_docs,
        embeddings=strict_embeddings
    )
)

# 6. Generate topic assignments
strict_exp2_assignments = strict_input.copy()

strict_exp2_assignments["topic"] = strict_exp2_topics

# 7. Extract topic information
strict_exp2_topic_info = strict_exp2_model.get_topic_info()

# 8. Extract top 10 terms for each non-outlier topic
strict_exp2_term_records = []

for topic_id in strict_exp2_model.get_topics():

    if topic_id == -1:
        continue

    topic_terms = strict_exp2_model.get_topic(topic_id)

    if topic_terms is None:
        continue

    for rank, (term, score) in enumerate(topic_terms[:10], start=1):

        strict_exp2_term_records.append({
            "topic_id": topic_id,
            "term_rank": rank,
            "term": term,
            "term_score": score
        })

strict_exp2_topic_terms = pd.DataFrame(
    strict_exp2_term_records
)

# 9. Save outputs
strict_exp2_assignments_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_STRICT_FILTER_EXPERIMENT2_ASSIGNMENTS.csv"
)

strict_exp2_info_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_STRICT_FILTER_EXPERIMENT2_TOPIC_INFO.csv"
)

strict_exp2_terms_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_STRICT_FILTER_EXPERIMENT2_TOPIC_TERMS.csv"
)

strict_exp2_assignments.to_csv(
    strict_exp2_assignments_path,
    index=False
)

strict_exp2_topic_info.to_csv(
    strict_exp2_info_path,
    index=False
)

strict_exp2_topic_terms.to_csv(
    strict_exp2_terms_path,
    index=False
)

# 10. Calculate model statistics
total_segments = len(strict_exp2_topics)

outlier_count = sum(
    topic == -1 for topic in strict_exp2_topics
)

assigned_count = total_segments - outlier_count

non_outlier_topics = len(
    set(strict_exp2_topics) - {-1}
)

outlier_percentage = (
    outlier_count / total_segments
) * 100

assigned_percentage = (
    assigned_count / total_segments
) * 100

# 11. Display results
print("\n" + "=" * 70)
print("STRICT-FILTER EXPERIMENT 2 — RESULTS")
print("=" * 70)

print("Total segments:", total_segments)
print("Number of topics:", non_outlier_topics)
print("Assigned segments:", assigned_count)
print("Outlier segments:", outlier_count)

print(f"Assigned percentage: {assigned_percentage:.2f}%")
print(f"Outlier percentage: {outlier_percentage:.2f}%")

print("\nTopic information shape:",
      strict_exp2_topic_info.shape)

print("Topic terms shape:",
      strict_exp2_topic_terms.shape)

print("\nSaved files:")
print(strict_exp2_assignments_path)
print(strict_exp2_info_path)
print(strict_exp2_terms_path)

print("\nStrict-filter Experiment 2 completed.")

In [ ]:
# ============================================================
# ICMB-130 — BROAD-FILTER BASELINE
# HDBSCAN: min_cluster_size=50, min_samples=10
# ============================================================

import os
import numpy as np
import pandas as pd
import umap
import hdbscan

from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

print("=" * 70)
print("BROAD-FILTER BASELINE — STARTING")
print("=" * 70)

# 1. Configure UMAP
broad_baseline_umap = umap.UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# 2. Configure HDBSCAN — Original Baseline settings
broad_baseline_hdbscan = hdbscan.HDBSCAN(
    min_cluster_size=50,
    min_samples=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

# 3. Configure CountVectorizer
broad_baseline_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.80,
    max_features=30000
)

# 4. Build BERTopic model
broad_baseline_model = BERTopic(
    embedding_model=None,
    umap_model=broad_baseline_umap,
    hdbscan_model=broad_baseline_hdbscan,
    vectorizer_model=broad_baseline_vectorizer,
    calculate_probabilities=False,
    verbose=True
)

# 5. Fit using existing broad-filter embeddings
print("\nFitting Broad-filter Baseline...")

broad_baseline_topics, broad_baseline_probabilities = (
    broad_baseline_model.fit_transform(
        broad_docs,
        embeddings=broad_embeddings
    )
)

# 6. Generate assignments
broad_baseline_assignments = broad_input.copy()
broad_baseline_assignments["topic"] = broad_baseline_topics

# 7. Extract topic information
broad_baseline_topic_info = (
    broad_baseline_model.get_topic_info()
)

# 8. Extract top 10 terms per topic
broad_baseline_term_records = []

for topic_id in broad_baseline_model.get_topics():

    if topic_id == -1:
        continue

    topic_terms = broad_baseline_model.get_topic(topic_id)

    if topic_terms is None:
        continue

    for rank, (term, score) in enumerate(
        topic_terms[:10], start=1
    ):
        broad_baseline_term_records.append({
            "topic": topic_id,
            "term_rank": rank,
            "term": term,
            "c_tf_idf_score": score
        })

broad_baseline_topic_terms = pd.DataFrame(
    broad_baseline_term_records
)

# 9. Save outputs
broad_baseline_assignments_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BROAD_FILTER_BASELINE_ASSIGNMENTS.csv"
)

broad_baseline_info_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BROAD_FILTER_BASELINE_TOPIC_INFO.csv"
)

broad_baseline_terms_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BROAD_FILTER_BASELINE_TOPIC_TERMS.csv"
)

broad_baseline_assignments.to_csv(
    broad_baseline_assignments_path,
    index=False
)

broad_baseline_topic_info.to_csv(
    broad_baseline_info_path,
    index=False
)

broad_baseline_topic_terms.to_csv(
    broad_baseline_terms_path,
    index=False
)

# 10. Calculate statistics
total_segments = len(broad_baseline_topics)

outlier_count = sum(
    topic == -1 for topic in broad_baseline_topics
)

assigned_count = total_segments - outlier_count

non_outlier_topics = len(
    set(broad_baseline_topics) - {-1}
)

outlier_percentage = (
    outlier_count / total_segments
) * 100

assigned_percentage = (
    assigned_count / total_segments
) * 100

# 11. Display results
print("\n" + "=" * 70)
print("BROAD-FILTER BASELINE — RESULTS")
print("=" * 70)

print("Total segments:", total_segments)
print("Number of topics:", non_outlier_topics)
print("Assigned segments:", assigned_count)
print("Outlier segments:", outlier_count)

print(f"Assigned percentage: {assigned_percentage:.2f}%")
print(f"Outlier percentage: {outlier_percentage:.2f}%")

print("\nTopic information shape:",
      broad_baseline_topic_info.shape)

print("Topic terms shape:",
      broad_baseline_topic_terms.shape)

print("\nSaved files:")
print(broad_baseline_assignments_path)
print(broad_baseline_info_path)
print(broad_baseline_terms_path)

print("\nBroad-filter Baseline completed.")

In [ ]:
# ============================================================
# ICMB-130 — BROAD-FILTER EXPERIMENT 2
# HDBSCAN: min_cluster_size=30, min_samples=5
# ============================================================

import os
import numpy as np
import pandas as pd
import umap
import hdbscan

from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

print("=" * 70)
print("BROAD-FILTER EXPERIMENT 2 — STARTING")
print("=" * 70)

# 1. Configure UMAP
broad_exp2_umap = umap.UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# 2. Configure HDBSCAN — Experiment 2 settings
broad_exp2_hdbscan = hdbscan.HDBSCAN(
    min_cluster_size=30,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

# 3. Configure CountVectorizer
broad_exp2_vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.80,
    max_features=30000
)

# 4. Build BERTopic model
broad_exp2_model = BERTopic(
    embedding_model=None,
    umap_model=broad_exp2_umap,
    hdbscan_model=broad_exp2_hdbscan,
    vectorizer_model=broad_exp2_vectorizer,
    calculate_probabilities=False,
    verbose=True
)

# 5. Fit using existing embeddings
print("\nFitting Broad-filter Experiment 2...")

broad_exp2_topics, broad_exp2_probabilities = (
    broad_exp2_model.fit_transform(
        broad_docs,
        embeddings=broad_embeddings
    )
)

# 6. Generate assignments
broad_exp2_assignments = broad_input.copy()
broad_exp2_assignments["topic"] = broad_exp2_topics

# 7. Extract topic information
broad_exp2_topic_info = broad_exp2_model.get_topic_info()

# 8. Extract top 10 terms per topic
broad_exp2_term_records = []

for topic_id in broad_exp2_model.get_topics():

    if topic_id == -1:
        continue

    topic_terms = broad_exp2_model.get_topic(topic_id)

    if topic_terms is None:
        continue

    for rank, (term, score) in enumerate(
        topic_terms[:10], start=1
    ):
        broad_exp2_term_records.append({
            "topic_id": topic_id,
            "term_rank": rank,
            "term": term,
            "term_score": score
        })

broad_exp2_topic_terms = pd.DataFrame(
    broad_exp2_term_records
)

# 9. Save outputs
broad_exp2_assignments_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BROAD_FILTER_EXPERIMENT2_ASSIGNMENTS.csv"
)

broad_exp2_info_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BROAD_FILTER_EXPERIMENT2_TOPIC_INFO.csv"
)

broad_exp2_terms_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BROAD_FILTER_EXPERIMENT2_TOPIC_TERMS.csv"
)

broad_exp2_assignments.to_csv(
    broad_exp2_assignments_path,
    index=False
)

broad_exp2_topic_info.to_csv(
    broad_exp2_info_path,
    index=False
)

broad_exp2_topic_terms.to_csv(
    broad_exp2_terms_path,
    index=False
)

# 10. Calculate statistics
total_segments = len(broad_exp2_topics)

outlier_count = sum(
    topic == -1 for topic in broad_exp2_topics
)

assigned_count = total_segments - outlier_count

non_outlier_topics = len(
    set(broad_exp2_topics) - {-1}
)

outlier_percentage = (
    outlier_count / total_segments
) * 100

assigned_percentage = (
    assigned_count / total_segments
) * 100

# 11. Display results
print("\n" + "=" * 70)
print("BROAD-FILTER EXPERIMENT 2 — RESULTS")
print("=" * 70)

print("Total segments:", total_segments)
print("Number of topics:", non_outlier_topics)
print("Assigned segments:", assigned_count)
print("Outlier segments:", outlier_count)

print(f"Assigned percentage: {assigned_percentage:.2f}%")
print(f"Outlier percentage: {outlier_percentage:.2f}%")

print("\nTopic information shape:",
      broad_exp2_topic_info.shape)

print("Topic terms shape:",
      broad_exp2_topic_terms.shape)

print("\nSaved files:")
print(broad_exp2_assignments_path)
print(broad_exp2_info_path)
print(broad_exp2_terms_path)

print("\nBroad-filter Experiment 2 completed.")

In [ ]:
# ============================================================
# ICMB-130 — CONSOLIDATED SIX-MODEL COMPARISON
# ============================================================

import os
import pandas as pd
import numpy as np

model_files = {
    "Original Baseline":
        "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv",

    "Original Experiment 2":
        "BERTOPIC_V2_CHUNK_TOPIC_ASSIGNMENTS.csv",

    "Strict-filter Baseline":
        "BERTOPIC_STRICT_FILTER_BASELINE_ASSIGNMENTS.csv",

    "Strict-filter Experiment 2":
        "BERTOPIC_STRICT_FILTER_EXPERIMENT2_ASSIGNMENTS.csv",

    "Broad-filter Baseline":
        "BERTOPIC_BROAD_FILTER_BASELINE_ASSIGNMENTS.csv",

    "Broad-filter Experiment 2":
        "BERTOPIC_BROAD_FILTER_EXPERIMENT2_ASSIGNMENTS.csv"
}

comparison_records = []

for model_name, filename in model_files.items():

    file_path = os.path.join(OUTPUT_DIR, filename)

    if not os.path.isfile(file_path):
        raise FileNotFoundError(file_path)

    df = pd.read_csv(file_path)

    if "topic" not in df.columns:
        raise ValueError(
            f"'topic' column missing in {filename}"
        )

    total = len(df)

    outliers = int((df["topic"] == -1).sum())

    assigned = total - outliers

    topic_count = int(
        df.loc[df["topic"] != -1, "topic"].nunique()
    )

    comparison_records.append({
        "model": model_name,
        "total_segments": total,
        "number_of_topics": topic_count,
        "assigned_segments": assigned,
        "outlier_segments": outliers,
        "assigned_percentage": round(
            assigned / total * 100, 2
        ),
        "outlier_percentage": round(
            outliers / total * 100, 2
        )
    })

# Create comparison dataframe
model_comparison = pd.DataFrame(comparison_records)

# Save consolidated comparison
comparison_path = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_SIX_MODEL_SONG_ROBUSTNESS_COMPARISON.csv"
)

model_comparison.to_csv(
    comparison_path,
    index=False
)

# Display results
print("=" * 85)
print("ICMB-130 — SIX-MODEL SONG ROBUSTNESS COMPARISON")
print("=" * 85)

display(model_comparison)

print("\nVerification:")
print("Models compared:", len(model_comparison))
print("Total original segments:",
      model_comparison.iloc[0]["total_segments"])

print("\nSaved file:")
print(comparison_path)

print("\nSix-model comparison completed.")

In [ ]:
# ============================================================
# ICMB-130 — TOPIC CONTINUITY AFTER SONG FILTERING
# Compare original models with their filtered counterparts
# ============================================================

import os
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Define topic-term files
comparison_pairs = {
    "Baseline — Strict Filter": (
        "BERTOPIC_TOPIC_TERMS.csv",
        "BERTOPIC_STRICT_FILTER_BASELINE_TOPIC_TERMS.csv"
    ),

    "Baseline — Broad Filter": (
        "BERTOPIC_TOPIC_TERMS.csv",
        "BERTOPIC_BROAD_FILTER_BASELINE_TOPIC_TERMS.csv"
    ),

    "Experiment 2 — Strict Filter": (
        "BERTOPIC_V2_TOPIC_TERMS.csv",
        "BERTOPIC_STRICT_FILTER_EXPERIMENT2_TOPIC_TERMS.csv"
    ),

    "Experiment 2 — Broad Filter": (
        "BERTOPIC_V2_TOPIC_TERMS.csv",
        "BERTOPIC_BROAD_FILTER_EXPERIMENT2_TOPIC_TERMS.csv"
    )
}

def load_topic_keywords(filename):

    path = os.path.join(OUTPUT_DIR, filename)
    df = pd.read_csv(path)

    # Identify topic ID column across different saved schemas
    possible_columns = ["topic", "topic_id", "Topic"]

    topic_column = next(
        (col for col in possible_columns if col in df.columns),
        None
    )

    if topic_column is None:
        raise ValueError(
            f"No topic ID column found in {filename}"
        )

    # Exclude outlier topic if present
    df = df[df[topic_column] != -1].copy()

    df["term"] = df["term"].fillna("").astype(str)

    # Retain top 10 terms per topic
    df = df.sort_values(
        [topic_column, "term_rank"]
    )

    keyword_data = {}

    for topic_id, group in df.groupby(topic_column):

        terms = group["term"].head(10).tolist()

        keyword_data[int(topic_id)] = " ".join(terms)

    return keyword_data


# 2. Compare original and filtered topic keywords
all_match_records = []

for comparison_name, (original_file, filtered_file) in (
    comparison_pairs.items()
):

    original_topics = load_topic_keywords(original_file)
    filtered_topics = load_topic_keywords(filtered_file)

    original_ids = list(original_topics.keys())
    filtered_ids = list(filtered_topics.keys())

    original_documents = [
        original_topics[topic_id]
        for topic_id in original_ids
    ]

    filtered_documents = [
        filtered_topics[topic_id]
        for topic_id in filtered_ids
    ]

    # Fit TF-IDF jointly so both models share one vocabulary
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b\w+\b"
    )

    combined_documents = (
        original_documents + filtered_documents
    )

    tfidf_matrix = vectorizer.fit_transform(
        combined_documents
    )

    original_matrix = tfidf_matrix[
        :len(original_documents)
    ]

    filtered_matrix = tfidf_matrix[
        len(original_documents):
    ]

    similarity_matrix = cosine_similarity(
        original_matrix,
        filtered_matrix
    )

    # Find top 3 filtered matches for each original topic
    for i, original_id in enumerate(original_ids):

        similarities = similarity_matrix[i]

        top_indices = np.argsort(
            similarities
        )[::-1][:3]

        for rank, j in enumerate(top_indices, start=1):

            filtered_id = filtered_ids[j]

            all_match_records.append({

                "comparison": comparison_name,

                "original_topic": original_id,

                "original_keywords":
                    original_topics[original_id],

                "filtered_topic": filtered_id,

                "filtered_keywords":
                    filtered_topics[filtered_id],

                "similarity": round(
                    float(similarities[j]), 4
                ),

                "match_rank": rank
            })


# 3. Create consolidated results
topic_continuity = pd.DataFrame(
    all_match_records
)

# 4. Save results
continuity_path = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_SONG_FILTER_TOPIC_CONTINUITY_MATCHES.csv"
)

topic_continuity.to_csv(
    continuity_path,
    index=False
)

# 5. Display strongest matches
print("=" * 85)
print("ICMB-130 — TOPIC CONTINUITY AFTER SONG FILTERING")
print("=" * 85)

print("\nTotal topic matches generated:",
      len(topic_continuity))

for comparison_name in comparison_pairs:

    subset = topic_continuity[
        (topic_continuity["comparison"] == comparison_name)
        & (topic_continuity["match_rank"] == 1)
    ]

    print("\n" + "-" * 75)
    print(comparison_name)

    print("Original topics:", len(subset))

    print("Mean strongest-match similarity:",
          round(subset["similarity"].mean(), 4))

    print("\nTop 5 strongest matches:")

    display(
        subset.sort_values(
            "similarity",
            ascending=False
        ).head(5)[[
            "original_topic",
            "original_keywords",
            "filtered_topic",
            "filtered_keywords",
            "similarity"
        ]]
    )

print("\nSaved file:")
print(continuity_path)

print("\nTopic continuity comparison completed.")

In [ ]:
# ============================================================
# ICMB-130 — TOPIC CONTINUITY STRENGTH AND COLLISION ANALYSIS
# ============================================================

import os
import pandas as pd
import numpy as np

input_path = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_SONG_FILTER_TOPIC_CONTINUITY_MATCHES.csv"
)

matches = pd.read_csv(input_path)

# Use only the strongest match for each original topic
strongest_matches = matches[
    matches["match_rank"] == 1
].copy()

# 1. Classify similarity strength
def classify_similarity(score):

    if score >= 0.50:
        return "Strong (>=0.50)"
    elif score >= 0.30:
        return "Moderate (0.30–0.49)"
    elif score >= 0.15:
        return "Weak (0.15–0.29)"
    else:
        return "Very weak (<0.15)"


strongest_matches["similarity_band"] = (
    strongest_matches["similarity"].apply(
        classify_similarity
    )
)

# 2. Summarize each comparison
summary_records = []

for comparison, group in strongest_matches.groupby(
    "comparison"
):

    match_counts = group["similarity_band"].value_counts()

    # Count how many filtered topics are matched
    # by more than one original topic
    filtered_topic_counts = (
        group["filtered_topic"].value_counts()
    )

    collision_count = int(
        (filtered_topic_counts > 1).sum()
    )

    summary_records.append({

        "comparison": comparison,

        "original_topics": len(group),

        "mean_similarity": round(
            group["similarity"].mean(), 4
        ),

        "median_similarity": round(
            group["similarity"].median(), 4
        ),

        "strong_matches": int(
            match_counts.get("Strong (>=0.50)", 0)
        ),

        "moderate_matches": int(
            match_counts.get("Moderate (0.30–0.49)", 0)
        ),

        "weak_matches": int(
            match_counts.get("Weak (0.15–0.29)", 0)
        ),

        "very_weak_matches": int(
            match_counts.get("Very weak (<0.15)", 0)
        ),

        "unique_filtered_topics_matched":
            int(group["filtered_topic"].nunique()),

        "filtered_topics_with_multiple_matches":
            collision_count,

        "original_topics_with_match":
            int(group["similarity"].notna().sum())
    })


continuity_summary = pd.DataFrame(summary_records)

# 3. Save summary
summary_path = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_SONG_FILTER_CONTINUITY_STRENGTH_SUMMARY.csv"
)

continuity_summary.to_csv(
    summary_path,
    index=False
)

# 4. Display results
print("=" * 85)
print("TOPIC CONTINUITY — STRENGTH AND COLLISION ANALYSIS")
print("=" * 85)

display(continuity_summary)

# 5. Inspect original topics with weakest matches
print("\n" + "=" * 85)
print("10 WEAKEST STRONGEST-MATCH RESULTS")
print("=" * 85)

display(
    strongest_matches.sort_values(
        "similarity",
        ascending=True
    ).groupby(
        "comparison",
        group_keys=False
    ).head(10)[[
        "comparison",
        "original_topic",
        "filtered_topic",
        "similarity",
        "original_keywords",
        "filtered_keywords"
    ]]
)

print("\nSaved summary:")
print(summary_path)

print("\nContinuity strength analysis completed.")

In [ ]:
# ============================================================
# ICMB-130 — REPRESENTATIVE DOCUMENT VALIDATION
# Stable and changed topics after song filtering
# ============================================================

import os
import ast
import pandas as pd
import numpy as np

# 1. Define topic information files
topic_info_files = {
    "Original Baseline":
        "BERTOPIC_TOPIC_INFO.csv",

    "Original Experiment 2":
        "BERTOPIC_V2_TOPIC_INFO.csv",

    "Strict Baseline":
        "BERTOPIC_STRICT_FILTER_BASELINE_TOPIC_INFO.csv",

    "Strict Experiment 2":
        "BERTOPIC_STRICT_FILTER_EXPERIMENT2_TOPIC_INFO.csv",

    "Broad Baseline":
        "BERTOPIC_BROAD_FILTER_BASELINE_TOPIC_INFO.csv",

    "Broad Experiment 2":
        "BERTOPIC_BROAD_FILTER_EXPERIMENT2_TOPIC_INFO.csv"
}

# 2. Define comparison-to-model mapping
comparison_models = {
    "Baseline — Strict Filter": (
        "Original Baseline",
        "Strict Baseline"
    ),

    "Baseline — Broad Filter": (
        "Original Baseline",
        "Broad Baseline"
    ),

    "Experiment 2 — Strict Filter": (
        "Original Experiment 2",
        "Strict Experiment 2"
    ),

    "Experiment 2 — Broad Filter": (
        "Original Experiment 2",
        "Broad Experiment 2"
    )
}

# 3. Load topic information
topic_info_data = {}

for model_name, filename in topic_info_files.items():

    path = os.path.join(OUTPUT_DIR, filename)

    df = pd.read_csv(path)

    topic_col = next(
        (col for col in ["Topic", "topic", "topic_id"]
         if col in df.columns),
        None
    )

    if topic_col is None:
        raise ValueError(
            f"Topic column missing in {filename}"
        )

    df = df.rename(columns={topic_col: "topic_id"})

    df["topic_id"] = df["topic_id"].astype(int)

    topic_info_data[model_name] = df.set_index("topic_id")


# 4. Helper to extract representative documents
def extract_representative_docs(value):

    if isinstance(value, list):
        docs = value

    elif pd.isna(value):
        return ""

    else:
        try:
            docs = ast.literal_eval(str(value))

            if not isinstance(docs, list):
                docs = [str(value)]

        except Exception:
            docs = [str(value)]

    return " || ".join(
        str(doc).replace("\n", " ").strip()
        for doc in docs[:2]
    )


# 5. Load strongest-match records
matches_path = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_SONG_FILTER_TOPIC_CONTINUITY_MATCHES.csv"
)

matches = pd.read_csv(matches_path)

strongest_matches = matches[
    matches["match_rank"] == 1
].copy()


# 6. Select strongest and weakest topic matches
inspection_records = []

for comparison, (original_model, filtered_model) in (
    comparison_models.items()
):

    subset = strongest_matches[
        strongest_matches["comparison"] == comparison
    ].copy()

    strongest = subset.nlargest(2, "similarity")
    weakest = subset.nsmallest(2, "similarity")

    selected = pd.concat([
        strongest.assign(match_type="Strongest"),
        weakest.assign(match_type="Weakest")
    ])

    original_info = topic_info_data[original_model]
    filtered_info = topic_info_data[filtered_model]

    for _, row in selected.iterrows():

        original_id = int(row["original_topic"])
        filtered_id = int(row["filtered_topic"])

        original_row = original_info.loc[original_id]
        filtered_row = filtered_info.loc[filtered_id]

        inspection_records.append({

            "comparison": comparison,

            "match_type": row["match_type"],

            "original_topic": original_id,

            "original_topic_name":
                original_row.get("Name", ""),

            "original_topic_size":
                original_row.get("Count", np.nan),

            "original_representative_docs":
                extract_representative_docs(
                    original_row.get(
                        "Representative_Docs", ""
                    )
                ),

            "filtered_topic": filtered_id,

            "filtered_topic_name":
                filtered_row.get("Name", ""),

            "filtered_topic_size":
                filtered_row.get("Count", np.nan),

            "filtered_representative_docs":
                extract_representative_docs(
                    filtered_row.get(
                        "Representative_Docs", ""
                    )
                ),

            "keyword_similarity":
                row["similarity"]
        })


# 7. Create inspection dataframe
representative_validation = pd.DataFrame(
    inspection_records
)

# 8. Save validation output
validation_path = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_SONG_FILTER_REPRESENTATIVE_DOC_VALIDATION.csv"
)

representative_validation.to_csv(
    validation_path,
    index=False
)

# 9. Display selected records
print("=" * 85)
print("REPRESENTATIVE DOCUMENT VALIDATION")
print("=" * 85)

for comparison in comparison_models:

    print("\n" + "=" * 85)
    print(comparison)
    print("=" * 85)

    subset = representative_validation[
        representative_validation["comparison"] == comparison
    ]

    for _, row in subset.iterrows():

        print("\nMatch type:", row["match_type"])
        print("Keyword similarity:", row["keyword_similarity"])

        print(
            f"Original topic {row['original_topic']}: "
            f"{row['original_topic_name']}"
        )

        print(
            "Original representative dialogue:",
            str(row["original_representative_docs"])[:700]
        )

        print(
            f"\nFiltered topic {row['filtered_topic']}: "
            f"{row['filtered_topic_name']}"
        )

        print(
            "Filtered representative dialogue:",
            str(row["filtered_representative_docs"])[:700]
        )

        print("-" * 75)


print("\nSaved validation file:")
print(validation_path)

print("\nRepresentative document validation completed.")

In [ ]:
# ============================================================
# ICMB-130 — CROSS-FILM THEME ANALYSIS
# STEP 1: VERIFY ASSIGNMENT DATA STRUCTURE
# ============================================================

import os
import pandas as pd
import numpy as np

# File paths
original_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

broad_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BROAD_FILTER_BASELINE_ASSIGNMENTS.csv"
)

# Verify files
for path in [original_path, broad_path]:

    if not os.path.isfile(path):
        raise FileNotFoundError(path)

# Load assignments
original_df = pd.read_csv(original_path)
broad_df = pd.read_csv(broad_path)

# Display dataset structure
print("=" * 80)
print("ICMB-130 — ASSIGNMENT DATA STRUCTURE VERIFICATION")
print("=" * 80)

datasets = {
    "Original Baseline": original_df,
    "Broad-filter Baseline": broad_df
}

for name, df in datasets.items():

    print("\n" + "-" * 75)
    print(name)
    print("-" * 75)

    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print("\nFirst 3 rows:")
    display(df.head(3))

    print("\nUnique films:")
    if "movie_record_id" in df.columns:
        print(df["movie_record_id"].nunique())

    print("\nAvailable year column:")
    print(
        [
            col for col in df.columns
            if "year" in col.lower()
        ]
    )

    print("\nAvailable movie/title columns:")
    print(
        [
            col for col in df.columns
            if any(
                word in col.lower()
                for word in ["movie", "film", "title"]
            )
        ]
    )

print("\nAssignment structure inspection completed.")

In [ ]:
# ============================================================
# ICMB-130 — ORIGINAL BASELINE
# CROSS-FILM TOPIC COVERAGE ANALYSIS
# ============================================================

import os
import pandas as pd
import numpy as np

# 1. Load original Baseline topic assignments
baseline_df = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
    )
)

# 2. Load topic information and topic keywords
baseline_info = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "BERTOPIC_TOPIC_INFO.csv"
    )
)

baseline_terms = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "BERTOPIC_TOPIC_TERMS.csv"
    )
)

# 3. Exclude outliers
assigned_df = baseline_df[
    baseline_df["topic"] != -1
].copy()

print("=" * 80)
print("ICMB-130 — ORIGINAL BASELINE CROSS-FILM TOPIC ANALYSIS")
print("=" * 80)

print("Total assigned segments:", len(assigned_df))
print("Distinct films:", assigned_df["movie_record_id"].nunique())
print("Distinct years:", assigned_df["year"].nunique())
print("Non-outlier topics:", assigned_df["topic"].nunique())

# 4. Calculate topic-level coverage
topic_records = []

for topic_id, group in assigned_df.groupby("topic"):

    total_segments = len(group)

    movie_counts = (
        group.groupby(
            ["movie_record_id", "movie_title", "year"]
        )
        .size()
        .reset_index(name="segment_count")
        .sort_values(
            "segment_count",
            ascending=False
        )
    )

    unique_movies = movie_counts["movie_record_id"].nunique()
    unique_years = group["year"].nunique()

    largest_movie_count = movie_counts["segment_count"].max()

    largest_movie_share = (
        largest_movie_count / total_segments
    ) * 100

    top_movies = movie_counts.head(5)

    top_movie_details = " | ".join(
        f"{row.movie_title} ({row.year}): "
        f"{row.segment_count} segments"
        for row in top_movies.itertuples()
    )

    topic_records.append({

        "topic_id": int(topic_id),

        "total_segments": total_segments,

        "unique_movies": unique_movies,

        "unique_years": unique_years,

        "largest_movie": movie_counts.iloc[0]["movie_title"],

        "largest_movie_year": int(
            movie_counts.iloc[0]["year"]
        ),

        "largest_movie_segments": int(
            largest_movie_count
        ),

        "largest_movie_share_pct": round(
            largest_movie_share, 2
        ),

        "top_5_movies": top_movie_details
    })


topic_coverage = pd.DataFrame(topic_records)

# 5. Add BERTopic-generated topic names
baseline_info_clean = baseline_info[
    baseline_info["Topic"] != -1
][["Topic", "Name", "Count"]].copy()

baseline_info_clean = baseline_info_clean.rename(
    columns={
        "Topic": "topic_id",
        "Name": "bertopic_name",
        "Count": "bertopic_count"
    }
)

topic_coverage = topic_coverage.merge(
    baseline_info_clean,
    on="topic_id",
    how="left"
)

# 6. Add top 10 keywords
keyword_records = []

for topic_id, group in baseline_terms.groupby("topic"):

    group = group.sort_values("term_rank")

    keywords = ", ".join(
        group["term"].head(10).astype(str)
    )

    keyword_records.append({
        "topic_id": int(topic_id),
        "top_10_keywords": keywords
    })

keyword_df = pd.DataFrame(keyword_records)

topic_coverage = topic_coverage.merge(
    keyword_df,
    on="topic_id",
    how="left"
)

# 7. Organize columns
topic_coverage = topic_coverage[
    [
        "topic_id",
        "bertopic_name",
        "top_10_keywords",
        "total_segments",
        "unique_movies",
        "unique_years",
        "largest_movie",
        "largest_movie_year",
        "largest_movie_segments",
        "largest_movie_share_pct",
        "top_5_movies"
    ]
]

topic_coverage = topic_coverage.sort_values(
    "total_segments",
    ascending=False
).reset_index(drop=True)

# 8. Save complete topic coverage
coverage_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BASELINE_CROSS_FILM_TOPIC_COVERAGE.csv"
)

topic_coverage.to_csv(
    coverage_path,
    index=False
)

# 9. Display overall topic coverage
print("\n" + "=" * 80)
print("OVERALL TOPIC COVERAGE SUMMARY")
print("=" * 80)

print("Total topics:", len(topic_coverage))

print(
    "Mean films per topic:",
    round(topic_coverage["unique_movies"].mean(), 2)
)

print(
    "Median films per topic:",
    round(topic_coverage["unique_movies"].median(), 2)
)

print(
    "Mean years per topic:",
    round(topic_coverage["unique_years"].mean(), 2)
)

# 10. Display 15 most widely distributed topics
print("\n" + "=" * 80)
print("TOPICS WITH HIGHEST CROSS-FILM COVERAGE")
print("=" * 80)

display(
    topic_coverage.sort_values(
        ["unique_movies", "unique_years"],
        ascending=False
    ).head(15)
)

# 11. Display 10 most concentrated topics
print("\n" + "=" * 80)
print("TOPICS WITH HIGHEST SINGLE-FILM CONCENTRATION")
print("=" * 80)

display(
    topic_coverage.sort_values(
        "largest_movie_share_pct",
        ascending=False
    ).head(10)
)

print("\nSaved file:")
print(coverage_path)

print("\nCross-film topic coverage analysis completed.")

In [ ]:
# ============================================================
# ICMB-130 — BROAD-FILTER BASELINE
# CROSS-FILM TOPIC COVERAGE ANALYSIS
# ============================================================

import os
import pandas as pd
import numpy as np

# 1. Load broad-filter assignments
broad_df = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "BERTOPIC_BROAD_FILTER_BASELINE_ASSIGNMENTS.csv"
    )
)

# 2. Load topic information and keywords
broad_info = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "BERTOPIC_BROAD_FILTER_BASELINE_TOPIC_INFO.csv"
    )
)

broad_terms = pd.read_csv(
    os.path.join(
        OUTPUT_DIR,
        "BERTOPIC_BROAD_FILTER_BASELINE_TOPIC_TERMS.csv"
    )
)

# 3. Exclude outliers
assigned_df = broad_df[
    broad_df["topic"] != -1
].copy()

print("=" * 80)
print("ICMB-130 — BROAD-FILTER BASELINE CROSS-FILM ANALYSIS")
print("=" * 80)

print("Total assigned segments:", len(assigned_df))
print("Distinct films:", assigned_df["movie_record_id"].nunique())
print("Distinct years:", assigned_df["year"].nunique())
print("Non-outlier topics:", assigned_df["topic"].nunique())

# 4. Calculate topic-level coverage
topic_records = []

for topic_id, group in assigned_df.groupby("topic"):

    total_segments = len(group)

    movie_counts = (
        group.groupby(
            ["movie_record_id", "movie_title", "year"]
        )
        .size()
        .reset_index(name="segment_count")
        .sort_values(
            "segment_count",
            ascending=False
        )
    )

    unique_movies = movie_counts["movie_record_id"].nunique()
    unique_years = group["year"].nunique()

    largest_movie_count = movie_counts["segment_count"].max()

    largest_movie_share = (
        largest_movie_count / total_segments
    ) * 100

    top_movies = movie_counts.head(5)

    top_movie_details = " | ".join(
        f"{row.movie_title} ({row.year}): "
        f"{row.segment_count} segments"
        for row in top_movies.itertuples()
    )

    topic_records.append({

        "topic_id": int(topic_id),

        "total_segments": total_segments,

        "unique_movies": unique_movies,

        "unique_years": unique_years,

        "largest_movie": movie_counts.iloc[0]["movie_title"],

        "largest_movie_year": int(
            movie_counts.iloc[0]["year"]
        ),

        "largest_movie_segments": int(
            largest_movie_count
        ),

        "largest_movie_share_pct": round(
            largest_movie_share, 2
        ),

        "top_5_movies": top_movie_details
    })


broad_coverage = pd.DataFrame(topic_records)

# 5. Add BERTopic-generated names
broad_info_clean = broad_info[
    broad_info["Topic"] != -1
][["Topic", "Name", "Count"]].copy()

broad_info_clean = broad_info_clean.rename(
    columns={
        "Topic": "topic_id",
        "Name": "bertopic_name",
        "Count": "bertopic_count"
    }
)

broad_coverage = broad_coverage.merge(
    broad_info_clean,
    on="topic_id",
    how="left"
)

# 6. Add top 10 keywords
topic_column = next(
    col for col in ["topic", "topic_id", "Topic"]
    if col in broad_terms.columns
)

keyword_records = []

for topic_id, group in broad_terms.groupby(topic_column):

    group = group.sort_values("term_rank")

    keywords = ", ".join(
        group["term"].head(10).astype(str)
    )

    keyword_records.append({
        "topic_id": int(topic_id),
        "top_10_keywords": keywords
    })

keyword_df = pd.DataFrame(keyword_records)

broad_coverage = broad_coverage.merge(
    keyword_df,
    on="topic_id",
    how="left"
)

# 7. Organize columns
broad_coverage = broad_coverage[
    [
        "topic_id",
        "bertopic_name",
        "top_10_keywords",
        "total_segments",
        "unique_movies",
        "unique_years",
        "largest_movie",
        "largest_movie_year",
        "largest_movie_segments",
        "largest_movie_share_pct",
        "top_5_movies"
    ]
]

broad_coverage = broad_coverage.sort_values(
    "total_segments",
    ascending=False
).reset_index(drop=True)

# 8. Save coverage report
coverage_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BROAD_BASELINE_CROSS_FILM_TOPIC_COVERAGE.csv"
)

broad_coverage.to_csv(
    coverage_path,
    index=False
)

# 9. Overall summary
print("\n" + "=" * 80)
print("BROAD-FILTER TOPIC COVERAGE SUMMARY")
print("=" * 80)

print("Total topics:", len(broad_coverage))

print(
    "Mean films per topic:",
    round(broad_coverage["unique_movies"].mean(), 2)
)

print(
    "Median films per topic:",
    round(broad_coverage["unique_movies"].median(), 2)
)

print(
    "Mean years per topic:",
    round(broad_coverage["unique_years"].mean(), 2)
)

# 10. Most widely distributed topics
print("\n" + "=" * 80)
print("TOPICS WITH HIGHEST CROSS-FILM COVERAGE")
print("=" * 80)

display(
    broad_coverage.sort_values(
        ["unique_movies", "unique_years"],
        ascending=False
    ).head(15)
)

# 11. Most concentrated topics
print("\n" + "=" * 80)
print("TOPICS WITH HIGHEST SINGLE-FILM CONCENTRATION")
print("=" * 80)

display(
    broad_coverage.sort_values(
        "largest_movie_share_pct",
        ascending=False
    ).head(10)
)

print("\nSaved file:")
print(coverage_path)

print("\nBroad-filter cross-film analysis completed.")

In [ ]:
# ============================================================
# ICMB-130 — ORIGINAL vs BROAD BASELINE
# CORRECTED CROSS-MODEL THEMATIC COMPARISON
# ============================================================

import os
import pandas as pd
import numpy as np

# 1. Select only Original Baseline vs Broad Filter matches
matches = continuity[
    (continuity["comparison"] == "Baseline — Broad Filter")
    & (continuity["match_rank"] == 1)
].copy()

# 2. Prepare Original Baseline coverage
original_columns = original_coverage[[
    "topic_id",
    "bertopic_name",
    "total_segments",
    "unique_movies",
    "unique_years",
    "largest_movie_share_pct"
]].rename(columns={
    "topic_id": "original_topic",
    "bertopic_name": "original_topic_name",
    "total_segments": "original_segments",
    "unique_movies": "original_films",
    "unique_years": "original_years",
    "largest_movie_share_pct": "original_largest_film_share"
})

# 3. Prepare Broad-filter Baseline coverage
broad_columns = broad_coverage[[
    "topic_id",
    "bertopic_name",
    "total_segments",
    "unique_movies",
    "unique_years",
    "largest_movie_share_pct"
]].rename(columns={
    "topic_id": "filtered_topic",
    "bertopic_name": "filtered_topic_name",
    "total_segments": "filtered_segments",
    "unique_movies": "filtered_films",
    "unique_years": "filtered_years",
    "largest_movie_share_pct": "filtered_largest_film_share"
})

# 4. Merge continuity matches with coverage information
comparison = matches.merge(
    original_columns,
    on="original_topic",
    how="left"
)

comparison = comparison.merge(
    broad_columns,
    on="filtered_topic",
    how="left"
)

# 5. Calculate differences
comparison["film_coverage_change"] = (
    comparison["filtered_films"] - comparison["original_films"]
)

comparison["year_coverage_change"] = (
    comparison["filtered_years"] - comparison["original_years"]
)

comparison["segment_change_pct"] = (
    (comparison["filtered_segments"] - comparison["original_segments"])
    / comparison["original_segments"].replace(0, np.nan)
) * 100

# 6. Save comparison
comparison_path = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_BASELINE_CROSS_MODEL_THEME_COMPARISON.csv"
)

comparison.to_csv(comparison_path, index=False)

# 7. Display high-coverage themes
print("=" * 90)
print("CROSS-MODEL THEMATIC CONTINUITY — HIGH-COVERAGE TOPICS")
print("=" * 90)

display(
    comparison.sort_values(
        ["original_films", "similarity"],
        ascending=False
    ).head(20)[[
        "original_topic",
        "original_topic_name",
        "original_keywords",
        "original_films",
        "original_years",
        "filtered_topic",
        "filtered_topic_name",
        "filtered_keywords",
        "filtered_films",
        "filtered_years",
        "similarity",
        "film_coverage_change"
    ]]
)

# 8. Display largest changes in film coverage
print("\n" + "=" * 90)
print("LARGEST CHANGES IN FILM COVERAGE")
print("=" * 90)

display(
    comparison.sort_values(
        "film_coverage_change"
    ).head(10)[[
        "original_topic",
        "original_topic_name",
        "original_films",
        "filtered_topic",
        "filtered_topic_name",
        "filtered_films",
        "similarity",
        "film_coverage_change"
    ]]
)

print("\nSaved comparison:")
print(comparison_path)

print("\nCross-model thematic comparison completed.")

In [ ]:
# ============================================================
# ICMB-130 — REPRESENTATIVE DIALOGUE VALIDATION
# Seven provisional recurring themes
# ============================================================

import os
import pandas as pd

# Load original baseline chunk assignments
assignments_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

assignments = pd.read_csv(assignments_path)

# Load embedding input to retrieve actual dialogue text
embedding_input_path = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_BERTOPIC_EMBEDDING_INPUT.csv"
)

embedding_input = pd.read_csv(embedding_input_path)

print("Assignment columns:")
print(assignments.columns.tolist())

print("\nEmbedding input columns:")
print(embedding_input.columns.tolist())

print("\nAssignment shape:", assignments.shape)
print("Embedding input shape:", embedding_input.shape)

In [ ]:
# ============================================================
# ICMB-130 — REPRESENTATIVE DIALOGUE VALIDATION
# ORIGINAL BASELINE
# ============================================================

import pandas as pd
import os

# Seven provisional recurring themes
candidate_topics = {
    0: "Love and relationships",
    1: "Love and relationships",
    2: "Sports and competition",
    3: "Marriage and weddings",
    4: "Family and parent-child relationships",
    7: "Money, business and finance",
    9: "Police and law enforcement",
    10: "Healthcare and medicine"
}

# Confirm segment alignment before extraction
segment_keys = [
    "movie_record_id",
    "source_chunk_id",
    "part_number"
]

assert len(assignments) == len(embedding_input)
assert not assignments.duplicated(segment_keys).any()
assert not embedding_input.duplicated(segment_keys).any()

# Merge actual dialogue with topic assignments
dialogue_data = assignments.merge(
    embedding_input[
        segment_keys + ["text"]
    ],
    on=segment_keys,
    how="inner",
    suffixes=("_assignment", "_input"),
    validate="one_to_one"
)

assert len(dialogue_data) == len(assignments)

# Confirm text alignment
assert (
    dialogue_data["text_assignment"]
    == dialogue_data["text_input"]
).all()

dialogue_data["text"] = dialogue_data["text_assignment"]

# Extract five representative segments per topic
representative_samples = []

for topic_id, theme_name in candidate_topics.items():

    topic_data = dialogue_data[
        dialogue_data["topic"] == topic_id
    ].copy()

    topic_data = topic_data.sort_values(
        "token_count",
        ascending=False
    )

    samples = topic_data.head(5).copy()

    samples["provisional_theme"] = theme_name
    samples["sample_rank"] = range(1, len(samples) + 1)

    representative_samples.append(samples)

# Combine samples
representative_samples = pd.concat(
    representative_samples,
    ignore_index=True
)

# Select useful columns
representative_samples = representative_samples[[
    "topic",
    "provisional_theme",
    "sample_rank",
    "movie_title",
    "year",
    "token_count",
    "text"
]]

# Save for detailed review
output_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BASELINE_REPRESENTATIVE_DIALOGUE_VALIDATION.csv"
)

representative_samples.to_csv(
    output_path,
    index=False
)

# Display samples
for topic_id, theme_name in candidate_topics.items():

    print("\n" + "=" * 90)
    print(f"TOPIC {topic_id} — {theme_name}")
    print("=" * 90)

    topic_samples = representative_samples[
        representative_samples["topic"] == topic_id
    ]

    for _, row in topic_samples.iterrows():
        print(
            f"\nSample {row['sample_rank']} | "
            f"{row['movie_title']} ({row['year']}) | "
            f"{row['token_count']} tokens"
        )
        print(row["text"])

print("\nSaved representative dialogue:")
print(output_path)

In [ ]:
# ============================================================
# ICMB-130 — CROSS-FILM REPRESENTATIVE DIALOGUE VALIDATION
# One representative segment per film
# ============================================================

import pandas as pd
import os

candidate_topics = {
    0: "Love and relationships",
    1: "Love and relationships",
    2: "Sports and competition",
    3: "Marriage and weddings",
    4: "Family and parent-child relationships",
    7: "Money, business and finance",
    9: "Police and law enforcement",
    10: "Healthcare and medicine"
}

# Use the already validated dialogue_data dataframe
cross_film_samples = []

for topic_id, theme_name in candidate_topics.items():

    topic_data = dialogue_data[
        dialogue_data["topic"] == topic_id
    ].copy()

    # Count topic segments contributed by each film
    film_counts = (
        topic_data.groupby(
            ["movie_record_id", "movie_title", "year"]
        )
        .size()
        .reset_index(name="film_topic_segments")
        .sort_values(
            "film_topic_segments",
            ascending=False
        )
    )

    # Select five different films with the highest contribution
    selected_films = film_counts.head(5)

    for _, film in selected_films.iterrows():

        film_segments = topic_data[
            topic_data["movie_record_id"]
            == film["movie_record_id"]
        ].copy()

        # Select the segment with the highest token count
        sample = film_segments.sort_values(
            "token_count",
            ascending=False
        ).head(1).copy()

        sample["provisional_theme"] = theme_name
        sample["film_topic_segments"] = film["film_topic_segments"]

        cross_film_samples.append(sample)

# Combine all selected samples
cross_film_samples = pd.concat(
    cross_film_samples,
    ignore_index=True
)

# Save the cross-film validation output
cross_film_samples = cross_film_samples[[
    "topic",
    "provisional_theme",
    "movie_title",
    "year",
    "film_topic_segments",
    "token_count",
    "text"
]]

output_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BASELINE_CROSS_FILM_REPRESENTATIVE_VALIDATION.csv"
)

cross_film_samples.to_csv(
    output_path,
    index=False
)

# Display results
for topic_id, theme_name in candidate_topics.items():

    print("\n" + "=" * 90)
    print(f"TOPIC {topic_id} — {theme_name}")
    print("=" * 90)

    samples = cross_film_samples[
        cross_film_samples["topic"] == topic_id
    ]

    for _, row in samples.iterrows():

        print(
            f"\nFilm: {row['movie_title']} ({row['year']})"
            f"\nSegments assigned to this topic in film: "
            f"{row['film_topic_segments']}"
            f"\nSelected segment tokens: {row['token_count']}"
        )

        print(row["text"])

print("\nSaved cross-film validation:")
print(output_path)

In [ ]:
# ============================================================
# ICMB-130 — SCREEN REMAINING BROAD TOPICS
# ============================================================

reviewed_topics = [0, 1, 2, 3, 4, 7, 9, 10]

remaining_topics = original_coverage[
    (~original_coverage["topic_id"].isin(reviewed_topics))
    & (original_coverage["unique_movies"] >= 20)
    & (original_coverage["unique_years"] >= 10)
].copy()

remaining_topics = remaining_topics.sort_values(
    ["unique_movies", "unique_years", "total_segments"],
    ascending=False
)

print("=" * 90)
print("REMAINING CROSS-FILM TOPICS FOR THEMATIC REVIEW")
print("=" * 90)

print("Topics already reviewed:", len(reviewed_topics))
print("Remaining broad topics:", len(remaining_topics))

display(
    remaining_topics[[
        "topic_id",
        "bertopic_name",
        "top_10_keywords",
        "total_segments",
        "unique_movies",
        "unique_years",
        "largest_movie",
        "largest_movie_share_pct"
    ]]
)

print("\nScreening completed.")

In [ ]:
# ============================================================
# ICMB-130 — REPRESENTATIVE DIALOGUE VALIDATION
# ADDITIONAL THEMATIC CANDIDATES
# ============================================================

additional_candidates = {
    12: "Politics and political leadership",
    16: "Terrorism and national security",
    20: "Courts and legal proceedings",
    26: "Education and academic institutions",
    30: "Military and armed conflict",
    35: "Religion and devotional beliefs",
    36: "Police and criminal justice",
    56: "Government and public administration"
}

additional_samples = []

for topic_id, theme_name in additional_candidates.items():

    topic_data = dialogue_data[
        dialogue_data["topic"] == topic_id
    ].copy()

    # Count contribution from each film
    film_counts = (
        topic_data.groupby(
            ["movie_record_id", "movie_title", "year"]
        )
        .size()
        .reset_index(name="film_topic_segments")
        .sort_values(
            "film_topic_segments",
            ascending=False
        )
    )

    # Select five different films
    selected_films = film_counts.head(5)

    for _, film in selected_films.iterrows():

        film_segments = topic_data[
            topic_data["movie_record_id"]
            == film["movie_record_id"]
        ].copy()

        sample = film_segments.sort_values(
            "token_count",
            ascending=False
        ).head(1).copy()

        sample["provisional_theme"] = theme_name
        sample["film_topic_segments"] = film["film_topic_segments"]

        additional_samples.append(sample)

additional_samples = pd.concat(
    additional_samples,
    ignore_index=True
)

# Save validation output
additional_samples = additional_samples[[
    "topic",
    "provisional_theme",
    "movie_title",
    "year",
    "film_topic_segments",
    "token_count",
    "text"
]]

output_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BASELINE_ADDITIONAL_THEME_VALIDATION.csv"
)

additional_samples.to_csv(
    output_path,
    index=False
)

# Display samples
for topic_id, theme_name in additional_candidates.items():

    print("\n" + "=" * 90)
    print(f"TOPIC {topic_id} — {theme_name}")
    print("=" * 90)

    samples = additional_samples[
        additional_samples["topic"] == topic_id
    ]

    for _, row in samples.iterrows():

        print(
            f"\nFilm: {row['movie_title']} ({row['year']})"
            f"\nTopic segments in film: {row['film_topic_segments']}"
        )

        print(row["text"])

print("\nSaved validation file:")
print(output_path)

print("\nAdditional thematic validation completed.")

In [ ]:
# ============================================================
# ICMB-130 — VALIDATION OF OVERLAPPING THEMATIC TOPICS
# ============================================================

overlap_candidates = {
    9: "Police and law enforcement",
    16: "National security and terrorism",
    20: "Courts and legal proceedings",
    30: "Military and armed conflict",
    36: "Police and criminal justice",
    56: "Government and public administration"
}

overlap_samples = []

for topic_id, theme_name in overlap_candidates.items():

    topic_data = dialogue_data[
        dialogue_data["topic"] == topic_id
    ].copy()

    film_counts = (
        topic_data.groupby(
            ["movie_record_id", "movie_title", "year"]
        )
        .size()
        .reset_index(name="film_topic_segments")
        .sort_values(
            "film_topic_segments",
            ascending=False
        )
    )

    selected_films = film_counts.head(5)

    for _, film in selected_films.iterrows():

        film_segments = topic_data[
            topic_data["movie_record_id"]
            == film["movie_record_id"]
        ].copy()

        sample = film_segments.sort_values(
            "token_count",
            ascending=False
        ).head(1).copy()

        sample["provisional_theme"] = theme_name
        sample["film_topic_segments"] = film["film_topic_segments"]

        overlap_samples.append(sample)

overlap_samples = pd.concat(
    overlap_samples,
    ignore_index=True
)

overlap_samples = overlap_samples[[
    "topic",
    "provisional_theme",
    "movie_title",
    "year",
    "film_topic_segments",
    "text"
]]

output_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BASELINE_OVERLAPPING_THEME_VALIDATION.csv"
)

overlap_samples.to_csv(
    output_path,
    index=False
)

for topic_id, theme_name in overlap_candidates.items():

    print("\n" + "=" * 90)
    print(f"TOPIC {topic_id} — {theme_name}")
    print("=" * 90)

    samples = overlap_samples[
        overlap_samples["topic"] == topic_id
    ]

    for _, row in samples.iterrows():

        print(
            f"\n{row['movie_title']} ({row['year']})"
            f" | {row['film_topic_segments']} topic segments"
        )
        print(row["text"])

print("\nSaved:", output_path)

In [ ]:
# ============================================================
# ICMB-130 — NEXT THEMATIC VALIDATION BATCH
# ============================================================

next_candidates = {
    8: "Food and everyday life",
    23: "Marriage and relationship situations",
    38: "Transportation and driving",
    49: "Crime, theft and imprisonment",
    50: "Death, violence and suicide",
    54: "Railways and public transportation",
    55: "Alcohol consumption",
    60: "Romantic relationships and emotions"
}

next_samples = []

for topic_id, theme_name in next_candidates.items():

    topic_data = dialogue_data[
        dialogue_data["topic"] == topic_id
    ].copy()

    film_counts = (
        topic_data.groupby(
            ["movie_record_id", "movie_title", "year"]
        )
        .size()
        .reset_index(name="film_topic_segments")
        .sort_values(
            "film_topic_segments",
            ascending=False
        )
    )

    for _, film in film_counts.head(5).iterrows():

        film_segments = topic_data[
            topic_data["movie_record_id"]
            == film["movie_record_id"]
        ]

        sample = film_segments.sort_values(
            "token_count",
            ascending=False
        ).head(1).copy()

        sample["provisional_theme"] = theme_name
        sample["film_topic_segments"] = film["film_topic_segments"]

        next_samples.append(sample)

next_samples = pd.concat(
    next_samples,
    ignore_index=True
)

next_samples = next_samples[[
    "topic",
    "provisional_theme",
    "movie_title",
    "year",
    "film_topic_segments",
    "text"
]]

output_path = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BASELINE_NEXT_THEME_VALIDATION.csv"
)

next_samples.to_csv(
    output_path,
    index=False
)

for topic_id, theme_name in next_candidates.items():

    print("\n" + "=" * 90)
    print(f"TOPIC {topic_id} — {theme_name}")
    print("=" * 90)

    samples = next_samples[
        next_samples["topic"] == topic_id
    ]

    for _, row in samples.iterrows():
        print(
            f"\n{row['movie_title']} ({row['year']})"
            f" | {row['film_topic_segments']} topic segments"
        )
        print(row["text"])

print("\nSaved:", output_path)

In [ ]:
# ============================================================
# ICMB-130 — LOCATE EXISTING MOVIE METADATA
# ============================================================

import os
import glob
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

search_dirs = [
    os.path.join(PROJECT_ROOT, "01_data"),
    os.path.join(PROJECT_ROOT, "06_outputs")
]

keywords = [
    "metadata", "movie", "feature", "master",
    "final", "tmdb", "enriched"
]

metadata_candidates = []

for folder in search_dirs:
    if os.path.exists(folder):
        for path in glob.glob(
            os.path.join(folder, "**", "*.csv"),
            recursive=True
        ):
            filename = os.path.basename(path).lower()

            if any(word in filename for word in keywords):
                metadata_candidates.append(path)

print("=" * 85)
print("POTENTIAL MOVIE METADATA FILES")
print("=" * 85)

for i, path in enumerate(metadata_candidates, 1):

    try:
        columns = pd.read_csv(path, nrows=0).columns.tolist()

        print(f"\n{i}. {os.path.basename(path)}")
        print("Path:", path)
        print("Columns:", columns)

        relevant_columns = [
            col for col in columns
            if any(term in col.lower() for term in [
                "movie", "title", "year", "genre",
                "overview", "summary", "description",
                "plot", "language", "tmdb", "record_id"
            ])
        ]

        if relevant_columns:
            preview = pd.read_csv(
                path,
                usecols=relevant_columns,
                nrows=2
            )

            print("Preview:")
            display(preview)

    except Exception as e:
        print("Could not inspect:", e)

if not metadata_candidates:
    print("\nNo likely metadata CSV found in the searched folders.")

In [ ]:
# ============================================================
# ICMB-130 — METADATA QUALITY AND MERGE READINESS
# ============================================================

import os
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/drive/MyDrive/IndianCinemaTrendAnalytics"

METADATA_PATH = os.path.join(
    PROJECT_ROOT,
    "06_outputs/topic_modelling/ICMB_130_TOPIC_ANALYSIS_METADATA.csv"
)

metadata = pd.read_csv(METADATA_PATH)

print("=" * 75)
print("MOVIE METADATA QUALITY CHECK")
print("=" * 75)

print("Shape:", metadata.shape)
print("Unique movie IDs:", metadata["movie_record_id"].nunique())
print("Duplicate movie IDs:", metadata["movie_record_id"].duplicated().sum())

required_columns = [
    "movie_record_id",
    "movie_title",
    "year",
    "genres",
    "overview",
    "industry",
    "film_language"
]

print("\nRequired column availability:")
for col in required_columns:
    print(f"{col}: {'Available' if col in metadata.columns else 'MISSING'}")

print("\nMetadata completeness:")
for col in ["genres", "overview", "industry", "film_language"]:
    if col in metadata.columns:
        non_missing = metadata[col].notna().sum()
        non_empty = metadata[col].astype(str).str.strip().ne("").sum()

        print(
            f"{col}: {non_empty}/{len(metadata)} "
            f"({non_empty / len(metadata) * 100:.2f}%)"
        )

print("\nYear coverage:")
print(
    metadata["year"].value_counts().sort_index().to_string()
)

print("\nSample metadata:")
display(
    metadata[
        [
            "movie_record_id",
            "movie_title",
            "year",
            "film_language",
            "genres",
            "overview",
            "industry"
        ]
    ].head(8)
)

print("\nMetadata quality check completed.")

In [ ]:
# ============================================================
# NOTEBOOK 09 — BUILD TOPIC–MOVIE METADATA CONTEXT
# Source: Original baseline chunk-topic assignments
# ============================================================

import pandas as pd
import os

OUTPUT_DIR = "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"

ASSIGNMENT_PATH = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

METADATA_PATH = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_TOPIC_ANALYSIS_METADATA.csv"
)

# ------------------------------------------------------------
# 1. Load files
# ------------------------------------------------------------

assignments = pd.read_csv(ASSIGNMENT_PATH)
metadata = pd.read_csv(METADATA_PATH)

print("=" * 70)
print("BASELINE ASSIGNMENT–METADATA INTEGRATION")
print("=" * 70)

print("\nAssignment shape:", assignments.shape)
print("Metadata shape:", metadata.shape)

print("\nAssignment columns:")
print(assignments.columns.tolist())

# ------------------------------------------------------------
# 2. Identify required assignment columns
# ------------------------------------------------------------

required_assignment_cols = ["movie_record_id", "topic"]

missing = [
    col for col in required_assignment_cols
    if col not in assignments.columns
]

if missing:
    raise ValueError(
        f"Required columns missing from assignments: {missing}"
    )

assert metadata["movie_record_id"].is_unique
assert metadata["movie_record_id"].notna().all()

# ------------------------------------------------------------
# 3. Create topic–movie chunk distribution
# ------------------------------------------------------------

topic_movie_counts = (
    assignments
    .groupby(["topic", "movie_record_id"])
    .size()
    .reset_index(name="topic_chunks")
)

# ------------------------------------------------------------
# 4. Add movie metadata
# ------------------------------------------------------------

metadata_subset = metadata[
    [
        "movie_record_id",
        "movie_title",
        "year",
        "film_language",
        "genres",
        "overview",
        "industry"
    ]
].copy()

topic_context = topic_movie_counts.merge(
    metadata_subset,
    on="movie_record_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("\nMerge status:")
print(topic_context["_merge"].value_counts())

assert (topic_context["_merge"] == "both").all()

topic_context.drop(columns="_merge", inplace=True)

# ------------------------------------------------------------
# 5. Summarize topic coverage
# ------------------------------------------------------------

topic_summary = (
    topic_context
    .groupby("topic")
    .agg(
        total_chunks=("topic_chunks", "sum"),
        unique_movies=("movie_record_id", "nunique"),
        unique_years=("year", "nunique")
    )
    .reset_index()
)

print("\nIntegrated topic summary shape:", topic_summary.shape)
print("Unique topics:", topic_summary["topic"].nunique())
print("Unique movies:", topic_context["movie_record_id"].nunique())

# ------------------------------------------------------------
# 6. Save outputs
# ------------------------------------------------------------

CONTEXT_PATH = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BASELINE_TOPIC_MOVIE_METADATA_CONTEXT.csv"
)

SUMMARY_PATH = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BASELINE_TOPIC_METADATA_SUMMARY.csv"
)

topic_context.to_csv(CONTEXT_PATH, index=False)
topic_summary.to_csv(SUMMARY_PATH, index=False)

print("\nSaved topic–movie context:", CONTEXT_PATH)
print("Saved topic summary:", SUMMARY_PATH)

# ------------------------------------------------------------
# 7. Display sample records
# ------------------------------------------------------------

display(topic_context.head(10))
display(topic_summary.head(10))

print("\nIntegration completed successfully.")

In [ ]:
# ============================================================
# NOTEBOOK 09 — AMBIGUOUS TOPIC CONTEXTUAL VALIDATION
# ============================================================

import pandas as pd
import ast
import os

OUTPUT_DIR = "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"

CONTEXT_PATH = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_BASELINE_TOPIC_MOVIE_METADATA_CONTEXT.csv"
)

df_context = pd.read_csv(CONTEXT_PATH)

# Candidate topics requiring contextual review
candidate_topics = [5, 6, 8, 30, 36, 49, 55, 56]

# ------------------------------------------------------------
# Helper: extract readable genre names
# ------------------------------------------------------------

def extract_genres(value):
    try:
        parsed = ast.literal_eval(value) if isinstance(value, str) else []
        if isinstance(parsed, list):
            return ", ".join(
                item.get("name", "")
                for item in parsed
                if isinstance(item, dict) and item.get("name")
            )
    except (ValueError, SyntaxError):
        pass
    return str(value) if pd.notna(value) else ""

df_context["genre_names"] = df_context["genres"].apply(extract_genres)

# ------------------------------------------------------------
# Display candidate topic evidence
# ------------------------------------------------------------

for topic_id in candidate_topics:

    topic_df = (
        df_context[df_context["topic"] == topic_id]
        .sort_values("topic_chunks", ascending=False)
    )

    if topic_df.empty:
        print(f"\nTopic {topic_id}: Not found")
        continue

    print("\n" + "=" * 90)
    print(f"TOPIC {topic_id}")
    print("=" * 90)

    print(
        "Total chunks:", topic_df["topic_chunks"].sum(),
        "| Unique films:", topic_df["movie_record_id"].nunique(),
        "| Unique years:", topic_df["year"].nunique()
    )

    print("\nTop contributing films:")

    for _, row in topic_df.head(8).iterrows():

        overview = str(row["overview"])
        if len(overview) > 450:
            overview = overview[:450] + "..."

        print(f"\nFilm: {row['movie_title']} ({row['year']})")
        print("Language:", row["film_language"])
        print("Industry:", row["industry"])
        print("Genres:", row["genre_names"])
        print("Topic chunks:", row["topic_chunks"])
        print("Overview:", overview)

print("\n\nContextual inspection completed.")

In [ ]:
# ============================================================
# NOTEBOOK 09 — FINAL CURATED THEME REGISTER
# ============================================================

import pandas as pd
import os

OUTPUT_DIR = "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"

theme_records = [
    [1, "Love and romantic relationships", "0, 1, 60",
     "Included", "High",
     "Broad recurring relationship vocabulary; overlapping topic clusters."],

    [2, "Marriage and weddings", "3, 23",
     "Included", "High",
     "Consistent marriage and wedding-related dialogue."],

    [3, "Family and parent-child relationships", "4",
     "Included", "High",
     "Broad cross-film family and parent-child dialogue."],

    [4, "Sports and competition", "2",
     "Included", "High",
     "Clear sports-related dialogue across multiple films and years."],

    [5, "Money, wealth and economic transactions", "7",
     "Included", "High",
     "Recurring financial and economic vocabulary."],

    [6, "Police and law enforcement", "9, 36",
     "Included", "High",
     "Topic 36 overlaps the established police and law-enforcement theme."],

    [7, "Healthcare and medicine", "10",
     "Included", "Medium",
     "Medical and healthcare dialogue; narrower interpretation required."],

    [8, "Politics, governance and corruption", "12, 56",
     "Included", "High",
     "Political institutions, elections, governance and corruption."],

    [9, "National security and terrorism", "16",
     "Included", "High",
     "Recurring national security, intelligence and terrorism-related dialogue."],

    [10, "Courts and legal proceedings", "20",
     "Included", "Medium",
     "Legal and court-related dialogue; some film concentration."],

    [11, "Education and academic institutions", "26",
     "Included", "Medium-High",
     "Recurring educational and institutional context."],

    [12, "Religion and devotional beliefs", "35",
     "Included", "Medium-High",
     "Religious beliefs, deities and rituals across multiple films."],

    [None, "Alcohol consumption and related behaviour", "55",
     "Exploratory", "Low",
     "Small cluster; retain separately for exploratory discussion."],

    [None, "Film-concentrated mixed cluster", "5",
     "Excluded", "Low",
     "Dominated by Enthiran and Vikram."],

    [None, "Generic mixed vocabulary", "6",
     "Excluded", "Low",
     "No sufficiently coherent standalone theme."],

    [None, "Food and everyday-life candidate", "8",
     "Excluded", "Low",
     "Dialogue and metadata do not establish a distinct recurring theme."],

    [None, "Military and mixed action vocabulary", "30",
     "Excluded", "Low",
     "Mixed contexts; insufficient evidence for a standalone theme."],

    [None, "Crime and theft candidate", "49",
     "Excluded", "Low",
     "Small and scattered cluster."],

]

theme_register = pd.DataFrame(
    theme_records,
    columns=[
        "theme_number",
        "curated_theme",
        "baseline_topic_ids",
        "decision",
        "confidence",
        "decision_rationale"
    ]
)

SAVE_PATH = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_FINAL_CURATED_THEME_REGISTER.csv"
)

theme_register.to_csv(SAVE_PATH, index=False)

print("=" * 75)
print("FINAL CURATED THEME REGISTER")
print("=" * 75)

print("\nMain themes:", (theme_register["decision"] == "Included").sum())
print("Exploratory themes:", (theme_register["decision"] == "Exploratory").sum())
print("Excluded candidates:", (theme_register["decision"] == "Excluded").sum())

display(theme_register)

print("\nSaved:", SAVE_PATH)

In [ ]:
# ============================================================
# NOTEBOOK 09 — YEAR-WISE CURATED THEMATIC PREVALENCE
# Period: 2000–2025
# ============================================================

import pandas as pd
import numpy as np
import os

OUTPUT_DIR = "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"

ASSIGNMENT_PATH = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

REGISTER_PATH = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_FINAL_CURATED_THEME_REGISTER.csv"
)

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

assignments = pd.read_csv(ASSIGNMENT_PATH)
register = pd.read_csv(REGISTER_PATH)

# ------------------------------------------------------------
# 2. Define the curated topic-to-theme mapping
# ------------------------------------------------------------

theme_mapping = {
    "Love and romantic relationships": [0, 1, 60],
    "Marriage and weddings": [3, 23],
    "Family and parent-child relationships": [4],
    "Sports and competition": [2],
    "Money, wealth and economic transactions": [7],
    "Police and law enforcement": [9, 36],
    "Healthcare and medicine": [10],
    "Politics, governance and corruption": [12, 56],
    "National security and terrorism": [16],
    "Courts and legal proceedings": [20],
    "Education and academic institutions": [26],
    "Religion and devotional beliefs": [35]
}

# ------------------------------------------------------------
# 3. Validate assignment data
# ------------------------------------------------------------

required_cols = ["movie_record_id", "year", "topic"]

for col in required_cols:
    assert col in assignments.columns, f"Missing column: {col}"

assignments["topic"] = pd.to_numeric(
    assignments["topic"], errors="coerce"
)

assignments["year"] = pd.to_numeric(
    assignments["year"], errors="coerce"
)

assert assignments["movie_record_id"].notna().all()
assert assignments["topic"].notna().all()
assert assignments["year"].notna().all()

# Exclude outliers from thematic prevalence
assigned = assignments[assignments["topic"] != -1].copy()

# ------------------------------------------------------------
# 4. Calculate annual denominators
# ------------------------------------------------------------

annual_denominators = (
    assigned.groupby("year")
    .agg(
        total_assigned_chunks=("topic", "size"),
        total_movies=("movie_record_id", "nunique")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. Calculate theme-level annual prevalence
# ------------------------------------------------------------

results = []

for theme, topic_ids in theme_mapping.items():

    theme_data = assigned[
        assigned["topic"].isin(topic_ids)
    ].copy()

    annual_theme = (
        theme_data.groupby("year")
        .agg(
            theme_chunks=("topic", "size"),
            films_with_theme=("movie_record_id", "nunique")
        )
        .reset_index()
    )

    annual_theme["curated_theme"] = theme

    results.append(annual_theme)

theme_year = pd.concat(results, ignore_index=True)

# ------------------------------------------------------------
# 6. Create complete 2000–2025 theme-year grid
# ------------------------------------------------------------

years = list(range(2000, 2026))

grid = pd.MultiIndex.from_product(
    [list(theme_mapping.keys()), years],
    names=["curated_theme", "year"]
).to_frame(index=False)

theme_year = grid.merge(
    theme_year,
    on=["curated_theme", "year"],
    how="left"
)

theme_year = theme_year.merge(
    annual_denominators,
    on="year",
    how="left"
)

count_cols = [
    "theme_chunks",
    "films_with_theme",
    "total_assigned_chunks",
    "total_movies"
]

theme_year[count_cols] = theme_year[count_cols].fillna(0)

# ------------------------------------------------------------
# 7. Calculate normalized prevalence
# ------------------------------------------------------------

theme_year["film_coverage_pct"] = np.where(
    theme_year["total_movies"] > 0,
    theme_year["films_with_theme"] /
    theme_year["total_movies"] * 100,
    0
)

theme_year["dialogue_prevalence_pct"] = np.where(
    theme_year["total_assigned_chunks"] > 0,
    theme_year["theme_chunks"] /
    theme_year["total_assigned_chunks"] * 100,
    0
)

theme_year["film_coverage_pct"] = (
    theme_year["film_coverage_pct"].round(2)
)

theme_year["dialogue_prevalence_pct"] = (
    theme_year["dialogue_prevalence_pct"].round(2)
)

theme_year = theme_year.sort_values(
    ["curated_theme", "year"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 8. Save detailed year-wise results
# ------------------------------------------------------------

DETAIL_PATH = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_CURATED_THEME_YEARLY_PREVALENCE.csv"
)

theme_year.to_csv(DETAIL_PATH, index=False)

# ------------------------------------------------------------
# 9. Create pivot tables for analysis
# ------------------------------------------------------------

film_coverage_pivot = theme_year.pivot(
    index="curated_theme",
    columns="year",
    values="films_with_theme"
)

dialogue_prevalence_pivot = theme_year.pivot(
    index="curated_theme",
    columns="year",
    values="dialogue_prevalence_pct"
)

film_coverage_pivot.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ICMB_130_THEME_YEARLY_FILM_COVERAGE_MATRIX.csv"
    )
)

dialogue_prevalence_pivot.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ICMB_130_THEME_YEARLY_DIALOGUE_PREVALENCE_MATRIX.csv"
    )
)

# ------------------------------------------------------------
# 10. Validation and preview
# ------------------------------------------------------------

print("=" * 75)
print("YEAR-WISE CURATED THEMATIC PREVALENCE")
print("=" * 75)

print("\nTotal original chunks:", len(assignments))
print("Non-outlier assigned chunks:", len(assigned))
print("Outlier chunks excluded:", (assignments["topic"] == -1).sum())

print("\nNumber of curated themes:", len(theme_mapping))
print("Number of years:", len(years))
print("Expected theme-year combinations:", len(theme_mapping) * len(years))
print("Actual theme-year combinations:", len(theme_year))

print("\nMissing theme-year records:",
      theme_year[["theme_chunks", "films_with_theme"]].isna().any(axis=1).sum())

print("\nAnnual denominator check:")
display(annual_denominators)

print("\nFilm coverage matrix (number of films out of five):")
display(film_coverage_pivot)

print("\nDialogue prevalence matrix (% of assigned chunks):")
display(dialogue_prevalence_pivot.round(2))

print("\nSaved files:")
print(DETAIL_PATH)
print("ICMB_130_THEME_YEARLY_FILM_COVERAGE_MATRIX.csv")
print("ICMB_130_THEME_YEARLY_DIALOGUE_PREVALENCE_MATRIX.csv")

print("\nYear-wise thematic prevalence calculation completed.")

In [ ]:
# ============================================================
# NOTEBOOK 09 — ANNUAL THEME CONCENTRATION BY FILM
# Corrected metadata merge
# ============================================================

import pandas as pd
import numpy as np
import os

OUTPUT_DIR = "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"

ASSIGNMENT_PATH = os.path.join(
    OUTPUT_DIR,
    "BERTOPIC_CHUNK_TOPIC_ASSIGNMENTS.csv"
)

METADATA_PATH = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_TOPIC_ANALYSIS_METADATA.csv"
)

theme_mapping = {
    "Love and romantic relationships": [0, 1, 60],
    "Marriage and weddings": [3, 23],
    "Family and parent-child relationships": [4],
    "Sports and competition": [2],
    "Money, wealth and economic transactions": [7],
    "Police and law enforcement": [9, 36],
    "Healthcare and medicine": [10],
    "Politics, governance and corruption": [12, 56],
    "National security and terrorism": [16],
    "Courts and legal proceedings": [20],
    "Education and academic institutions": [26],
    "Religion and devotional beliefs": [35]
}

# ------------------------------------------------------------
# 1. Load files
# ------------------------------------------------------------

assignments = pd.read_csv(ASSIGNMENT_PATH)
metadata = pd.read_csv(METADATA_PATH)

# ------------------------------------------------------------
# 2. Prepare assignment data
# ------------------------------------------------------------

assigned = assignments[
    assignments["topic"] != -1
].copy()

assigned["topic"] = pd.to_numeric(assigned["topic"])
assigned["year"] = pd.to_numeric(assigned["year"])

# ------------------------------------------------------------
# 3. Merge only non-duplicate metadata columns
# ------------------------------------------------------------

movie_info = metadata[
    [
        "movie_record_id",
        "film_language",
        "industry"
    ]
].drop_duplicates("movie_record_id")

assigned = assigned.merge(
    movie_info,
    on="movie_record_id",
    how="left",
    validate="many_to_one"
)

assert assigned["movie_title"].notna().all()
assert assigned["film_language"].notna().all()
assert assigned["industry"].notna().all()

print("Metadata merge successful.")
print("Assigned chunks:", len(assigned))
print("Unique movies:", assigned["movie_record_id"].nunique())

# ------------------------------------------------------------
# 4. Calculate film contributions by theme and year
# ------------------------------------------------------------

records = []

for theme, topic_ids in theme_mapping.items():

    theme_data = assigned[
        assigned["topic"].isin(topic_ids)
    ].copy()

    film_counts = (
        theme_data
        .groupby(
            [
                "year",
                "movie_record_id",
                "movie_title",
                "film_language",
                "industry"
            ]
        )
        .size()
        .reset_index(name="theme_chunks")
    )

    annual_totals = (
        film_counts.groupby("year")["theme_chunks"]
        .sum()
        .rename("annual_theme_chunks")
    )

    film_counts = film_counts.merge(
        annual_totals,
        on="year",
        how="left"
    )

    film_counts["film_share_pct"] = (
        film_counts["theme_chunks"] /
        film_counts["annual_theme_chunks"] * 100
    )

    film_counts["curated_theme"] = theme

    records.append(film_counts)

concentration = pd.concat(records, ignore_index=True)

# ------------------------------------------------------------
# 5. Identify leading film per theme-year
# ------------------------------------------------------------

leading_films = (
    concentration
    .sort_values(
        ["curated_theme", "year", "theme_chunks"],
        ascending=[True, True, False]
    )
    .groupby(["curated_theme", "year"])
    .head(1)
    .reset_index(drop=True)
)

leading_films = leading_films.rename(
    columns={
        "movie_title": "leading_film",
        "film_language": "leading_film_language",
        "industry": "leading_film_industry",
        "theme_chunks": "leading_film_theme_chunks",
        "film_share_pct": "leading_film_share_pct"
    }
)

# ------------------------------------------------------------
# 6. Create annual concentration summary
# ------------------------------------------------------------

annual_concentration = (
    concentration
    .groupby(["curated_theme", "year"])
    .agg(
        films_contributing=("movie_record_id", "nunique"),
        total_theme_chunks=("theme_chunks", "sum")
    )
    .reset_index()
)

annual_concentration = annual_concentration.merge(
    leading_films[
        [
            "curated_theme",
            "year",
            "leading_film",
            "leading_film_language",
            "leading_film_industry",
            "leading_film_theme_chunks",
            "leading_film_share_pct"
        ]
    ],
    on=["curated_theme", "year"],
    how="left"
)

annual_concentration["leading_film_share_pct"] = (
    annual_concentration["leading_film_share_pct"].round(2)
)

# ------------------------------------------------------------
# 7. Save outputs
# ------------------------------------------------------------

FILM_DETAIL_PATH = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_THEME_YEAR_FILM_CONTRIBUTIONS.csv"
)

ANNUAL_PATH = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_THEME_YEAR_FILM_CONCENTRATION.csv"
)

concentration.to_csv(FILM_DETAIL_PATH, index=False)
annual_concentration.to_csv(ANNUAL_PATH, index=False)

# ------------------------------------------------------------
# 8. Display results
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("THEME CONCENTRATION BY FILM AND YEAR")
print("=" * 75)

print("\nFilm-level records:", len(concentration))
print("Theme-year records:", len(annual_concentration))

print("\nHighest leading-film concentration:")
display(
    annual_concentration.sort_values(
        "leading_film_share_pct",
        ascending=False
    ).head(25)
)

print("\nAverage leading-film share by theme:")
display(
    annual_concentration
    .groupby("curated_theme")["leading_film_share_pct"]
    .mean()
    .sort_values(ascending=False)
    .round(2)
    .reset_index(name="average_leading_film_share_pct")
)

print("\nSaved files:")
print(FILM_DETAIL_PATH)
print(ANNUAL_PATH)

print("\nAnalysis completed successfully.")

In [ ]:
# ============================================================
# NOTEBOOK 09 — ROBUST THEME CONCENTRATION DIAGNOSTIC
# ============================================================

import pandas as pd
import numpy as np
import os

OUTPUT_DIR = "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"

PATH = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_THEME_YEAR_FILM_CONCENTRATION.csv"
)

annual = pd.read_csv(PATH)

# ------------------------------------------------------------
# 1. Separate meaningful theme-year observations
# ------------------------------------------------------------

annual["leading_film_share_pct"] = pd.to_numeric(
    annual["leading_film_share_pct"],
    errors="coerce"
)

annual["total_theme_chunks"] = pd.to_numeric(
    annual["total_theme_chunks"],
    errors="coerce"
)

annual["films_contributing"] = pd.to_numeric(
    annual["films_contributing"],
    errors="coerce"
)

# Use 10 chunks as a minimum support threshold
supported = annual[
    annual["total_theme_chunks"] >= 10
].copy()

# ------------------------------------------------------------
# 2. Calculate robust concentration by theme
# ------------------------------------------------------------

summary = (
    supported.groupby("curated_theme")
    .agg(
        supported_years=("year", "nunique"),
        mean_theme_chunks=("total_theme_chunks", "mean"),
        mean_leading_film_share=(
            "leading_film_share_pct", "mean"
        ),
        median_leading_film_share=(
            "leading_film_share_pct", "median"
        ),
        mean_films_contributing=(
            "films_contributing", "mean"
        )
    )
    .reset_index()
)

summary["mean_leading_film_share"] = (
    summary["mean_leading_film_share"].round(2)
)

summary["median_leading_film_share"] = (
    summary["median_leading_film_share"].round(2)
)

summary["mean_films_contributing"] = (
    summary["mean_films_contributing"].round(2)
)

# ------------------------------------------------------------
# 3. Identify highly concentrated supported years
# ------------------------------------------------------------

supported["concentration_flag"] = np.where(
    supported["leading_film_share_pct"] >= 70,
    "High concentration",
    "Distributed across films"
)

concentration_counts = (
    supported.groupby(
        ["curated_theme", "concentration_flag"]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# ------------------------------------------------------------
# 4. Display results
# ------------------------------------------------------------

print("=" * 75)
print("ROBUST THEME CONCENTRATION DIAGNOSTIC")
print("=" * 75)

print("\nAll theme-year combinations in expected grid:", 312)
print("Observed theme-year combinations:", len(annual))
print("Supported theme-year combinations (10+ chunks):", len(supported))

print("\nTheme concentration summary:")
display(
    summary.sort_values(
        "mean_leading_film_share",
        ascending=False
    )
)

print("\nConcentration distribution:")
display(concentration_counts)

# ------------------------------------------------------------
# 5. Save diagnostic
# ------------------------------------------------------------

SUMMARY_PATH = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_THEME_CONCENTRATION_ROBUST_DIAGNOSTIC.csv"
)

FLAG_PATH = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_THEME_CONCENTRATION_YEAR_FLAGS.csv"
)

summary.to_csv(SUMMARY_PATH, index=False)
concentration_counts.to_csv(FLAG_PATH, index=False)

print("\nSaved:")
print(SUMMARY_PATH)
print(FLAG_PATH)

print("\nDiagnostic completed.")

In [ ]:
# ============================================================
# NOTEBOOK 09 — FINAL TEMPORAL THEME VISUALIZATIONS
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

OUTPUT_DIR = "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"

FILM_PATH = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_THEME_YEARLY_FILM_COVERAGE_MATRIX.csv"
)

PREVALENCE_PATH = os.path.join(
    OUTPUT_DIR,
    "ICMB_130_THEME_YEARLY_DIALOGUE_PREVALENCE_MATRIX.csv"
)

# ------------------------------------------------------------
# 1. Load matrices
# ------------------------------------------------------------

film_matrix = pd.read_csv(FILM_PATH, index_col=0)
prevalence_matrix = pd.read_csv(PREVALENCE_PATH, index_col=0)

film_matrix.columns = film_matrix.columns.astype(int)
prevalence_matrix.columns = prevalence_matrix.columns.astype(int)

years = list(range(2000, 2026))

film_matrix = film_matrix.reindex(columns=years, fill_value=0)
prevalence_matrix = prevalence_matrix.reindex(columns=years, fill_value=0)

assert film_matrix.shape == (12, 26)
assert prevalence_matrix.shape == (12, 26)

print("Film coverage matrix:", film_matrix.shape)
print("Dialogue prevalence matrix:", prevalence_matrix.shape)

# ------------------------------------------------------------
# 2. Figure 1 — Dialogue prevalence trends
# ------------------------------------------------------------

plt.figure(figsize=(17, 9))

for theme in prevalence_matrix.index:
    plt.plot(
        years,
        prevalence_matrix.loc[theme],
        marker="o",
        markersize=3,
        linewidth=1.6,
        label=theme
    )

plt.title(
    "Longitudinal Trends in Curated Dialogue Themes (2000–2025)",
    fontsize=15
)
plt.xlabel("Year")
plt.ylabel("Share of Assigned Dialogue Chunks (%)")
plt.xticks(years, rotation=45)
plt.grid(alpha=0.3)
plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8
)
plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, "FIG_01_THEME_DIALOGUE_TRENDS.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 3. Figure 2 — Film coverage heatmap
# ------------------------------------------------------------

plt.figure(figsize=(18, 8))

plt.imshow(
    film_matrix.values,
    aspect="auto",
    interpolation="nearest"
)

plt.colorbar(label="Number of Films (out of 5)")
plt.xticks(
    range(len(years)),
    years,
    rotation=45
)
plt.yticks(
    range(len(film_matrix.index)),
    film_matrix.index
)

plt.title(
    "Cross-Film Coverage of Curated Themes (2000–2025)",
    fontsize=15
)
plt.xlabel("Year")
plt.ylabel("Curated Theme")

plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, "FIG_02_THEME_FILM_COVERAGE_HEATMAP.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ------------------------------------------------------------
# 4. Figure 3 — Dialogue prevalence heatmap
# ------------------------------------------------------------

plt.figure(figsize=(18, 8))

plt.imshow(
    prevalence_matrix.values,
    aspect="auto",
    interpolation="nearest"
)

plt.colorbar(label="Dialogue Prevalence (%)")
plt.xticks(
    range(len(years)),
    years,
    rotation=45
)
plt.yticks(
    range(len(prevalence_matrix.index)),
    prevalence_matrix.index
)

plt.title(
    "Annual Dialogue Prevalence of Curated Themes (2000–2025)",
    fontsize=15
)
plt.xlabel("Year")
plt.ylabel("Curated Theme")

plt.tight_layout()

plt.savefig(
    os.path.join(OUTPUT_DIR, "FIG_03_THEME_DIALOGUE_HEATMAP.png"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nAll three temporal visualizations saved successfully.")

In [ ]:
# ============================================================
# ICMB-130: THREE-PERIOD LONGITUDINAL THEME COMPARISON
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"
)

PREVALENCE_PATH = OUTPUT_DIR / "ICMB_130_THEME_YEARLY_DIALOGUE_PREVALENCE_MATRIX.csv"
FILM_PATH = OUTPUT_DIR / "ICMB_130_THEME_YEARLY_FILM_COVERAGE_MATRIX.csv"

prevalence = pd.read_csv(PREVALENCE_PATH, index_col=0)
film_coverage = pd.read_csv(FILM_PATH, index_col=0)

# Ensure year columns are integers
prevalence.columns = prevalence.columns.astype(int)
film_coverage.columns = film_coverage.columns.astype(int)

periods = {
    "Early (2000–2008)": list(range(2000, 2009)),
    "Middle (2009–2017)": list(range(2009, 2018)),
    "Recent (2018–2025)": list(range(2018, 2026))
}

rows = []

for theme in prevalence.index:
    row = {"Theme": theme}

    for period_name, years in periods.items():
        row[f"{period_name} Dialogue Share (%)"] = round(
            prevalence.loc[theme, years].mean(), 2
        )

        row[f"{period_name} Average Films"] = round(
            film_coverage.loc[theme, years].mean(), 2
        )

    row["Years Present"] = int(
        (film_coverage.loc[theme] > 0).sum()
    )

    rows.append(row)

period_comparison = pd.DataFrame(rows)

period_comparison = period_comparison.sort_values(
    "Recent (2018–2025) Dialogue Share (%)",
    ascending=False
)

output_path = OUTPUT_DIR / "ICMB_130_THREE_PERIOD_THEME_COMPARISON.csv"
period_comparison.to_csv(output_path, index=False)

print("=" * 90)
print("THREE-PERIOD LONGITUDINAL THEME COMPARISON")
print("=" * 90)

display(period_comparison)

print("\nSaved:", output_path)

In [ ]:
# ============================================================
# ICMB-130 | FINAL OUTPUT CONSOLIDATION
# STEP 1: VERIFY SOURCE FILES AND SCHEMAS
# ============================================================

import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"
)

files_to_check = [
    "ICMB_130_FINAL_CURATED_THEME_REGISTER.csv",
    "ICMB_130_THREE_PERIOD_THEME_COMPARISON.csv",
    "ICMB_130_CURATED_THEME_YEARLY_PREVALENCE.csv",
    "ICMB_130_THEME_YEARLY_FILM_COVERAGE_MATRIX.csv",
    "ICMB_130_THEME_YEARLY_DIALOGUE_PREVALENCE_MATRIX.csv",
    "ICMB_130_THEME_CONCENTRATION_ROBUST_DIAGNOSTIC.csv",
    "ICMB_130_THEME_CONCENTRATION_YEAR_FLAGS.csv",
    "ICMB_130_BASELINE_CROSS_MODEL_THEME_COMPARISON.csv",
    "BERTOPIC_BASELINE_TOPIC_SPECIFIC_NPMI.csv",
    "BERTOPIC_EXPERIMENT_COMPARISON.csv"
]

print("=" * 90)
print("ICMB-130 FINAL OUTPUT SOURCE AUDIT")
print("=" * 90)

for filename in files_to_check:
    path = OUTPUT_DIR / filename

    print(f"\n{'-' * 80}")
    print("FILE:", filename)

    if not path.exists():
        print("STATUS: NOT FOUND")
        continue

    df = pd.read_csv(path)

    print("STATUS: FOUND")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    if filename == "ICMB_130_FINAL_CURATED_THEME_REGISTER.csv":
        display(df)

    elif filename == "ICMB_130_THREE_PERIOD_THEME_COMPARISON.csv":
        display(df)

print("\nSOURCE AUDIT COMPLETED.")

In [ ]:
# ============================================================
# ICMB-130 | FINAL TOPIC MODELLING OUTPUT PACKAGE
# Consolidates existing results; does not rerun BERTopic.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"
)

# ------------------------------------------------------------
# 1. Load verified source files
# ------------------------------------------------------------

register = pd.read_csv(
    OUTPUT_DIR / "ICMB_130_FINAL_CURATED_THEME_REGISTER.csv"
)

periods = pd.read_csv(
    OUTPUT_DIR / "ICMB_130_THREE_PERIOD_THEME_COMPARISON.csv"
)

yearly = pd.read_csv(
    OUTPUT_DIR / "ICMB_130_CURATED_THEME_YEARLY_PREVALENCE.csv"
)

concentration = pd.read_csv(
    OUTPUT_DIR / "ICMB_130_THEME_CONCENTRATION_ROBUST_DIAGNOSTIC.csv"
)

concentration_flags = pd.read_csv(
    OUTPUT_DIR / "ICMB_130_THEME_CONCENTRATION_YEAR_FLAGS.csv"
)

film_matrix = pd.read_csv(
    OUTPUT_DIR / "ICMB_130_THEME_YEARLY_FILM_COVERAGE_MATRIX.csv"
)

dialogue_matrix = pd.read_csv(
    OUTPUT_DIR / "ICMB_130_THEME_YEARLY_DIALOGUE_PREVALENCE_MATRIX.csv"
)

model_comparison = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_EXPERIMENT_COMPARISON.csv"
)

npmi = pd.read_csv(
    OUTPUT_DIR / "BERTOPIC_BASELINE_TOPIC_SPECIFIC_NPMI.csv"
)

cross_model = pd.read_csv(
    OUTPUT_DIR / "ICMB_130_BASELINE_CROSS_MODEL_THEME_COMPARISON.csv"
)

# ------------------------------------------------------------
# 2. Prepare final included theme register
# ------------------------------------------------------------

included = register[
    register["decision"].str.strip().str.lower() == "included"
].copy()

included["theme_number"] = pd.to_numeric(
    included["theme_number"], errors="coerce"
)

included = included.sort_values("theme_number")

assert len(included) == 12, (
    f"Expected 12 included themes, found {len(included)}"
)

final_register = included[
    [
        "theme_number",
        "curated_theme",
        "baseline_topic_ids",
        "confidence",
        "decision_rationale"
    ]
].copy()

# Preserve the complete decision register separately.
full_register = register.copy()

# ------------------------------------------------------------
# 3. Prepare consolidated longitudinal theme summary
# ------------------------------------------------------------

final_summary = periods.merge(
    final_register[
        [
            "curated_theme",
            "theme_number",
            "baseline_topic_ids",
            "confidence"
        ]
    ],
    left_on="Theme",
    right_on="curated_theme",
    how="left",
    validate="one_to_one"
)

final_summary = final_summary.drop(columns=["curated_theme"])

final_summary = final_summary[
    [
        "theme_number",
        "Theme",
        "baseline_topic_ids",
        "confidence",
        "Early (2000–2008) Dialogue Share (%)",
        "Middle (2009–2017) Dialogue Share (%)",
        "Recent (2018–2025) Dialogue Share (%)",
        "Early (2000–2008) Average Films",
        "Middle (2009–2017) Average Films",
        "Recent (2018–2025) Average Films",
        "Years Present"
    ]
].sort_values("theme_number")

assert final_summary["Theme"].notna().all()
assert len(final_summary) == 12

# ------------------------------------------------------------
# 4. Prepare concentration diagnostic
# ------------------------------------------------------------

final_concentration = concentration.merge(
    concentration_flags,
    on="curated_theme",
    how="left",
    validate="one_to_one"
)

final_concentration = final_concentration.merge(
    final_register[
        ["curated_theme", "theme_number", "confidence"]
    ],
    on="curated_theme",
    how="inner",
    validate="one_to_one"
)

final_concentration = final_concentration.sort_values("theme_number")

# ------------------------------------------------------------
# 5. Prepare annual and matrix outputs
# ------------------------------------------------------------

yearly_final = yearly.merge(
    final_register[
        ["curated_theme", "theme_number", "baseline_topic_ids", "confidence"]
    ],
    on="curated_theme",
    how="inner",
    validate="many_to_one"
).sort_values(["theme_number", "year"])

film_matrix_final = film_matrix.copy()
dialogue_matrix_final = dialogue_matrix.copy()

film_matrix_final = film_matrix_final.rename(
    columns={"curated_theme": "Theme"}
)

dialogue_matrix_final = dialogue_matrix_final.rename(
    columns={"curated_theme": "Theme"}
)

# ------------------------------------------------------------
# 6. Prepare model diagnostic summary
# ------------------------------------------------------------

npmi_summary = pd.DataFrame({
    "Metric": [
        "Baseline non-outlier topics",
        "Baseline mean topic-specific NPMI",
        "Baseline median topic-specific NPMI",
        "Baseline average keyword-pair coverage (%)",
        "Baseline topics passing support threshold"
    ],
    "Value": [
        len(npmi),
        round(npmi["topic_specific_npmi"].mean(), 4),
        round(npmi["topic_specific_npmi"].median(), 4),
        round(npmi["pair_coverage_pct"].mean(), 2),
        int((npmi["support_category"].astype(str).str.lower() != "weak").sum())
    ]
})

# ------------------------------------------------------------
# 7. Validation checks
# ------------------------------------------------------------

assert len(final_register) == 12
assert yearly_final["curated_theme"].nunique() == 12
assert yearly_final["year"].nunique() == 26
assert len(yearly_final) == 312
assert final_concentration["curated_theme"].nunique() == 12
assert len(film_matrix_final) == 12
assert len(dialogue_matrix_final) == 12

print("=" * 90)
print("FINAL CONSOLIDATION VALIDATION")
print("=" * 90)

print("Included themes:", len(final_register))
print("Complete decision register:", len(full_register))
print("Annual theme records:", len(yearly_final))
print("Years covered:", yearly_final["year"].min(), "to", yearly_final["year"].max())
print("Concentration records:", len(final_concentration))
print("Film coverage matrix:", film_matrix_final.shape)
print("Dialogue prevalence matrix:", dialogue_matrix_final.shape)
print("Baseline NPMI topics:", len(npmi))
print("Cross-model comparison records:", len(cross_model))

# ------------------------------------------------------------
# 8. Save final CSV outputs
# ------------------------------------------------------------

csv_outputs = {
    "ICMB_130_FINAL_12_THEME_REGISTER.csv": final_register,
    "ICMB_130_FINAL_LONGITUDINAL_THEME_SUMMARY.csv": final_summary,
    "ICMB_130_FINAL_ANNUAL_THEME_PREVALENCE.csv": yearly_final,
    "ICMB_130_FINAL_THEME_CONCENTRATION.csv": final_concentration,
    "ICMB_130_FINAL_FILM_COVERAGE_MATRIX.csv": film_matrix_final,
    "ICMB_130_FINAL_DIALOGUE_PREVALENCE_MATRIX.csv": dialogue_matrix_final
}

for filename, dataframe in csv_outputs.items():
    dataframe.to_csv(OUTPUT_DIR / filename, index=False)

# ------------------------------------------------------------
# 9. Create one consolidated Excel workbook
# ------------------------------------------------------------

excel_path = OUTPUT_DIR / "ICMB_130_FINAL_TOPIC_MODELLING_RESULTS.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:

    final_summary.to_excel(
        writer, sheet_name="Longitudinal Summary", index=False
    )

    final_register.to_excel(
        writer, sheet_name="12 Curated Themes", index=False
    )

    full_register.to_excel(
        writer, sheet_name="Full Theme Decisions", index=False
    )

    yearly_final.to_excel(
        writer, sheet_name="Annual Prevalence", index=False
    )

    film_matrix_final.to_excel(
        writer, sheet_name="Film Coverage Matrix", index=False
    )

    dialogue_matrix_final.to_excel(
        writer, sheet_name="Dialogue Matrix", index=False
    )

    final_concentration.to_excel(
        writer, sheet_name="Film Concentration", index=False
    )

    model_comparison.to_excel(
        writer, sheet_name="Model Comparison", index=False
    )

    npmi.to_excel(
        writer, sheet_name="Baseline NPMI", index=False
    )

    npmi_summary.to_excel(
        writer, sheet_name="NPMI Summary", index=False
    )

    cross_model.to_excel(
        writer, sheet_name="Cross Model Comparison", index=False
    )

    # Basic workbook formatting
    for worksheet in writer.book.worksheets:
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions

        for column_cells in worksheet.columns:
            max_length = max(
                len(str(cell.value)) if cell.value is not None else 0
                for cell in column_cells
            )
            column_letter = column_cells[0].column_letter
            worksheet.column_dimensions[column_letter].width = min(
                max(max_length + 2, 12), 45
            )

# ------------------------------------------------------------
# 10. Final completion report
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("FINAL OUTPUT PACKAGE CREATED")
print("=" * 90)

print("\nExcel workbook:")
print(excel_path)

print("\nCSV files:")
for filename in csv_outputs:
    print(" -", filename)

print("\nFinal longitudinal summary:")
display(final_summary)

print("\nFinal concentration summary:")
display(final_concentration)

print("\nAll consolidation checks passed.")

In [ ]:
# ============================================================
# ICMB-130 | NMF BASELINE COMPLETION AUDIT
# ============================================================

from pathlib import Path
import pandas as pd

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/IndianCinemaTrendAnalytics/06_outputs/topic_modelling"
)

files = {
    "Original NMF topic terms": "NMF_TOPIC_TERMS.csv",
    "Refined NMF topic terms": "NMF_TOPIC_TERMS_REFINED.csv",
}

print("=" * 75)
print("NMF BASELINE AND REFINEMENT VERIFICATION")
print("=" * 75)

for label, filename in files.items():
    path = OUTPUT_DIR / filename

    print(f"\n{label}")
    print("-" * 60)

    if path.exists():
        print("Status: FOUND")
        print("File size (KB):", round(path.stat().st_size / 1024, 2))

        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        display(df.head(10))
    else:
        print("Status: NOT FOUND")

# Check for other NMF-related saved artifacts
print("\n" + "=" * 75)
print("OTHER NMF-RELATED OUTPUT FILES")
print("=" * 75)

nmf_files = sorted(
    p.name for p in OUTPUT_DIR.glob("*NMF*")
    if p.is_file()
)

if nmf_files:
    for filename in nmf_files:
        print("-", filename)
else:
    print("No NMF-related files found.")

print("\nNMF verification completed.")